In [149]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Link to Access Google Drive For Datasets and Saved Graphs and Results

https://drive.google.com/drive/folders/1Mhxlbf6HhAx30t7Gmp70pEF541MRdUgx?usp=drive_link



In [150]:
!pip install torch-geometric -q

In [151]:
import os
import json
import pickle
import copy
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from tqdm import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [152]:
ferguson_path = "/content/drive/MyDrive/7880TermProjectDatasets/ferguson"

In [153]:
PAPER_IRL_EPISODES = 1000
PAPER_GUIDANCE_EPISODE_LIMIT = 500
PAPER_GUIDANCE_FREQUENCY = 2
PAPER_NEGATIVES_PER_EXPERT = 100
PAPER_NEGATIVE_SIMILARITY_MU = 0.8
PAPER_DEFENSE_BETA = 8.0
PAPER_GCN_HIDDEN_DIM = 64
PAPER_PHEME_EPOCHS = 120
PAPER_WEIBO_EPOCHS = 60
PAPER_TARGET_MODEL_LR = 0.0001

print("Paper configuration is ready.")
print("IRL episodes:", PAPER_IRL_EPISODES)
print("Guidance episode limit:", PAPER_GUIDANCE_EPISODE_LIMIT)
print("Guidance frequency:", PAPER_GUIDANCE_FREQUENCY)
print("Negatives per expert:", PAPER_NEGATIVES_PER_EXPERT)
print("Negative similarity μ:", PAPER_NEGATIVE_SIMILARITY_MU)
print("Defense beta:", PAPER_DEFENSE_BETA)

Paper configuration is ready.
IRL episodes: 1000
Guidance episode limit: 500
Guidance frequency: 2
Negatives per expert: 100
Negative similarity μ: 0.8
Defense beta: 8.0


In [154]:



#BUILDING GRAPH FROM FERGUSON




In [155]:
#HELPERS
def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_first_json_file(folder_path):
    if not os.path.exists(folder_path):
        return None

    files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

    if len(files) == 0:
        return None

    return os.path.join(folder_path, files[0])


def safe_get_tweet_id(tweet):
    tweet_id = tweet.get("id")
    return str(tweet_id) if tweet_id is not None else None


def safe_get_user_id(tweet):
    user = tweet.get("user", {})
    user_id = user.get("id")
    return str(user_id) if user_id is not None else None


def safe_get_reply_to_status_id(tweet):
    reply_id = tweet.get("in_reply_to_status_id")
    return str(reply_id) if reply_id is not None else None


def make_message_node(tweet_id):
    return f"message_{tweet_id}"


def make_comment_node(tweet_id):
    return f"comment_{tweet_id}"


def make_user_node(user_id):
    return f"user_{user_id}"

In [156]:
def collect_ferguson_conversations(base_dir, max_per_class=None):
    categories = {
        "rumours": 1,
        "non-rumours": 0
    }

    selected_conversations = []

    for category, label in categories.items():
        category_path = os.path.join(base_dir, category)

        if not os.path.exists(category_path):
            print(f"Missing category folder: {category_path}")
            continue

        conversations = []

        for conversation_id in sorted(os.listdir(category_path)):
            conversation_path = os.path.join(category_path, conversation_id)

            if os.path.isdir(conversation_path):
                conversations.append((conversation_id, conversation_path, label, category))

        if max_per_class is not None:
            conversations = conversations[:max_per_class]

        selected_conversations.extend(conversations)

    return selected_conversations

In [157]:
def build_ferguson_graph(base_dir, max_per_class=None):
    G = nx.MultiDiGraph()

    tweet_author = {}
    conversation_labels = {}

    all_conversations = collect_ferguson_conversations(
        base_dir,
        max_per_class=max_per_class
    )

    print("Conversations selected:", len(all_conversations))
    print("Label distribution:", Counter([x[2] for x in all_conversations]))

    for conversation_id, conversation_path, label, category in all_conversations:
        source_folder = os.path.join(conversation_path, "source-tweet")
        reactions_folder = os.path.join(conversation_path, "reactions")

        source_json_path = get_first_json_file(source_folder)

        if source_json_path is None:
            print("No source tweet for:", conversation_id)
            continue

        source_tweet = load_json_file(source_json_path)

        source_tweet_id = safe_get_tweet_id(source_tweet)
        source_user_id = safe_get_user_id(source_tweet)

        if source_tweet_id is None or source_user_id is None:
            print("Missing source tweet id or source user id for:", conversation_id)
            continue

        message_node = make_message_node(source_tweet_id)
        source_user_node = make_user_node(source_user_id)

        G.add_node(
            message_node,
            node_type="message",
            tweet_id=source_tweet_id,
            conversation_id=conversation_id,
            label=label,
            category=category,
            text=source_tweet.get("text", "")
        )

        G.add_node(
            source_user_node,
            node_type="user",
            user_id=source_user_id,
            screen_name=source_tweet.get("user", {}).get("screen_name", "")
        )

        G.add_edge(
            source_user_node,
            message_node,
            edge_type="writes_message",
            relation="user-message",
            conversation_id=conversation_id
        )

        tweet_author[source_tweet_id] = source_user_id
        conversation_labels[conversation_id] = label

        reaction_tweets = []

        if os.path.exists(reactions_folder):
            for filename in sorted(os.listdir(reactions_folder)):
                if not filename.endswith(".json"):
                    continue

                reaction_path = os.path.join(reactions_folder, filename)

                try:
                    reaction = load_json_file(reaction_path)
                    reaction_tweets.append(reaction)
                except Exception as e:
                    print("Could not read:", reaction_path, e)

        # First pass: comments + reaction users
        for reaction in reaction_tweets:
            reaction_tweet_id = safe_get_tweet_id(reaction)
            reaction_user_id = safe_get_user_id(reaction)

            if reaction_tweet_id is None or reaction_user_id is None:
                continue

            comment_node = make_comment_node(reaction_tweet_id)
            reaction_user_node = make_user_node(reaction_user_id)

            G.add_node(
                comment_node,
                node_type="comment",
                tweet_id=reaction_tweet_id,
                conversation_id=conversation_id,
                text=reaction.get("text", "")
            )

            G.add_node(
                reaction_user_node,
                node_type="user",
                user_id=reaction_user_id,
                screen_name=reaction.get("user", {}).get("screen_name", "")
            )

            G.add_edge(
                reaction_user_node,
                comment_node,
                edge_type="writes_comment",
                relation="user-comment",
                conversation_id=conversation_id
            )

            G.add_edge(
                message_node,
                comment_node,
                edge_type="has_comment",
                relation="message-comment",
                conversation_id=conversation_id
            )

            tweet_author[reaction_tweet_id] = reaction_user_id

        # Second pass: user-user reply edges
        for reaction in reaction_tweets:
            reaction_tweet_id = safe_get_tweet_id(reaction)
            reaction_user_id = safe_get_user_id(reaction)
            replied_to_tweet_id = safe_get_reply_to_status_id(reaction)

            if reaction_tweet_id is None or reaction_user_id is None:
                continue

            if replied_to_tweet_id is None:
                continue

            if replied_to_tweet_id not in tweet_author:
                continue

            replying_user_node = make_user_node(reaction_user_id)
            replied_to_user_node = make_user_node(tweet_author[replied_to_tweet_id])

            if replying_user_node == replied_to_user_node:
                continue

            G.add_edge(
                replying_user_node,
                replied_to_user_node,
                edge_type="replies_to_user",
                relation="user-user",
                conversation_id=conversation_id,
                reply_tweet_id=reaction_tweet_id,
                replied_to_tweet_id=replied_to_tweet_id
            )

    return G, conversation_labels

In [158]:
"""
G_full, conversation_labels_full = build_ferguson_graph(
    ferguson_path,
    max_per_class=None
)
"""

'\nG_full, conversation_labels_full = build_ferguson_graph(\n    ferguson_path,\n    max_per_class=None\n)\n'

In [159]:
""""
import pickle
import os

save_dir = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "ferguson_full_graph.pkl")

with open(save_path, "wb") as f:
    pickle.dump(
        {
            "graph": G_full,
            "conversation_labels": conversation_labels_full
        },
        f
    )

print("Saved to:", save_path)
"""

'"\nimport pickle\nimport os\n\nsave_dir = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs"\nos.makedirs(save_dir, exist_ok=True)\n\nsave_path = os.path.join(save_dir, "ferguson_full_graph.pkl")\n\nwith open(save_path, "wb") as f:\n    pickle.dump(\n        {\n            "graph": G_full,\n            "conversation_labels": conversation_labels_full\n        },\n        f\n    )\n\nprint("Saved to:", save_path)\n'

In [160]:
import pickle

load_path = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_full_graph.pkl"

with open(load_path, "rb") as f:
    saved_data = pickle.load(f)

G_full = saved_data["graph"]
conversation_labels_full = saved_data["conversation_labels"]

print("Loaded graph")
print("Number of nodes:", G_full.number_of_nodes())
print("Number of edges:", G_full.number_of_edges())

Loaded graph
Number of nodes: 34851
Number of edges: 68245


In [161]:
print("FULL GRAPH")
print("Number of nodes:", G_full.number_of_nodes())
print("Number of edges:", G_full.number_of_edges())

print("\nNode types:")
print(Counter(nx.get_node_attributes(G_full, "node_type").values()))

edge_type_counts_full = Counter()
relation_counts_full = Counter()

for u, v, data in G_full.edges(data=True):
    edge_type_counts_full[data.get("edge_type")] += 1
    relation_counts_full[data.get("relation")] += 1

print("\nEdge types:")
print(edge_type_counts_full)

print("\nRelations:")
print(relation_counts_full)

print("\nConversation labels:")
print(Counter(conversation_labels_full.values()))

FULL GRAPH
Number of nodes: 34851
Number of edges: 68245

Node types:
Counter({'comment': 22911, 'user': 10799, 'message': 1141})

Edge types:
Counter({'has_comment': 22911, 'writes_comment': 22911, 'replies_to_user': 21282, 'writes_message': 1141})

Relations:
Counter({'message-comment': 22911, 'user-comment': 22911, 'user-user': 21282, 'user-message': 1141})

Conversation labels:
Counter({0: 857, 1: 284})


In [162]:
def convert_networkx_to_pyg_data(G):
    nodes = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    x_list = []
    y_list = []
    message_mask_list = []

    for node in nodes:
        data = G.nodes[node]
        node_type = data.get("node_type")

        is_message = 1.0 if node_type == "message" else 0.0
        is_user = 1.0 if node_type == "user" else 0.0
        is_comment = 1.0 if node_type == "comment" else 0.0

        degree = float(G.degree(node))
        in_degree = float(G.in_degree(node))
        out_degree = float(G.out_degree(node))

        features = [
            is_message,
            is_user,
            is_comment,
            degree,
            in_degree,
            out_degree
        ]

        x_list.append(features)

        if node_type == "message":
            y_list.append(int(data.get("label")))
            message_mask_list.append(True)
        else:
            y_list.append(-1)
            message_mask_list.append(False)

    edge_list = []

    for u, v in G.edges():
        u_idx = node_to_idx[u]
        v_idx = node_to_idx[v]

        # Add both directions for GCN
        edge_list.append([u_idx, v_idx])
        edge_list.append([v_idx, u_idx])

    x = torch.tensor(x_list, dtype=torch.float)
    y = torch.tensor(y_list, dtype=torch.long)
    message_mask = torch.tensor(message_mask_list, dtype=torch.bool)

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    data = Data(
        x=x,
        edge_index=edge_index,
        y=y,
        message_mask=message_mask
    )

    data.node_names = nodes

    return data, node_to_idx

In [163]:
import torch
from collections import Counter

def create_train_test_masks(data, train_ratio=0.7, seed=42):
    torch.manual_seed(seed)

    message_indices = torch.where(data.message_mask)[0]
    message_labels = data.y[message_indices]

    rumor_indices = message_indices[message_labels == 1]
    nonrumor_indices = message_indices[message_labels == 0]

    rumor_indices = rumor_indices[torch.randperm(len(rumor_indices))]
    nonrumor_indices = nonrumor_indices[torch.randperm(len(nonrumor_indices))]

    num_rumor_train = int(len(rumor_indices) * train_ratio)
    num_nonrumor_train = int(len(nonrumor_indices) * train_ratio)

    train_indices = torch.cat([
        rumor_indices[:num_rumor_train],
        nonrumor_indices[:num_nonrumor_train]
    ])

    test_indices = torch.cat([
        rumor_indices[num_rumor_train:],
        nonrumor_indices[num_nonrumor_train:]
    ])

    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

    train_mask[train_indices] = True
    test_mask[test_indices] = True

    data.train_mask = train_mask
    data.test_mask = test_mask

    return data


In [164]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super().__init__()

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)

        return x

In [165]:
def evaluate(model, data, mask):
    model.eval()

    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)

        y_true = data.y[mask]
        y_pred = pred[mask]

        accuracy = (y_true == y_pred).sum().item() / mask.sum().item()

        results = {"accuracy": accuracy}

        for cls in [0, 1]:
            tp = ((y_pred == cls) & (y_true == cls)).sum().item()
            fp = ((y_pred == cls) & (y_true != cls)).sum().item()
            fn = ((y_pred != cls) & (y_true == cls)).sum().item()

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

            results[f"class_{cls}_precision"] = precision
            results[f"class_{cls}_recall"] = recall
            results[f"class_{cls}_f1"] = f1

    return results

In [166]:
def get_tweet_id_as_int(G, node):
    tweet_id = G.nodes[node].get("tweet_id")

    try:
        return int(tweet_id)
    except:
        return 10**30


def get_author_node(G, target_node, edge_type):
    for user_node, node, attrs in G.in_edges(target_node, data=True):
        if attrs.get("edge_type") == edge_type:
            return user_node

    return None

In [167]:
def select_comments_for_processed_graph(G, target_comments=557):
    comments_by_conversation = defaultdict(list)

    for node, attrs in G.nodes(data=True):
        if attrs.get("node_type") == "comment":
            conversation_id = attrs.get("conversation_id")
            comments_by_conversation[conversation_id].append(node)

    for conversation_id in comments_by_conversation:
        comments_by_conversation[conversation_id] = sorted(
            comments_by_conversation[conversation_id],
            key=lambda node: get_tweet_id_as_int(G, node)
        )

    selected_comment_nodes = []

    conversation_ids = sorted(comments_by_conversation.keys())

    round_number = 0

    while len(selected_comment_nodes) < target_comments:
        added_in_this_round = 0

        for conversation_id in conversation_ids:
            comments = comments_by_conversation[conversation_id]

            if round_number < len(comments):
                selected_comment_nodes.append(comments[round_number])
                added_in_this_round += 1

                if len(selected_comment_nodes) == target_comments:
                    break

        if added_in_this_round == 0:
            break

        round_number += 1

    return set(selected_comment_nodes)

In [168]:
def build_processed_graph_like_paper(
    G,
    target_comments=557,
    target_users=1008,
    target_edges=4401
):
    G_processed = nx.MultiDiGraph()

    # 1. Select all message nodes
    selected_message_nodes = set()

    for node, attrs in G.nodes(data=True):
        if attrs.get("node_type") == "message":
            selected_message_nodes.add(node)

    # 2. Select only 557 comments
    selected_comment_nodes = select_comments_for_processed_graph(
        G,
        target_comments=target_comments
    )

    # 3. Find source authors and selected comment authors
    message_author = {}
    comment_author = {}

    source_user_nodes = set()
    comment_user_nodes = set()

    for message_node in selected_message_nodes:
        author_node = get_author_node(G, message_node, "writes_message")

        if author_node is not None:
            message_author[message_node] = author_node
            source_user_nodes.add(author_node)

    for comment_node in selected_comment_nodes:
        author_node = get_author_node(G, comment_node, "writes_comment")

        if author_node is not None:
            comment_author[comment_node] = author_node
            comment_user_nodes.add(author_node)

    selected_user_nodes = set(source_user_nodes) | set(comment_user_nodes)

    # 4. If user count is too high, keep all source authors first,
    # then keep the most active/high-degree comment users.
    if len(selected_user_nodes) > target_users:
        comment_user_counts = Counter()

        for comment_node, author_node in comment_author.items():
            comment_user_counts[author_node] += 1

        extra_comment_users = list(selected_user_nodes - source_user_nodes)

        extra_comment_users = sorted(
            extra_comment_users,
            key=lambda user_node: (
                -comment_user_counts[user_node],
                -G.degree(user_node),
                user_node
            )
        )

        selected_user_nodes = set(source_user_nodes)

        for user_node in extra_comment_users:
            if len(selected_user_nodes) >= target_users:
                break

            selected_user_nodes.add(user_node)

    # 5. If user count is too low, add high-degree raw users
    if len(selected_user_nodes) < target_users:
        all_user_nodes = [
            node for node, attrs in G.nodes(data=True)
            if attrs.get("node_type") == "user"
        ]

        all_user_nodes = sorted(
            all_user_nodes,
            key=lambda user_node: (-G.degree(user_node), user_node)
        )

        for user_node in all_user_nodes:
            if len(selected_user_nodes) >= target_users:
                break

            selected_user_nodes.add(user_node)

    # 6. Add selected nodes
    selected_nodes = selected_message_nodes | selected_comment_nodes | selected_user_nodes

    for node in selected_nodes:
        G_processed.add_node(node, **G.nodes[node])

    # 7. Add user-message edges
    for user_node, message_node, attrs in G.edges(data=True):
        if attrs.get("edge_type") == "writes_message":
            if user_node in selected_user_nodes and message_node in selected_message_nodes:
                G_processed.add_edge(
                    user_node,
                    message_node,
                    edge_type="writes_message",
                    relation="user-message",
                    conversation_id=attrs.get("conversation_id")
                )

    # 8. Add message-comment edges
    for message_node, comment_node, attrs in G.edges(data=True):
        if attrs.get("edge_type") == "has_comment":
            if message_node in selected_message_nodes and comment_node in selected_comment_nodes:
                G_processed.add_edge(
                    message_node,
                    comment_node,
                    edge_type="has_comment",
                    relation="message-comment",
                    conversation_id=attrs.get("conversation_id")
                )

    # 9. Add existing reply-based user-user edges first
    added_user_user_pairs = set()

    for user_a, user_b, attrs in G.edges(data=True):
        if G_processed.number_of_edges() >= target_edges:
            break

        if attrs.get("edge_type") == "replies_to_user":
            if user_a in selected_user_nodes and user_b in selected_user_nodes:
                pair = (user_a, user_b)

                if pair not in added_user_user_pairs:
                    G_processed.add_edge(
                        user_a,
                        user_b,
                        edge_type="user_user",
                        relation="user-user",
                        conversation_id=attrs.get("conversation_id")
                    )

                    added_user_user_pairs.add(pair)

    # 10. If still fewer than target_edges, add community-style user-user edges
    conversation_users = defaultdict(set)

    for message_node, author_node in message_author.items():
        if author_node in selected_user_nodes:
            conversation_id = G.nodes[message_node].get("conversation_id")
            conversation_users[conversation_id].add(author_node)

    for comment_node, author_node in comment_author.items():
        if author_node in selected_user_nodes:
            conversation_id = G.nodes[comment_node].get("conversation_id")
            conversation_users[conversation_id].add(author_node)

    for conversation_id in sorted(conversation_users.keys()):
        if G_processed.number_of_edges() >= target_edges:
            break

        users = sorted(list(conversation_users[conversation_id]))

        for i in range(len(users)):
            if G_processed.number_of_edges() >= target_edges:
                break

            for j in range(i + 1, len(users)):
                if G_processed.number_of_edges() >= target_edges:
                    break

                user_a = users[i]
                user_b = users[j]

                pair = (user_a, user_b)

                if pair not in added_user_user_pairs:
                    G_processed.add_edge(
                        user_a,
                        user_b,
                        edge_type="user_user",
                        relation="user-user",
                        conversation_id=conversation_id,
                        construction="community_same_conversation"
                    )

                    added_user_user_pairs.add(pair)

    info = {
        "selected_messages": len(selected_message_nodes),
        "selected_comments": len(selected_comment_nodes),
        "selected_users": len(selected_user_nodes),
        "target_comments": target_comments,
        "target_users": target_users,
        "target_edges": target_edges
    }

    return G_processed, info

In [169]:
G_processed, processed_info = build_processed_graph_like_paper(
    G_full,
    target_comments=557,
    target_users=1008,
    target_edges=4401
)

In [170]:
#20
def add_user_user_edges_until_target(G_processed, G_full, target_edges=4401):
    selected_user_nodes = [
        node for node, attrs in G_processed.nodes(data=True)
        if attrs.get("node_type") == "user"
    ]

    selected_user_nodes = sorted(
        selected_user_nodes,
        key=lambda node: (-G_full.degree(node), node)
    )

    existing_user_user_pairs = set()

    for u, v, attrs in G_processed.edges(data=True):
        if attrs.get("relation") == "user-user":
            existing_user_user_pairs.add((u, v))

    added_edges = 0

    for i in range(len(selected_user_nodes)):
        if G_processed.number_of_edges() >= target_edges:
            break

        for j in range(i + 1, len(selected_user_nodes)):
            if G_processed.number_of_edges() >= target_edges:
                break

            user_a = selected_user_nodes[i]
            user_b = selected_user_nodes[j]

            if (user_a, user_b) in existing_user_user_pairs:
                continue

            G_processed.add_edge(
                user_a,
                user_b,
                edge_type="user_user",
                relation="user-user",
                construction="high_degree_user_community_fill"
            )

            existing_user_user_pairs.add((user_a, user_b))
            added_edges += 1

    print("Added user-user edges:", added_edges)
    print("New total edges:", G_processed.number_of_edges())

    return G_processed

In [171]:
G_processed = add_user_user_edges_until_target(
    G_processed,
    G_full,
    target_edges=4401
)

Added user-user edges: 979
New total edges: 4401


In [172]:
print("Processed graph nodes:", G_processed.number_of_nodes())
print("Processed graph edges:", G_processed.number_of_edges())

print("\nNode types:")
print(Counter(nx.get_node_attributes(G_processed, "node_type").values()))

edge_type_counts = Counter()
relation_counts = Counter()

for u, v, attrs in G_processed.edges(data=True):
    edge_type_counts[attrs.get("edge_type")] += 1
    relation_counts[attrs.get("relation")] += 1

print("\nEdge types:")
print(edge_type_counts)

print("\nRelations:")
print(relation_counts)

message_labels = []

for node, attrs in G_processed.nodes(data=True):
    if attrs.get("node_type") == "message":
        message_labels.append(attrs.get("label"))

print("\nMessage label distribution:")
print(Counter(message_labels))

Processed graph nodes: 2706
Processed graph edges: 4401

Node types:
Counter({'message': 1141, 'user': 1008, 'comment': 557})

Edge types:
Counter({'user_user': 2703, 'writes_message': 1141, 'has_comment': 557})

Relations:
Counter({'user-user': 2703, 'user-message': 1141, 'message-comment': 557})

Message label distribution:
Counter({0: 857, 1: 284})


In [173]:
save_dir = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs"
os.makedirs(save_dir, exist_ok=True)

processed_graph_path = os.path.join(save_dir, "ferguson_processed_graph_like_paper.pkl")

with open(processed_graph_path, "wb") as f:
    pickle.dump(
        {
            "graph": G_processed,
            "processed_info": processed_info
        },
        f
    )

print("Saved processed graph to:", processed_graph_path)

Saved processed graph to: /content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_graph_like_paper.pkl


In [174]:
data, node_to_idx = convert_networkx_to_pyg_data(G_processed)

print(data)
print("Feature matrix:", data.x.shape)
print("Edge index:", data.edge_index.shape)
print("Labels:", data.y.shape)
print("Message nodes:", data.message_mask.sum().item())
print("Label distribution among messages:", Counter(data.y[data.message_mask].tolist()))

Data(x=[2706, 6], edge_index=[2, 8802], y=[2706], message_mask=[2706], node_names=[2706])
Feature matrix: torch.Size([2706, 6])
Edge index: torch.Size([2, 8802])
Labels: torch.Size([2706])
Message nodes: 1141
Label distribution among messages: Counter({0: 857, 1: 284})


In [175]:
save_pyg_processed_path = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_pyg_data.pt"

torch.save(data.cpu(), save_pyg_processed_path)

print("Saved processed PyG data to:", save_pyg_processed_path)

Saved processed PyG data to: /content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_pyg_data.pt


In [176]:
data = create_train_test_masks(data, train_ratio=0.7, seed=42)

print("Train messages:", data.train_mask.sum().item())
print("Test messages:", data.test_mask.sum().item())

print("Train label distribution:", Counter(data.y[data.train_mask].tolist()))
print("Test label distribution:", Counter(data.y[data.test_mask].tolist()))

Train messages: 797
Test messages: 344
Train label distribution: Counter({0: 599, 1: 198})
Test label distribution: Counter({0: 258, 1: 86})


In [177]:
from sklearn.feature_extraction.text import TfidfVectorizer
import torch

data = data.cpu()

message_texts = []
message_indices = []

for idx, node in enumerate(data.node_names):
    attrs = G_processed.nodes[node]

    if attrs.get("node_type") == "message":
        message_texts.append(attrs.get("text", ""))
        message_indices.append(idx)

print("Number of message texts:", len(message_texts))
print("Number of message indices:", len(message_indices))

vectorizer = TfidfVectorizer(
    max_features=300,
    stop_words="english",
    lowercase=True,
    min_df=2
)

message_tfidf = vectorizer.fit_transform(message_texts).toarray()

print("TF-IDF shape:", message_tfidf.shape)

tfidf_features = torch.zeros(
    (data.num_nodes, message_tfidf.shape[1]),
    dtype=torch.float
)

tfidf_features[message_indices] = torch.tensor(
    message_tfidf,
    dtype=torch.float
)

x_structural = data.x[:, :6].clone()

# Normalize degree features
x_structural[:, 3:6] = torch.log1p(x_structural[:, 3:6])

for col in [3, 4, 5]:
    max_value = x_structural[:, col].max()
    if max_value > 0:
        x_structural[:, col] = x_structural[:, col] / max_value

data.x = torch.cat([x_structural, tfidf_features], dim=1)

print("New feature matrix:", data.x.shape)
print("Structural features:", x_structural.shape)
print("Text features:", tfidf_features.shape)

print("Train messages:", data.train_mask.sum().item())
print("Test messages:", data.test_mask.sum().item())

Number of message texts: 1141
Number of message indices: 1141
TF-IDF shape: (1141, 300)
New feature matrix: torch.Size([2706, 306])
Structural features: torch.Size([2706, 6])
Text features: torch.Size([2706, 300])
Train messages: 797
Test messages: 344


In [178]:
save_pyg_processed_tfidf_path = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_pyg_tfidf_data.pt"

torch.save(data.cpu(), save_pyg_processed_tfidf_path)

print("Saved processed TF-IDF PyG data to:", save_pyg_processed_tfidf_path)

Saved processed TF-IDF PyG data to: /content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_pyg_tfidf_data.pt


In [179]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

data = data.to(device)

train_labels = data.y[data.train_mask]
class_counts = torch.bincount(train_labels, minlength=2)

class_weights = class_counts.sum() / (2.0 * class_counts.float())
class_weights = class_weights.to(device)

print("Class counts:", class_counts.detach().cpu())
print("Class weights:", class_weights.detach().cpu())

model = GCN(
    input_dim=data.x.shape[1],
    hidden_dim=PAPER_GCN_HIDDEN_DIM,
    output_dim=2
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=PAPER_TARGET_MODEL_LR,
    weight_decay=5e-4
)

num_epochs = PAPER_PHEME_EPOCHS

history = []

for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()

    out = model(data)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask],
        weight=class_weights
    )

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        train_results = evaluate(model, data, data.train_mask)
        test_results = evaluate(model, data, data.test_mask)

        history.append({
            "epoch": epoch,
            "loss": loss.item(),
            "train_accuracy": train_results["accuracy"],
            "test_accuracy": test_results["accuracy"],
            "test_rumor_f1": test_results["class_1_f1"],
            "test_nonrumor_f1": test_results["class_0_f1"]
        })

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Train Acc: {train_results['accuracy']:.4f} | "
            f"Test Acc: {test_results['accuracy']:.4f} | "
            f"Rumor F1: {test_results['class_1_f1']:.4f} | "
            f"Non-rumor F1: {test_results['class_0_f1']:.4f}"
        )

Device: cuda
Class counts: tensor([599, 198])
Class weights: tensor([0.6653, 2.0126])
Epoch 001 | Loss: 0.7071 | Train Acc: 0.2497 | Test Acc: 0.2558 | Rumor F1: 0.3991 | Non-rumor F1: 0.0229
Epoch 010 | Loss: 0.6998 | Train Acc: 0.2522 | Test Acc: 0.2558 | Rumor F1: 0.3991 | Non-rumor F1: 0.0229
Epoch 020 | Loss: 0.6988 | Train Acc: 0.2585 | Test Acc: 0.2674 | Rumor F1: 0.4000 | Non-rumor F1: 0.0597
Epoch 030 | Loss: 0.6930 | Train Acc: 0.2811 | Test Acc: 0.2907 | Rumor F1: 0.4078 | Non-rumor F1: 0.1159
Epoch 040 | Loss: 0.6919 | Train Acc: 0.3174 | Test Acc: 0.3198 | Rumor F1: 0.4179 | Non-rumor F1: 0.1818
Epoch 050 | Loss: 0.6890 | Train Acc: 0.3488 | Test Acc: 0.3343 | Rumor F1: 0.4203 | Non-rumor F1: 0.2184
Epoch 060 | Loss: 0.6852 | Train Acc: 0.4103 | Test Acc: 0.3721 | Rumor F1: 0.4286 | Non-rumor F1: 0.3032
Epoch 070 | Loss: 0.6819 | Train Acc: 0.4693 | Test Acc: 0.4477 | Rumor F1: 0.4571 | Non-rumor F1: 0.4379
Epoch 080 | Loss: 0.6806 | Train Acc: 0.5508 | Test Acc: 0.5320 | 

In [180]:
#30
final_results = evaluate(model, data, data.test_mask)

print("FINAL TEST RESULTS")
for key, value in final_results.items():
    print(f"{key}: {value:.4f}")

model.eval()

with torch.no_grad():
    out = model(data)
    pred = out.argmax(dim=1)

test_y_true = data.y[data.test_mask].cpu()
test_y_pred = pred[data.test_mask].cpu()

print("\nTrue test labels:")
print(Counter(test_y_true.tolist()))

print("\nPredicted test labels:")
print(Counter(test_y_pred.tolist()))

tn = ((test_y_true == 0) & (test_y_pred == 0)).sum().item()
fp = ((test_y_true == 0) & (test_y_pred == 1)).sum().item()
fn = ((test_y_true == 1) & (test_y_pred == 0)).sum().item()
tp = ((test_y_true == 1) & (test_y_pred == 1)).sum().item()

print("\nConfusion matrix:")
print("TN non-rumour correctly predicted:", tn)
print("FP non-rumour predicted as rumour:", fp)
print("FN rumour predicted as non-rumour:", fn)
print("TP rumour correctly predicted:", tp)

save_model_path = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_gcn_class_weighted.pt"

torch.save(
    {
        "model_state_dict": model.cpu().state_dict(),
        "data": data.cpu(),
        "history": history,
        "final_results": final_results,
        "class_weights": class_weights.detach().cpu()
    },
    save_model_path
)

print("\nSaved model to:", save_model_path)

majority_class = Counter(data.y[data.train_mask].cpu().tolist()).most_common(1)[0][0]

test_y_true = data.y[data.test_mask].cpu()
majority_pred = torch.full_like(test_y_true, fill_value=majority_class)

majority_accuracy = (majority_pred == test_y_true).float().mean().item()

print("\nMajority class from training set:", majority_class)
print("Majority baseline test accuracy:", majority_accuracy)
print("GCN test accuracy:", final_results["accuracy"])

FINAL TEST RESULTS
accuracy: 0.7413
class_0_precision: 0.9333
class_0_recall: 0.7054
class_0_f1: 0.8035
class_1_precision: 0.4899
class_1_recall: 0.8488
class_1_f1: 0.6213

True test labels:
Counter({0: 258, 1: 86})

Predicted test labels:
Counter({0: 195, 1: 149})

Confusion matrix:
TN non-rumour correctly predicted: 182
FP non-rumour predicted as rumour: 76
FN rumour predicted as non-rumour: 13
TP rumour correctly predicted: 73

Saved model to: /content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/ferguson_processed_gcn_class_weighted.pt

Majority class from training set: 0
Majority baseline test accuracy: 0.75
GCN test accuracy: 0.7412790697674418


In [181]:



#ATTACKS




In [182]:
#PAGERANK

In [183]:
saved_dir = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs"

data_path = os.path.join(saved_dir, "ferguson_processed_pyg_tfidf_data.pt")
graph_path = os.path.join(saved_dir, "ferguson_processed_graph_like_paper.pkl")
model_path = os.path.join(saved_dir, "ferguson_processed_gcn_class_weighted.pt")

def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

data = safe_torch_load(data_path, map_location="cpu")

with open(graph_path, "rb") as f:
    G_processed = pickle.load(f)

if isinstance(G_processed, dict):
    G_processed = G_processed["graph"]

model = GCN(
    input_dim=data.x.shape[1],
    hidden_dim=PAPER_GCN_HIDDEN_DIM,
    output_dim=2
).to(device)

checkpoint = safe_torch_load(model_path, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

for param in model.parameters():
    param.requires_grad = False

data = data.to(device)

print("Frozen target GCN loaded.")
print(data)
print("Feature shape:", data.x.shape)
print("Message label distribution:", Counter(data.y[data.message_mask].detach().cpu().tolist()))
print("Graph type:", type(G_processed))
print("Graph nodes:", G_processed.number_of_nodes())
print("Graph edges:", G_processed.number_of_edges())

Frozen target GCN loaded.
Data(x=[2706, 306], edge_index=[2, 8802], y=[2706], message_mask=[2706], node_names=[2706], train_mask=[2706], test_mask=[2706])
Feature shape: torch.Size([2706, 306])
Message label distribution: Counter({0: 857, 1: 284})
Graph type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
Graph nodes: 2706
Graph edges: 4401


In [184]:
def evaluate_for_attack(model, data, mask, name="Graph"):
    model.eval()

    with torch.no_grad():
        out = model(data)
        probs = F.softmax(out, dim=1)
        pred = out.argmax(dim=1)

    y_true = data.y[mask].detach().cpu().numpy()
    y_pred = pred[mask].detach().cpu().numpy()

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    rumor_mask = mask & (data.y == 1)
    nonrumor_mask = mask & (data.y == 0)

    if rumor_mask.sum().item() > 0:
        avg_rumor_prob_on_true_rumors = probs[rumor_mask, 1].mean().item()
        target_rumor_loss = F.cross_entropy(
            out[rumor_mask],
            data.y[rumor_mask],
            reduction="mean"
        ).item()
    else:
        avg_rumor_prob_on_true_rumors = None
        target_rumor_loss = None

    if nonrumor_mask.sum().item() > 0:
        avg_rumor_prob_on_true_nonrumors = probs[nonrumor_mask, 1].mean().item()
    else:
        avg_rumor_prob_on_true_nonrumors = None

    result = {
        "name": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "nonrumor_precision": precision[0],
        "nonrumor_recall": recall[0],
        "nonrumor_f1": f1[0],
        "rumor_precision": precision[1],
        "rumor_recall": recall[1],
        "rumor_f1": f1[1],
        "avg_rumor_prob_on_true_rumors": avg_rumor_prob_on_true_rumors,
        "avg_rumor_prob_on_true_nonrumors": avg_rumor_prob_on_true_nonrumors,
        "target_rumor_loss": target_rumor_loss,
        "predicted_nonrumor": int((y_pred == 0).sum()),
        "predicted_rumor": int((y_pred == 1).sum())
    }

    print("\n==============================")
    print(name)
    print("==============================")
    print(pd.DataFrame([result]))
    print("\nConfusion matrix [0 non-rumor, 1 rumor]")
    print(cm)

    return result

In [185]:
node_names = list(data.node_names)

idx_to_node = {i: str(node_names[i]) for i in range(len(node_names))}
node_to_idx = {str(node_names[i]): i for i in range(len(node_names))}

controllable_message_indices = [
    i for i in range(data.num_nodes)
    if bool(data.test_mask[i]) and bool(data.message_mask[i])
]

target_rumor_indices = [
    i for i in controllable_message_indices
    if int(data.y[i].item()) == 1
]

controllable_user_indices = []

for node, attrs in G_processed.nodes(data=True):
    node_str = str(node)
    if attrs.get("node_type") == "user" and node_str in node_to_idx:
        controllable_user_indices.append(node_to_idx[node_str])

controllable_message_nodes = [idx_to_node[i] for i in controllable_message_indices]
target_rumor_nodes = [idx_to_node[i] for i in target_rumor_indices]
controllable_user_nodes = [idx_to_node[i] for i in controllable_user_indices]

print("Controllable test message nodes:", len(controllable_message_nodes))
print("Target rumor nodes:", len(target_rumor_nodes))
print("Controllable user nodes:", len(controllable_user_nodes))
print("Target rumor label check:", Counter(data.y[target_rumor_indices].detach().cpu().tolist()))
print("Example target rumor nodes:", target_rumor_nodes[:5])
print("Example controllable user nodes:", controllable_user_nodes[:5])

Controllable test message nodes: 344
Target rumor nodes: 86
Controllable user nodes: 1008
Target rumor label check: Counter({1: 86})
Example target rumor nodes: ['message_500354773133299713', 'message_500359983301939200', 'message_500284494201757696', 'message_500327120770301952', 'message_500361847070269441']
Example controllable user nodes: ['user_176934206', 'user_14321959', 'user_85867265', 'user_16253142', 'user_1066796935']


In [186]:
def graph_to_edge_index(G, node_to_idx):
    edge_set = set()

    for u, v in G.edges():
        u = str(u)
        v = str(v)

        if u in node_to_idx and v in node_to_idx:
            u_idx = node_to_idx[u]
            v_idx = node_to_idx[v]

            edge_set.add((u_idx, v_idx))
            edge_set.add((v_idx, u_idx))

    edge_index = torch.tensor(list(edge_set), dtype=torch.long).t().contiguous()
    return edge_index


def create_attacked_data(data, G_attacked, node_to_idx):
    data_attacked = copy.deepcopy(data).cpu()
    data_attacked.edge_index = graph_to_edge_index(G_attacked, node_to_idx)
    return data_attacked.to(device)

In [187]:
model = model.to(device)
data = data.to(device)

data_clean = create_attacked_data(data, G_processed, node_to_idx)

clean_result = evaluate_for_attack(
    model=model,
    data=data_clean,
    mask=data_clean.test_mask,
    name="Clean graph"
)

data = data_clean

print("Model device:", next(model.parameters()).device)
print("Data device:", data.x.device)


Clean graph
          name  accuracy  nonrumor_precision  nonrumor_recall  nonrumor_f1  \
0  Clean graph  0.741279            0.933333         0.705426     0.803532   

   rumor_precision  rumor_recall  rumor_f1  avg_rumor_prob_on_true_rumors  \
0         0.489933      0.848837  0.621277                       0.515993   

   avg_rumor_prob_on_true_nonrumors  target_rumor_loss  predicted_nonrumor  \
0                          0.494519           0.662006                 195   

   predicted_rumor  
0              149  

Confusion matrix [0 non-rumor, 1 rumor]
[[182  76]
 [ 13  73]]
Model device: cuda:0
Data device: cuda:0


In [188]:
T = 5

G_attacked = copy.deepcopy(G_processed)

G_pr = G_processed.to_undirected()
pagerank_scores = nx.pagerank(G_pr)

user_nonrumor_counts = defaultdict(int)
user_rumor_counts = defaultdict(int)

for u, v, attrs in G_processed.edges(data=True):
    u = str(u)
    v = str(v)

    if u.startswith("user_") and v.startswith("message_"):
        user_node = u
        message_node = v
    elif v.startswith("user_") and u.startswith("message_"):
        user_node = v
        message_node = u
    else:
        continue

    if message_node not in G_processed.nodes:
        continue

    message_label = G_processed.nodes[message_node].get("label")

    if message_label == 0:
        user_nonrumor_counts[user_node] += 1
    elif message_label == 1:
        user_rumor_counts[user_node] += 1

normal_user_scores = {}

for user_node in controllable_user_nodes:
    nonrumor_count = user_nonrumor_counts[user_node]
    rumor_count = user_rumor_counts[user_node]
    pr_score = pagerank_scores.get(user_node, 0)

    normal_user_scores[user_node] = (nonrumor_count + 1) / (rumor_count + 1) * pr_score

ranked_users = sorted(
    controllable_user_nodes,
    key=lambda node: normal_user_scores.get(node, 0),
    reverse=True
)

ranked_target_rumors = sorted(
    target_rumor_nodes,
    key=lambda node: pagerank_scores.get(node, 0),
    reverse=True
)

added_edges = []

user_pointer = 0
rumor_pointer = 0

while len(added_edges) < T and user_pointer < len(ranked_users) and rumor_pointer < len(ranked_target_rumors):
    user_node = ranked_users[user_pointer]
    rumor_node = ranked_target_rumors[rumor_pointer]

    if not G_attacked.has_edge(user_node, rumor_node) and not G_attacked.has_edge(rumor_node, user_node):
        G_attacked.add_edge(user_node, rumor_node)
        added_edges.append((user_node, rumor_node))

    user_pointer += 1

    if user_pointer >= len(ranked_users):
        user_pointer = 0
        rumor_pointer += 1

print("Added edges:", len(added_edges))
print("Added edge list:")
print(added_edges)

print("Original graph edges:", G_processed.number_of_edges())
print("Attacked graph edges:", G_attacked.number_of_edges())

Added edges: 5
Added edge list:
[('user_14090948', 'message_498305825341845504'), ('user_14849562', 'message_498305825341845504'), ('user_279390084', 'message_498305825341845504'), ('user_7768402', 'message_498305825341845504'), ('user_183341278', 'message_498305825341845504')]
Original graph edges: 4401
Attacked graph edges: 4406


In [189]:
data_attacked = create_attacked_data(data, G_attacked, node_to_idx)

attacked_result = evaluate_for_attack(
    model=model,
    data=data_attacked,
    mask=data_attacked.test_mask,
    name="PageRank attacked graph T=5"
)

comparison_attack_df = pd.DataFrame([clean_result, attacked_result])
comparison_attack_df


PageRank attacked graph T=5
                          name  accuracy  nonrumor_precision  nonrumor_recall  \
0  PageRank attacked graph T=5  0.744186            0.938144         0.705426   

   nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0      0.80531         0.493333      0.860465  0.627119   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                       0.516085                          0.494529   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0           0.661821                 194              150  

Confusion matrix [0 non-rumor, 1 rumor]
[[182  76]
 [ 12  74]]


,name,accuracy,nonrumor_precision,nonrumor_recall,nonrumor_f1,rumor_precision,rumor_recall,rumor_f1,avg_rumor_prob_on_true_rumors,avg_rumor_prob_on_true_nonrumors,target_rumor_loss,predicted_nonrumor,predicted_rumor
0,Clean graph,0.741279,0.933333,0.705426,0.803532,0.489933,0.848837,0.621277,0.515993,0.494519,0.662006,195,149
1,PageRank attacked graph T=5,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516085,0.494529,0.661821,194,150


In [190]:
#40
T = 20

G_attacked_T20 = copy.deepcopy(G_processed)

G_pr = G_processed.to_undirected()
pagerank_scores = nx.pagerank(G_pr)

user_nonrumor_counts = defaultdict(int)
user_rumor_counts = defaultdict(int)

for u, v, attrs in G_processed.edges(data=True):
    u = str(u)
    v = str(v)

    if u.startswith("user_") and v.startswith("message_"):
        user_node = u
        message_node = v
    elif v.startswith("user_") and u.startswith("message_"):
        user_node = v
        message_node = u
    else:
        continue

    if message_node not in G_processed.nodes:
        continue

    message_label = G_processed.nodes[message_node].get("label")

    if message_label == 0:
        user_nonrumor_counts[user_node] += 1
    elif message_label == 1:
        user_rumor_counts[user_node] += 1

normal_user_scores = {}

for user_node in controllable_user_nodes:
    nonrumor_count = user_nonrumor_counts[user_node]
    rumor_count = user_rumor_counts[user_node]
    pr_score = pagerank_scores.get(user_node, 0)

    normal_user_scores[user_node] = (nonrumor_count + 1) / (rumor_count + 1) * pr_score

ranked_users = sorted(
    controllable_user_nodes,
    key=lambda node: normal_user_scores.get(node, 0),
    reverse=True
)

ranked_target_rumors = sorted(
    target_rumor_nodes,
    key=lambda node: pagerank_scores.get(node, 0),
    reverse=True
)

added_edges_T20 = []

user_pointer = 0
rumor_pointer = 0

while len(added_edges_T20) < T and user_pointer < len(ranked_users) and rumor_pointer < len(ranked_target_rumors):
    user_node = ranked_users[user_pointer]
    rumor_node = ranked_target_rumors[rumor_pointer]

    if not G_attacked_T20.has_edge(user_node, rumor_node) and not G_attacked_T20.has_edge(rumor_node, user_node):
        G_attacked_T20.add_edge(user_node, rumor_node)
        added_edges_T20.append((user_node, rumor_node))

    user_pointer += 1

    if user_pointer >= len(ranked_users):
        user_pointer = 0
        rumor_pointer += 1

print("Added edges:", len(added_edges_T20))
print("Original graph edges:", G_processed.number_of_edges())
print("Attacked graph edges:", G_attacked_T20.number_of_edges())
print("First 10 added edges:")
print(added_edges_T20[:10])

Added edges: 20
Original graph edges: 4401
Attacked graph edges: 4421
First 10 added edges:
[('user_14090948', 'message_498305825341845504'), ('user_14849562', 'message_498305825341845504'), ('user_279390084', 'message_498305825341845504'), ('user_7768402', 'message_498305825341845504'), ('user_183341278', 'message_498305825341845504'), ('user_13393052', 'message_498305825341845504'), ('user_19468168', 'message_498305825341845504'), ('user_24165761', 'message_498305825341845504'), ('user_1291770157', 'message_498305825341845504'), ('user_5513002', 'message_498305825341845504')]


In [191]:
data_attacked_T20 = create_attacked_data(data, G_attacked_T20, node_to_idx)

attacked_result_T20 = evaluate_for_attack(
    model=model,
    data=data_attacked_T20,
    mask=data_attacked_T20.test_mask,
    name="PageRank attacked graph T=20"
)

comparison_attack_df = pd.DataFrame([
    clean_result,
    attacked_result,
    attacked_result_T20
])

comparison_attack_df


PageRank attacked graph T=20
                           name  accuracy  nonrumor_precision  \
0  PageRank attacked graph T=20  0.744186            0.938144   

   nonrumor_recall  nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0         0.705426      0.80531         0.493333      0.860465  0.627119   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                       0.516146                          0.494521   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0           0.661701                 194              150  

Confusion matrix [0 non-rumor, 1 rumor]
[[182  76]
 [ 12  74]]


,name,accuracy,nonrumor_precision,nonrumor_recall,nonrumor_f1,rumor_precision,rumor_recall,rumor_f1,avg_rumor_prob_on_true_rumors,avg_rumor_prob_on_true_nonrumors,target_rumor_loss,predicted_nonrumor,predicted_rumor
0,Clean graph,0.741279,0.933333,0.705426,0.803532,0.489933,0.848837,0.621277,0.515993,0.494519,0.662006,195,149
1,PageRank attacked graph T=5,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516085,0.494529,0.661821,194,150
2,PageRank attacked graph T=20,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516146,0.494521,0.661701,194,150


In [192]:
def get_target_rumor_loss(model, data, target_rumor_indices):
    model.eval()

    with torch.no_grad():
        out = model(data)
        target_indices_tensor = torch.tensor(
            target_rumor_indices,
            dtype=torch.long,
            device=data.x.device
        )

        loss = F.cross_entropy(
            out[target_indices_tensor],
            data.y[target_indices_tensor],
            reduction="mean"
        ).item()

    return loss


def greedy_attack_step(model, data, G_current, candidate_edges, node_to_idx, target_rumor_indices):
    best_edge = None
    best_loss = -float("inf")

    for user_node, rumor_node in candidate_edges:
        if G_current.has_edge(user_node, rumor_node) or G_current.has_edge(rumor_node, user_node):
            continue

        G_temp = copy.deepcopy(G_current)
        G_temp.add_edge(user_node, rumor_node)

        data_temp = create_attacked_data(data, G_temp, node_to_idx)
        temp_loss = get_target_rumor_loss(model, data_temp, target_rumor_indices)

        if temp_loss > best_loss:
            best_loss = temp_loss
            best_edge = (user_node, rumor_node)

    return best_edge, best_loss

In [193]:
T = 5
candidate_user_count = 30
candidate_rumor_count = 30

G_greedy_attacked = copy.deepcopy(G_processed)

G_pr = G_processed.to_undirected()
pagerank_scores = nx.pagerank(G_pr)

ranked_users = sorted(
    controllable_user_nodes,
    key=lambda node: pagerank_scores.get(node, 0),
    reverse=True
)

ranked_target_rumors = sorted(
    target_rumor_nodes,
    key=lambda node: pagerank_scores.get(node, 0),
    reverse=True
)

candidate_users = ranked_users[:candidate_user_count]
candidate_rumors = ranked_target_rumors[:candidate_rumor_count]

candidate_edges = [
    (user_node, rumor_node)
    for user_node in candidate_users
    for rumor_node in candidate_rumors
]

greedy_added_edges = []
greedy_losses = []

for step in range(T):
    best_edge, best_loss = greedy_attack_step(
        model=model,
        data=data,
        G_current=G_greedy_attacked,
        candidate_edges=candidate_edges,
        node_to_idx=node_to_idx,
        target_rumor_indices=target_rumor_indices
    )

    if best_edge is None:
        break

    G_greedy_attacked.add_edge(best_edge[0], best_edge[1])
    greedy_added_edges.append(best_edge)
    greedy_losses.append(best_loss)

    candidate_edges = [
        edge for edge in candidate_edges
        if edge != best_edge
    ]

    print("Step:", step + 1)
    print("Added edge:", best_edge)
    print("Best target rumor loss:", best_loss)

print("Total greedy added edges:", len(greedy_added_edges))
print("Original graph edges:", G_processed.number_of_edges())
print("Greedy attacked graph edges:", G_greedy_attacked.number_of_edges())

Step: 1
Added edge: ('user_13393052', 'message_500359377585704961')
Best target rumor loss: 0.6626209020614624
Step: 2
Added edge: ('user_13393052', 'message_500278858156085248')
Best target rumor loss: 0.6630634665489197
Step: 3
Added edge: ('user_341908197', 'message_500301519854379008')
Best target rumor loss: 0.6634966731071472
Step: 4
Added edge: ('user_954124423', 'message_500278944881725440')
Best target rumor loss: 0.6638891696929932
Step: 5
Added edge: ('user_14849562', 'message_500359088757555200')
Best target rumor loss: 0.6642611026763916
Total greedy added edges: 5
Original graph edges: 4401
Greedy attacked graph edges: 4406


In [194]:
data_greedy_attacked = create_attacked_data(data, G_greedy_attacked, node_to_idx)

greedy_attacked_result = evaluate_for_attack(
    model=model,
    data=data_greedy_attacked,
    mask=data_greedy_attacked.test_mask,
    name="Greedy target-loss attacked graph T=5"
)

comparison_attack_df = pd.DataFrame([
    clean_result,
    attacked_result,
    attacked_result_T20,
    greedy_attacked_result
])

comparison_attack_df


Greedy target-loss attacked graph T=5
                                    name  accuracy  nonrumor_precision  \
0  Greedy target-loss attacked graph T=5  0.738372             0.93299   

   nonrumor_recall  nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0          0.70155     0.800885         0.486667      0.848837  0.618644   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                       0.514801                          0.494604   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0           0.664261                 194              150  

Confusion matrix [0 non-rumor, 1 rumor]
[[181  77]
 [ 13  73]]


,name,accuracy,nonrumor_precision,nonrumor_recall,nonrumor_f1,rumor_precision,rumor_recall,rumor_f1,avg_rumor_prob_on_true_rumors,avg_rumor_prob_on_true_nonrumors,target_rumor_loss,predicted_nonrumor,predicted_rumor
0,Clean graph,0.741279,0.933333,0.705426,0.803532,0.489933,0.848837,0.621277,0.515993,0.494519,0.662006,195,149
1,PageRank attacked graph T=5,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516085,0.494529,0.661821,194,150
2,PageRank attacked graph T=20,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516146,0.494521,0.661701,194,150
3,Greedy target-loss attacked graph T=5,0.738372,0.932990,0.701550,0.800885,0.486667,0.848837,0.618644,0.514801,0.494604,0.664261,194,150


In [195]:
T = 20
candidate_user_count = 30
candidate_rumor_count = 30

G_greedy_attacked_T20 = copy.deepcopy(G_processed)

G_pr = G_processed.to_undirected()
pagerank_scores = nx.pagerank(G_pr)

ranked_users = sorted(
    controllable_user_nodes,
    key=lambda node: pagerank_scores.get(node, 0),
    reverse=True
)

ranked_target_rumors = sorted(
    target_rumor_nodes,
    key=lambda node: pagerank_scores.get(node, 0),
    reverse=True
)

candidate_users = ranked_users[:candidate_user_count]
candidate_rumors = ranked_target_rumors[:candidate_rumor_count]

candidate_edges = [
    (user_node, rumor_node)
    for user_node in candidate_users
    for rumor_node in candidate_rumors
]

greedy_added_edges_T20 = []
greedy_losses_T20 = []

for step in range(T):
    best_edge, best_loss = greedy_attack_step(
        model=model,
        data=data,
        G_current=G_greedy_attacked_T20,
        candidate_edges=candidate_edges,
        node_to_idx=node_to_idx,
        target_rumor_indices=target_rumor_indices
    )

    if best_edge is None:
        break

    G_greedy_attacked_T20.add_edge(best_edge[0], best_edge[1])
    greedy_added_edges_T20.append(best_edge)
    greedy_losses_T20.append(best_loss)

    candidate_edges = [
        edge for edge in candidate_edges
        if edge != best_edge
    ]

    print("Step:", step + 1)
    print("Added edge:", best_edge)
    print("Best target rumor loss:", best_loss)

print("Total greedy added edges:", len(greedy_added_edges_T20))
print("Original graph edges:", G_processed.number_of_edges())
print("Greedy attacked graph edges:", G_greedy_attacked_T20.number_of_edges())

Step: 1
Added edge: ('user_13393052', 'message_500359377585704961')
Best target rumor loss: 0.6626209020614624
Step: 2
Added edge: ('user_13393052', 'message_500278858156085248')
Best target rumor loss: 0.6630634665489197
Step: 3
Added edge: ('user_341908197', 'message_500301519854379008')
Best target rumor loss: 0.6634966731071472
Step: 4
Added edge: ('user_954124423', 'message_500278944881725440')
Best target rumor loss: 0.6638891696929932
Step: 5
Added edge: ('user_14849562', 'message_500359088757555200')
Best target rumor loss: 0.6642611026763916
Step: 6
Added edge: ('user_19468168', 'message_500361847070269441')
Best target rumor loss: 0.6645885109901428
Step: 7
Added edge: ('user_902528816', 'message_500360095738626049')
Best target rumor loss: 0.6648959517478943
Step: 8
Added edge: ('user_13393052', 'message_500275363558064128')
Best target rumor loss: 0.6651783585548401
Step: 9
Added edge: ('user_124227622', 'message_500281131057811456')
Best target rumor loss: 0.66544276475906

In [196]:
data_greedy_attacked_T20 = create_attacked_data(data, G_greedy_attacked_T20, node_to_idx)

greedy_attacked_result_T20 = evaluate_for_attack(
    model=model,
    data=data_greedy_attacked_T20,
    mask=data_greedy_attacked_T20.test_mask,
    name="Greedy target-loss attacked graph T=20"
)

comparison_attack_df = pd.DataFrame([
    clean_result,
    attacked_result,
    attacked_result_T20,
    greedy_attacked_result,
    greedy_attacked_result_T20
])

comparison_attack_df


Greedy target-loss attacked graph T=20
                                     name  accuracy  nonrumor_precision  \
0  Greedy target-loss attacked graph T=20  0.735465            0.932642   

   nonrumor_recall  nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0         0.697674     0.798226         0.483444      0.848837  0.616034   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                        0.51324                          0.494678   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0           0.667264                 193              151  

Confusion matrix [0 non-rumor, 1 rumor]
[[180  78]
 [ 13  73]]


,name,accuracy,nonrumor_precision,nonrumor_recall,nonrumor_f1,rumor_precision,rumor_recall,rumor_f1,avg_rumor_prob_on_true_rumors,avg_rumor_prob_on_true_nonrumors,target_rumor_loss,predicted_nonrumor,predicted_rumor
0,Clean graph,0.741279,0.933333,0.705426,0.803532,0.489933,0.848837,0.621277,0.515993,0.494519,0.662006,195,149
1,PageRank attacked graph T=5,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516085,0.494529,0.661821,194,150
2,PageRank attacked graph T=20,0.744186,0.938144,0.705426,0.805310,0.493333,0.860465,0.627119,0.516146,0.494521,0.661701,194,150
3,Greedy target-loss attacked graph T=5,0.738372,0.932990,0.701550,0.800885,0.486667,0.848837,0.618644,0.514801,0.494604,0.664261,194,150
4,Greedy target-loss attacked graph T=20,0.735465,0.932642,0.697674,0.798226,0.483444,0.848837,0.616034,0.513240,0.494678,0.667264,193,151


In [197]:



#PR-BCD




In [198]:
#50
def forward_with_edge_weight(model, data, edge_index, edge_weight):
    x = data.x
    x = model.conv1(x, edge_index, edge_weight=edge_weight)
    x = F.relu(x)
    x = F.dropout(x, p=0.5, training=False)
    x = model.conv2(x, edge_index, edge_weight=edge_weight)
    return x

In [199]:
def compute_state_metrics(model, data_state, mask, target_rumor_indices):
    model.eval()

    with torch.no_grad():
        out = model(data_state)
        probs = F.softmax(out, dim=1)
        pred = out.argmax(dim=1)

    y_true = data_state.y[mask].detach().cpu().numpy()
    y_pred = pred[mask].detach().cpu().numpy()

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0
    )

    target_indices_tensor = torch.tensor(
        target_rumor_indices,
        dtype=torch.long,
        device=data_state.x.device
    )

    target_loss = F.cross_entropy(
        out[target_indices_tensor],
        data_state.y[target_indices_tensor],
        reduction="mean"
    ).item()

    avg_target_rumor_prob = probs[target_indices_tensor, 1].mean().item()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "rumor_precision": precision[1],
        "rumor_recall": recall[1],
        "rumor_f1": f1[1],
        "target_rumor_loss": target_loss,
        "avg_target_rumor_prob": avg_target_rumor_prob,
        "predicted_rumor": int((y_pred == 1).sum()),
        "predicted_nonrumor": int((y_pred == 0).sum())
    }


def build_attack_trajectory(attack_name, edge_sequence, G_base, data, node_to_idx, target_rumor_indices):
    G_current = copy.deepcopy(G_base)
    rows = []

    for step, edge in enumerate(edge_sequence, start=1):
        user_node, rumor_node = edge

        data_before = create_attacked_data(data, G_current, node_to_idx)
        before_metrics = compute_state_metrics(
            model=model,
            data_state=data_before,
            mask=data_before.test_mask,
            target_rumor_indices=target_rumor_indices
        )

        G_current.add_edge(user_node, rumor_node)

        data_after = create_attacked_data(data, G_current, node_to_idx)
        after_metrics = compute_state_metrics(
            model=model,
            data_state=data_after,
            mask=data_after.test_mask,
            target_rumor_indices=target_rumor_indices
        )

        row = {
            "attack_name": attack_name,
            "trajectory_length": len(edge_sequence),
            "step": step,
            "user_node": user_node,
            "rumor_node": rumor_node,
            "before_accuracy": before_metrics["accuracy"],
            "after_accuracy": after_metrics["accuracy"],
            "accuracy_change": after_metrics["accuracy"] - before_metrics["accuracy"],
            "before_rumor_f1": before_metrics["rumor_f1"],
            "after_rumor_f1": after_metrics["rumor_f1"],
            "rumor_f1_change": after_metrics["rumor_f1"] - before_metrics["rumor_f1"],
            "before_target_rumor_loss": before_metrics["target_rumor_loss"],
            "after_target_rumor_loss": after_metrics["target_rumor_loss"],
            "target_rumor_loss_change": after_metrics["target_rumor_loss"] - before_metrics["target_rumor_loss"],
            "before_avg_target_rumor_prob": before_metrics["avg_target_rumor_prob"],
            "after_avg_target_rumor_prob": after_metrics["avg_target_rumor_prob"],
            "avg_target_rumor_prob_change": after_metrics["avg_target_rumor_prob"] - before_metrics["avg_target_rumor_prob"],
            "before_predicted_rumor": before_metrics["predicted_rumor"],
            "after_predicted_rumor": after_metrics["predicted_rumor"]
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [200]:



#Full PR-BCD Attack




In [201]:
with open(graph_path, "rb") as f:
    G_prbcd_base = pickle.load(f)

if isinstance(G_prbcd_base, dict):
    G_prbcd_base = G_prbcd_base["graph"]

print("Base graph nodes:", G_prbcd_base.number_of_nodes())
print("Base graph edges:", G_prbcd_base.number_of_edges())

data_prbcd_base = create_attacked_data(data, G_prbcd_base, node_to_idx)

full_prbcd_clean_result = evaluate_for_attack(
    model=model,
    data=data_prbcd_base,
    mask=data_prbcd_base.test_mask,
    name="Full PR-BCD clean baseline"
)

full_prbcd_candidate_edges = []

for user_node in controllable_user_nodes:
    for rumor_node in target_rumor_nodes:
        if not G_prbcd_base.has_edge(user_node, rumor_node) and not G_prbcd_base.has_edge(rumor_node, user_node):
            full_prbcd_candidate_edges.append((user_node, rumor_node))

print("Full PR-BCD candidate edges:", len(full_prbcd_candidate_edges))
print("Example candidates:")
print(full_prbcd_candidate_edges[:10])

full_prbcd_config = {
    "budget_T5": 5,
    "budget_T20": 20,
    "block_size": 5000,
    "optimization_steps": 100,
    "learning_rate": 200.0,
    "candidate_edges": len(full_prbcd_candidate_edges)
}

full_prbcd_config

Base graph nodes: 2706
Base graph edges: 4401

Full PR-BCD clean baseline
                         name  accuracy  nonrumor_precision  nonrumor_recall  \
0  Full PR-BCD clean baseline  0.741279            0.933333         0.705426   

   nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0     0.803532         0.489933      0.848837  0.621277   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                       0.515993                          0.494519   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0           0.662006                 195              149  

Confusion matrix [0 non-rumor, 1 rumor]
[[182  76]
 [ 13  73]]
Full PR-BCD candidate edges: 86602
Example candidates:
[('user_176934206', 'message_500354773133299713'), ('user_176934206', 'message_500359983301939200'), ('user_176934206', 'message_500284494201757696'), ('user_176934206', 'message_500327120770301952'), ('user_176934206', 'message_500361847070269441'), ('user_17693420

{'budget_T5': 5,
 'budget_T20': 20,
 'block_size': 5000,
 'optimization_steps': 100,
 'learning_rate': 200.0,
 'candidate_edges': 86602}

In [202]:
def project_to_budget_box_simplex(values, budget, max_iter=60):
    values = torch.clamp(values, min=0.0, max=1.0)

    if values.sum() <= budget:
        return values

    lower = values.min() - 1.0
    upper = values.max()

    for _ in range(max_iter):
        middle = (lower + upper) / 2.0
        projected = torch.clamp(values - middle, min=0.0, max=1.0)

        if projected.sum() > budget:
            lower = middle
        else:
            upper = middle

    projected = torch.clamp(values - upper, min=0.0, max=1.0)
    return projected

In [203]:
def build_block_edge_index_and_weights(data, candidate_edges, block_indices, p_values, node_to_idx):
    original_edge_index = data.edge_index.detach().cpu()

    block_directed_edges = []
    block_weights = []

    for candidate_idx in block_indices:
        user_node, rumor_node = candidate_edges[int(candidate_idx)]

        user_idx = node_to_idx[user_node]
        rumor_idx = node_to_idx[rumor_node]

        weight_value = p_values[int(candidate_idx)].item()

        block_directed_edges.append((user_idx, rumor_idx))
        block_directed_edges.append((rumor_idx, user_idx))

        block_weights.append(weight_value)
        block_weights.append(weight_value)

    block_edge_index = torch.tensor(
        block_directed_edges,
        dtype=torch.long
    ).t().contiguous()

    combined_edge_index = torch.cat(
        [original_edge_index, block_edge_index],
        dim=1
    ).to(device)

    original_weights = torch.ones(
        original_edge_index.shape[1],
        dtype=torch.float,
        device=device
    )

    block_weights = torch.tensor(
        block_weights,
        dtype=torch.float,
        device=device
    )

    edge_weight = torch.cat(
        [original_weights, block_weights],
        dim=0
    )

    edge_weight.requires_grad = True

    return combined_edge_index, edge_weight, original_edge_index.shape[1]


def full_prbcd_optimize(
    model,
    data,
    candidate_edges,
    node_to_idx,
    target_rumor_indices,
    budget,
    block_size=5000,
    optimization_steps=100,
    learning_rate=200.0,
    seed=42
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    num_candidates = len(candidate_edges)

    p_values = torch.zeros(
        num_candidates,
        dtype=torch.float
    )

    target_indices_tensor = torch.tensor(
        target_rumor_indices,
        dtype=torch.long,
        device=device
    )

    prbcd_history = []

    for step in range(1, optimization_steps + 1):
        active_indices = torch.where(p_values > 0)[0].tolist()

        remaining_budget_for_block = max(block_size - len(active_indices), 0)

        random_indices = np.random.choice(
            num_candidates,
            size=min(remaining_budget_for_block, num_candidates),
            replace=False
        ).tolist()

        block_indices = list(set(active_indices + random_indices))

        combined_edge_index, edge_weight, original_edge_count = build_block_edge_index_and_weights(
            data=data,
            candidate_edges=candidate_edges,
            block_indices=block_indices,
            p_values=p_values,
            node_to_idx=node_to_idx
        )

        model.eval()
        out = forward_with_edge_weight(
            model=model,
            data=data.to(device),
            edge_index=combined_edge_index,
            edge_weight=edge_weight
        )

        loss = F.cross_entropy(
            out[target_indices_tensor],
            data.y[target_indices_tensor],
            reduction="mean"
        )

        model.zero_grad()
        loss.backward()

        directed_grads = edge_weight.grad[original_edge_count:].detach().cpu()

        candidate_gradients = torch.zeros(len(block_indices), dtype=torch.float)

        for i in range(len(block_indices)):
            candidate_gradients[i] = directed_grads[2 * i] + directed_grads[2 * i + 1]

        block_indices_tensor = torch.tensor(block_indices, dtype=torch.long)

        p_values[block_indices_tensor] = p_values[block_indices_tensor] + learning_rate * candidate_gradients

        p_values = project_to_budget_box_simplex(
            p_values,
            budget=budget
        )

        prbcd_history.append({
            "step": step,
            "target_loss": loss.item(),
            "p_sum": p_values.sum().item(),
            "active_edges": int((p_values > 1e-6).sum().item()),
            "max_p": p_values.max().item()
        })

        if step % 10 == 0 or step == 1:
            print(
                f"Step {step:03d} | "
                f"Loss: {loss.item():.6f} | "
                f"p_sum: {p_values.sum().item():.4f} | "
                f"active_edges: {int((p_values > 1e-6).sum().item())} | "
                f"max_p: {p_values.max().item():.4f}"
            )

    prbcd_history_df = pd.DataFrame(prbcd_history)

    return p_values, prbcd_history_df

In [204]:
full_prbcd_p_values_T5, full_prbcd_history_T5_df = full_prbcd_optimize(
    model=model,
    data=data_prbcd_base,
    candidate_edges=full_prbcd_candidate_edges,
    node_to_idx=node_to_idx,
    target_rumor_indices=target_rumor_indices,
    budget=5,
    block_size=5000,
    optimization_steps=100,
    learning_rate=200.0,
    seed=42
)

full_prbcd_history_T5_df.tail()

Step 001 | Loss: 0.662006 | p_sum: 5.0000 | active_edges: 266 | max_p: 0.0888
Step 010 | Loss: 0.664293 | p_sum: 5.0000 | active_edges: 267 | max_p: 0.1079
Step 020 | Loss: 0.664561 | p_sum: 5.0000 | active_edges: 228 | max_p: 0.1309
Step 030 | Loss: 0.664370 | p_sum: 5.0000 | active_edges: 224 | max_p: 0.1438
Step 040 | Loss: 0.664374 | p_sum: 5.0000 | active_edges: 211 | max_p: 0.1381
Step 050 | Loss: 0.664712 | p_sum: 5.0000 | active_edges: 202 | max_p: 0.1555
Step 060 | Loss: 0.664564 | p_sum: 5.0000 | active_edges: 215 | max_p: 0.1775
Step 070 | Loss: 0.664689 | p_sum: 5.0000 | active_edges: 213 | max_p: 0.1878
Step 080 | Loss: 0.664680 | p_sum: 5.0000 | active_edges: 204 | max_p: 0.1840
Step 090 | Loss: 0.664771 | p_sum: 5.0000 | active_edges: 209 | max_p: 0.1909
Step 100 | Loss: 0.664465 | p_sum: 5.0000 | active_edges: 204 | max_p: 0.1862


,step,target_loss,p_sum,active_edges,max_p
95,96,0.664500,4.999999,215,0.184549
96,97,0.665287,5.000000,114,0.167876
97,98,0.664793,4.999999,201,0.186442
98,99,0.665362,4.999999,108,0.160452
99,100,0.664465,5.000000,204,0.186225


In [205]:
def discretize_prbcd_solution(p_values, candidate_edges, budget):
    top_values, top_indices = torch.topk(p_values, k=budget)

    selected_edges = []
    selected_rows = []

    for rank, candidate_idx in enumerate(top_indices.tolist(), start=1):
        user_node, rumor_node = candidate_edges[candidate_idx]
        p_value = top_values[rank - 1].item()

        selected_edges.append((user_node, rumor_node))

        selected_rows.append({
            "rank": rank,
            "candidate_index": candidate_idx,
            "user_node": user_node,
            "rumor_node": rumor_node,
            "p_value": p_value
        })

    selected_edges_df = pd.DataFrame(selected_rows)

    return selected_edges, selected_edges_df


full_prbcd_edges_T5, full_prbcd_edges_T5_df = discretize_prbcd_solution(
    p_values=full_prbcd_p_values_T5,
    candidate_edges=full_prbcd_candidate_edges,
    budget=5
)

print("Selected Full PR-BCD T=5 edges:")
display(full_prbcd_edges_T5_df)

G_full_prbcd_T5 = copy.deepcopy(G_prbcd_base)

for user_node, rumor_node in full_prbcd_edges_T5:
    G_full_prbcd_T5.add_edge(user_node, rumor_node)

print("Original graph edges:", G_prbcd_base.number_of_edges())
print("Full PR-BCD T=5 attacked graph edges:", G_full_prbcd_T5.number_of_edges())

data_full_prbcd_T5 = create_attacked_data(
    data,
    G_full_prbcd_T5,
    node_to_idx
)

full_prbcd_result_T5 = evaluate_for_attack(
    model=model,
    data=data_full_prbcd_T5,
    mask=data_full_prbcd_T5.test_mask,
    name="Full PR-BCD attacked graph T=5"
)

full_prbcd_result_T5

Selected Full PR-BCD T=5 edges:


,rank,candidate_index,user_node,rumor_node,p_value
0,1,46664,user_622969430,message_500278944881725440,0.186225
1,2,10323,user_15446531,message_500278858156085248,0.158628
2,3,11865,user_15907183,message_500359088757555200,0.146016
3,4,22442,user_97474887,message_500290867681587200,0.119770
4,5,46679,user_622969430,message_500360248767840256,0.111195


Original graph edges: 4401
Full PR-BCD T=5 attacked graph edges: 4406

Full PR-BCD attacked graph T=5
                             name  accuracy  nonrumor_precision  \
0  Full PR-BCD attacked graph T=5  0.741279            0.933333   

   nonrumor_recall  nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0         0.705426     0.803532         0.489933      0.848837  0.621277   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                       0.514999                          0.494594   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0           0.663889                 195              149  

Confusion matrix [0 non-rumor, 1 rumor]
[[182  76]
 [ 13  73]]


{'name': 'Full PR-BCD attacked graph T=5',
 'accuracy': 0.7412790697674418,
 'nonrumor_precision': np.float64(0.9333333333333333),
 'nonrumor_recall': np.float64(0.7054263565891473),
 'nonrumor_f1': np.float64(0.8035320088300221),
 'rumor_precision': np.float64(0.4899328859060403),
 'rumor_recall': np.float64(0.8488372093023255),
 'rumor_f1': np.float64(0.6212765957446809),
 'avg_rumor_prob_on_true_rumors': 0.5149993896484375,
 'avg_rumor_prob_on_true_nonrumors': 0.49459370970726013,
 'target_rumor_loss': 0.6638888716697693,
 'predicted_nonrumor': 195,
 'predicted_rumor': 149}

In [206]:
full_prbcd_p_values_T20, full_prbcd_history_T20_df = full_prbcd_optimize(
    model=model,
    data=data_prbcd_base,
    candidate_edges=full_prbcd_candidate_edges,
    node_to_idx=node_to_idx,
    target_rumor_indices=target_rumor_indices,
    budget=20,
    block_size=5000,
    optimization_steps=100,
    learning_rate=200.0,
    seed=42
)

full_prbcd_history_T20_df.tail()

Step 001 | Loss: 0.662006 | p_sum: 20.0000 | active_edges: 867 | max_p: 0.1164
Step 010 | Loss: 0.670572 | p_sum: 20.0000 | active_edges: 938 | max_p: 0.1668
Step 020 | Loss: 0.670169 | p_sum: 20.0000 | active_edges: 851 | max_p: 0.1804
Step 030 | Loss: 0.670281 | p_sum: 20.0000 | active_edges: 772 | max_p: 0.2005
Step 040 | Loss: 0.670610 | p_sum: 20.0000 | active_edges: 702 | max_p: 0.2137
Step 050 | Loss: 0.670715 | p_sum: 20.0000 | active_edges: 730 | max_p: 0.2294
Step 060 | Loss: 0.670914 | p_sum: 20.0000 | active_edges: 642 | max_p: 0.2492
Step 070 | Loss: 0.671011 | p_sum: 20.0000 | active_edges: 615 | max_p: 0.2673
Step 080 | Loss: 0.671307 | p_sum: 20.0000 | active_edges: 619 | max_p: 0.2828
Step 090 | Loss: 0.671082 | p_sum: 20.0000 | active_edges: 601 | max_p: 0.2831
Step 100 | Loss: 0.671269 | p_sum: 20.0000 | active_edges: 558 | max_p: 0.2969


,step,target_loss,p_sum,active_edges,max_p
95,96,0.671211,19.999998,595,0.294462
96,97,0.671479,20.000000,524,0.295519
97,98,0.671337,20.000000,581,0.294423
98,99,0.671393,19.999998,518,0.294249
99,100,0.671269,20.000000,558,0.296886


In [207]:
full_prbcd_edges_T20, full_prbcd_edges_T20_df = discretize_prbcd_solution(
    p_values=full_prbcd_p_values_T20,
    candidate_edges=full_prbcd_candidate_edges,
    budget=20
)

print("Selected Full PR-BCD T=20 edges:")
display(full_prbcd_edges_T20_df)

G_full_prbcd_T20 = copy.deepcopy(G_prbcd_base)

for user_node, rumor_node in full_prbcd_edges_T20:
    G_full_prbcd_T20.add_edge(user_node, rumor_node)

print("Original graph edges:", G_prbcd_base.number_of_edges())
print("Full PR-BCD T=20 attacked graph edges:", G_full_prbcd_T20.number_of_edges())

data_full_prbcd_T20 = create_attacked_data(
    data,
    G_full_prbcd_T20,
    node_to_idx
)

full_prbcd_result_T20 = evaluate_for_attack(
    model=model,
    data=data_full_prbcd_T20,
    mask=data_full_prbcd_T20.test_mask,
    name="Full PR-BCD attacked graph T=20"
)

full_prbcd_result_T20

Selected Full PR-BCD T=20 edges:


,rank,candidate_index,user_node,rumor_node,p_value
0,1,47603,user_1546896846,message_500263680026869760,0.296886
1,2,4091,user_13393052,message_500360095738626049,0.246198
2,3,22495,user_97474887,message_500283291493494784,0.212637
3,4,15444,user_17974481,message_500283877567770624,0.210562
4,5,10359,user_15446531,message_500364250461003777,0.198742
5,6,47615,user_1546896846,message_500284091305299968,0.176217
6,7,46648,user_622969430,message_500280847912955904,0.171979
7,8,45892,user_14606079,message_500278944881725440,0.168195
8,9,79005,user_69940000,message_500280249629036544,0.166252
9,10,11887,user_15907183,message_500297358908071936,0.160703


Original graph edges: 4401
Full PR-BCD T=20 attacked graph edges: 4421

Full PR-BCD attacked graph T=20
                              name  accuracy  nonrumor_precision  \
0  Full PR-BCD attacked graph T=20  0.735465            0.923858   

   nonrumor_recall  nonrumor_f1  rumor_precision  rumor_recall  rumor_f1  \
0         0.705426          0.8         0.482993      0.825581  0.609442   

   avg_rumor_prob_on_true_rumors  avg_rumor_prob_on_true_nonrumors  \
0                       0.512705                          0.494682   

   target_rumor_loss  predicted_nonrumor  predicted_rumor  
0            0.66835                 197              147  

Confusion matrix [0 non-rumor, 1 rumor]
[[182  76]
 [ 15  71]]


{'name': 'Full PR-BCD attacked graph T=20',
 'accuracy': 0.7354651162790697,
 'nonrumor_precision': np.float64(0.9238578680203046),
 'nonrumor_recall': np.float64(0.7054263565891473),
 'nonrumor_f1': np.float64(0.8),
 'rumor_precision': np.float64(0.48299319727891155),
 'rumor_recall': np.float64(0.8255813953488372),
 'rumor_f1': np.float64(0.6094420600858369),
 'avg_rumor_prob_on_true_rumors': 0.5127049088478088,
 'avg_rumor_prob_on_true_nonrumors': 0.494681715965271,
 'target_rumor_loss': 0.6683502197265625,
 'predicted_nonrumor': 197,
 'predicted_rumor': 147}

In [208]:
attack_summary_df = pd.DataFrame([
    {
        "attack": "PageRank T=5",
        "accuracy_change": attacked_result["accuracy"] - clean_result["accuracy"],
        "rumor_f1_change": attacked_result["rumor_f1"] - clean_result["rumor_f1"],
        "avg_rumor_prob_change": attacked_result["avg_rumor_prob_on_true_rumors"] - clean_result["avg_rumor_prob_on_true_rumors"],
        "target_rumor_loss_change": attacked_result["target_rumor_loss"] - clean_result["target_rumor_loss"],
    },
    {
        "attack": "PageRank T=20",
        "accuracy_change": attacked_result_T20["accuracy"] - clean_result["accuracy"],
        "rumor_f1_change": attacked_result_T20["rumor_f1"] - clean_result["rumor_f1"],
        "avg_rumor_prob_change": attacked_result_T20["avg_rumor_prob_on_true_rumors"] - clean_result["avg_rumor_prob_on_true_rumors"],
        "target_rumor_loss_change": attacked_result_T20["target_rumor_loss"] - clean_result["target_rumor_loss"],
    },
    {
        "attack": "Greedy T=5",
        "accuracy_change": greedy_attacked_result["accuracy"] - clean_result["accuracy"],
        "rumor_f1_change": greedy_attacked_result["rumor_f1"] - clean_result["rumor_f1"],
        "avg_rumor_prob_change": greedy_attacked_result["avg_rumor_prob_on_true_rumors"] - clean_result["avg_rumor_prob_on_true_rumors"],
        "target_rumor_loss_change": greedy_attacked_result["target_rumor_loss"] - clean_result["target_rumor_loss"],
    },
    {
        "attack": "Greedy T=20",
        "accuracy_change": greedy_attacked_result_T20["accuracy"] - clean_result["accuracy"],
        "rumor_f1_change": greedy_attacked_result_T20["rumor_f1"] - clean_result["rumor_f1"],
        "avg_rumor_prob_change": greedy_attacked_result_T20["avg_rumor_prob_on_true_rumors"] - clean_result["avg_rumor_prob_on_true_rumors"],
        "target_rumor_loss_change": greedy_attacked_result_T20["target_rumor_loss"] - clean_result["target_rumor_loss"],
    },
    {
        "attack": "Full PR-BCD T=5",
        "accuracy_change": full_prbcd_result_T5["accuracy"] - clean_result["accuracy"],
        "rumor_f1_change": full_prbcd_result_T5["rumor_f1"] - clean_result["rumor_f1"],
        "avg_rumor_prob_change": full_prbcd_result_T5["avg_rumor_prob_on_true_rumors"] - clean_result["avg_rumor_prob_on_true_rumors"],
        "target_rumor_loss_change": full_prbcd_result_T5["target_rumor_loss"] - clean_result["target_rumor_loss"],
    },
    {
        "attack": "Full PR-BCD T=20",
        "accuracy_change": full_prbcd_result_T20["accuracy"] - clean_result["accuracy"],
        "rumor_f1_change": full_prbcd_result_T20["rumor_f1"] - clean_result["rumor_f1"],
        "avg_rumor_prob_change": full_prbcd_result_T20["avg_rumor_prob_on_true_rumors"] - clean_result["avg_rumor_prob_on_true_rumors"],
        "target_rumor_loss_change": full_prbcd_result_T20["target_rumor_loss"] - clean_result["target_rumor_loss"],
    }
])

attack_summary_df

,attack,accuracy_change,rumor_f1_change,avg_rumor_prob_change,target_rumor_loss_change
0,PageRank T=5,0.002907,0.005842,0.000092,-0.000185
1,PageRank T=20,0.002907,0.005842,0.000153,-0.000306
2,Greedy T=5,-0.002907,-0.002633,-0.001192,0.002255
3,Greedy T=20,-0.005814,-0.005243,-0.002752,0.005257
4,Full PR-BCD T=5,0.000000,0.000000,-0.000993,0.001883
5,Full PR-BCD T=20,-0.005814,-0.011835,-0.003288,0.006344


In [209]:
results_dir = "/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs"
os.makedirs(results_dir, exist_ok=True)

full_prbcd_summary_path = os.path.join(results_dir, "attack_summary_with_full_prbcd.csv")
full_prbcd_comparison_path = os.path.join(results_dir, "comparison_attack_with_full_prbcd.csv")

full_prbcd_edges_t5_path = os.path.join(results_dir, "full_prbcd_edges_T5.csv")
full_prbcd_edges_t20_path = os.path.join(results_dir, "full_prbcd_edges_T20.csv")

full_prbcd_history_t5_path = os.path.join(results_dir, "full_prbcd_history_T5.csv")
full_prbcd_history_t20_path = os.path.join(results_dir, "full_prbcd_history_T20.csv")

comparison_attack_df = pd.DataFrame([
    clean_result,
    attacked_result,
    attacked_result_T20,
    greedy_attacked_result,
    greedy_attacked_result_T20,
    full_prbcd_result_T5,
    full_prbcd_result_T20
])

attack_summary_df.to_csv(full_prbcd_summary_path, index=False)
comparison_attack_df.to_csv(full_prbcd_comparison_path, index=False)

full_prbcd_edges_T5_df.to_csv(full_prbcd_edges_t5_path, index=False)
full_prbcd_edges_T20_df.to_csv(full_prbcd_edges_t20_path, index=False)

full_prbcd_history_T5_df.to_csv(full_prbcd_history_t5_path, index=False)
full_prbcd_history_T20_df.to_csv(full_prbcd_history_t20_path, index=False)

print("Saved Full PR-BCD results:")
print(full_prbcd_summary_path)
print(full_prbcd_comparison_path)
print(full_prbcd_edges_t5_path)
print(full_prbcd_edges_t20_path)
print(full_prbcd_history_t5_path)
print(full_prbcd_history_t20_path)

Saved Full PR-BCD results:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/attack_summary_with_full_prbcd.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/comparison_attack_with_full_prbcd.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/full_prbcd_edges_T5.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/full_prbcd_edges_T20.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/full_prbcd_history_T5.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/full_prbcd_history_T20.csv


In [210]:
expert_trajectory_specs = [
    {
        "attack_name": "PageRank_T5",
        "T": 5,
        "edge_sequence": added_edges
    },
    {
        "attack_name": "PageRank_T20",
        "T": 20,
        "edge_sequence": added_edges_T20
    },
    {
        "attack_name": "Greedy_GC_RWCS_T5",
        "T": 5,
        "edge_sequence": greedy_added_edges
    },
    {
        "attack_name": "Greedy_GC_RWCS_T20",
        "T": 20,
        "edge_sequence": greedy_added_edges_T20
    },
    {
        "attack_name": "Full_PR_BCD_T5",
        "T": 5,
        "edge_sequence": full_prbcd_edges_T5
    },
    {
        "attack_name": "Full_PR_BCD_T20",
        "T": 20,
        "edge_sequence": full_prbcd_edges_T20
    }
]

expert_trajectories = {}
expert_trajectory_dfs = []

for spec in expert_trajectory_specs:
    attack_name = spec["attack_name"]
    edge_sequence = spec["edge_sequence"]

    print("\n==============================")
    print("Building trajectory:", attack_name)
    print("Number of edges:", len(edge_sequence))
    print("==============================")

    trajectory_df = build_attack_trajectory(
        attack_name=attack_name,
        edge_sequence=edge_sequence,
        G_base=G_processed,
        data=data,
        node_to_idx=node_to_idx,
        target_rumor_indices=target_rumor_indices
    )

    trajectory_df["T"] = spec["T"]

    expert_trajectories[attack_name] = trajectory_df
    expert_trajectory_dfs.append(trajectory_df)

expert_trajectory_df = pd.concat(
    expert_trajectory_dfs,
    ignore_index=True
)

print("Total expert state-action samples:", len(expert_trajectory_df))
display(expert_trajectory_df.head())
display(expert_trajectory_df.tail())


Building trajectory: PageRank_T5
Number of edges: 5

Building trajectory: PageRank_T20
Number of edges: 20

Building trajectory: Greedy_GC_RWCS_T5
Number of edges: 5

Building trajectory: Greedy_GC_RWCS_T20
Number of edges: 20

Building trajectory: Full_PR_BCD_T5
Number of edges: 5

Building trajectory: Full_PR_BCD_T20
Number of edges: 20
Total expert state-action samples: 75


,attack_name,trajectory_length,step,user_node,rumor_node,before_accuracy,after_accuracy,accuracy_change,before_rumor_f1,after_rumor_f1,rumor_f1_change,before_target_rumor_loss,after_target_rumor_loss,target_rumor_loss_change,before_avg_target_rumor_prob,after_avg_target_rumor_prob,avg_target_rumor_prob_change,before_predicted_rumor,after_predicted_rumor,T
0,PageRank_T5,5,1,user_14090948,message_498305825341845504,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662006,0.661951,-0.000055,0.515993,0.516020,0.000027,149,149,5
1,PageRank_T5,5,2,user_14849562,message_498305825341845504,0.741279,0.738372,-0.002907,0.621277,0.618644,-0.002633,0.661951,0.661934,-0.000017,0.516020,0.516028,0.000008,149,150,5
2,PageRank_T5,5,3,user_279390084,message_498305825341845504,0.738372,0.738372,0.000000,0.618644,0.618644,0.000000,0.661934,0.661889,-0.000046,0.516028,0.516051,0.000023,150,150,5
3,PageRank_T5,5,4,user_7768402,message_498305825341845504,0.738372,0.744186,0.005814,0.618644,0.627119,0.008475,0.661889,0.661856,-0.000033,0.516051,0.516067,0.000017,150,150,5
4,PageRank_T5,5,5,user_183341278,message_498305825341845504,0.744186,0.744186,0.000000,0.627119,0.627119,0.000000,0.661856,0.661821,-0.000034,0.516067,0.516085,0.000017,150,150,5


,attack_name,trajectory_length,step,user_node,rumor_node,before_accuracy,after_accuracy,accuracy_change,before_rumor_f1,after_rumor_f1,rumor_f1_change,before_target_rumor_loss,after_target_rumor_loss,target_rumor_loss_change,before_avg_target_rumor_prob,after_avg_target_rumor_prob,avg_target_rumor_prob_change,before_predicted_rumor,after_predicted_rumor,T
70,Full_PR_BCD_T20,20,16,user_97474887,message_500309381930844160,0.735465,0.735465,0.0,0.609442,0.609442,0.0,0.667231,0.667498,0.000266,0.513286,0.513146,-0.000140,147,147,20
71,Full_PR_BCD_T20,20,17,user_18197912,message_500360248767840256,0.735465,0.735465,0.0,0.609442,0.609442,0.0,0.667498,0.667800,0.000303,0.513146,0.512987,-0.000158,147,147,20
72,Full_PR_BCD_T20,20,18,user_87359651,message_500360248767840256,0.735465,0.735465,0.0,0.609442,0.609442,0.0,0.667800,0.667921,0.000121,0.512987,0.512925,-0.000062,147,147,20
73,Full_PR_BCD_T20,20,19,user_18523773,message_500284377490665472,0.735465,0.735465,0.0,0.609442,0.609442,0.0,0.667921,0.668212,0.000291,0.512925,0.512776,-0.000150,147,147,20
74,Full_PR_BCD_T20,20,20,user_363767963,message_500278944881725440,0.735465,0.735465,0.0,0.609442,0.609442,0.0,0.668212,0.668350,0.000138,0.512776,0.512705,-0.000071,147,147,20


In [211]:
expert_trajectory_summary_rows = []

for attack_name, trajectory_df in expert_trajectories.items():
    if len(trajectory_df) == 0:
        continue

    first_row = trajectory_df.iloc[0]
    last_row = trajectory_df.iloc[-1]

    start_loss = first_row["before_target_rumor_loss"]
    final_loss = last_row["after_target_rumor_loss"]

    start_acc = first_row["before_accuracy"]
    final_acc = last_row["after_accuracy"]

    start_f1 = first_row["before_rumor_f1"]
    final_f1 = last_row["after_rumor_f1"]

    expert_trajectory_summary_rows.append({
        "attack_name": attack_name,
        "T": int(trajectory_df["T"].iloc[0]),
        "num_steps": len(trajectory_df),

        # Your code's practical attack direction:
        # if this is positive, the attack increased target rumor loss.
        "target_loss_increase": final_loss - start_loss,

        # Paper-style notation:
        # the paper defines ΔLA = LA(0) - LA(T).
        "paper_delta_LA_clean_minus_final": start_loss - final_loss,

        "start_target_rumor_loss": start_loss,
        "final_target_rumor_loss": final_loss,

        "accuracy_change": final_acc - start_acc,
        "rumor_f1_change": final_f1 - start_f1,

        "start_accuracy": start_acc,
        "final_accuracy": final_acc,
        "start_rumor_f1": start_f1,
        "final_rumor_f1": final_f1
    })

expert_trajectory_summary_df = pd.DataFrame(expert_trajectory_summary_rows)

display(expert_trajectory_summary_df.sort_values(
    by="target_loss_increase",
    ascending=False
))

,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,start_target_rumor_loss,final_target_rumor_loss,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1
5,Full_PR_BCD_T20,20,20,0.006344,-0.006344,0.662006,0.668350,-0.005814,-0.011835,0.741279,0.735465,0.621277,0.609442
3,Greedy_GC_RWCS_T20,20,20,0.005257,-0.005257,0.662006,0.667264,-0.005814,-0.005243,0.741279,0.735465,0.621277,0.616034
2,Greedy_GC_RWCS_T5,5,5,0.002255,-0.002255,0.662006,0.664261,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644
4,Full_PR_BCD_T5,5,5,0.001883,-0.001883,0.662006,0.663889,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277
0,PageRank_T5,5,5,-0.000185,0.000185,0.662006,0.661821,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119
1,PageRank_T20,20,20,-0.000306,0.000306,0.662006,0.661701,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119


In [212]:
expert_dir = os.path.join(results_dir, "expert_trajectories")
os.makedirs(expert_dir, exist_ok=True)

expert_trajectory_path = os.path.join(
    expert_dir,
    "all_expert_trajectories.csv"
)

expert_trajectory_summary_path = os.path.join(
    expert_dir,
    "expert_trajectory_summary.csv"
)

expert_trajectory_pickle_path = os.path.join(
    expert_dir,
    "expert_trajectories.pkl"
)

expert_trajectory_df.to_csv(expert_trajectory_path, index=False)
expert_trajectory_summary_df.to_csv(expert_trajectory_summary_path, index=False)

with open(expert_trajectory_pickle_path, "wb") as f:
    pickle.dump(expert_trajectories, f)

print("Saved expert trajectory files:")
print(expert_trajectory_path)
print(expert_trajectory_summary_path)
print(expert_trajectory_pickle_path)

Saved expert trajectory files:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/expert_trajectories/all_expert_trajectories.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/expert_trajectories/expert_trajectory_summary.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/expert_trajectories/expert_trajectories.pkl


In [213]:
print("Expert trajectory dataframe shape:", expert_trajectory_df.shape)
print("Columns:")
print(expert_trajectory_df.columns.tolist())

print("\nSamples per attack:")
display(expert_trajectory_df.groupby("attack_name").size().reset_index(name="num_samples"))

print("\nTrajectory summary:")
display(expert_trajectory_summary_df)

Expert trajectory dataframe shape: (75, 20)
Columns:
['attack_name', 'trajectory_length', 'step', 'user_node', 'rumor_node', 'before_accuracy', 'after_accuracy', 'accuracy_change', 'before_rumor_f1', 'after_rumor_f1', 'rumor_f1_change', 'before_target_rumor_loss', 'after_target_rumor_loss', 'target_rumor_loss_change', 'before_avg_target_rumor_prob', 'after_avg_target_rumor_prob', 'avg_target_rumor_prob_change', 'before_predicted_rumor', 'after_predicted_rumor', 'T']

Samples per attack:


,attack_name,num_samples
0,Full_PR_BCD_T20,20
1,Full_PR_BCD_T5,5
2,Greedy_GC_RWCS_T20,20
3,Greedy_GC_RWCS_T5,5
4,PageRank_T20,20
5,PageRank_T5,5



Trajectory summary:


,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,start_target_rumor_loss,final_target_rumor_loss,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1
0,PageRank_T5,5,5,-0.000185,0.000185,0.662006,0.661821,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119
1,PageRank_T20,20,20,-0.000306,0.000306,0.662006,0.661701,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119
2,Greedy_GC_RWCS_T5,5,5,0.002255,-0.002255,0.662006,0.664261,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644
3,Greedy_GC_RWCS_T20,20,20,0.005257,-0.005257,0.662006,0.667264,-0.005814,-0.005243,0.741279,0.735465,0.621277,0.616034
4,Full_PR_BCD_T5,5,5,0.001883,-0.001883,0.662006,0.663889,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277
5,Full_PR_BCD_T20,20,20,0.006344,-0.006344,0.662006,0.668350,-0.005814,-0.011835,0.741279,0.735465,0.621277,0.609442


In [214]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn.functional as F
import os
import pickle

def safe_get_node_attr(G, node, possible_keys, default=None):
    """
    Safely get a node attribute even if different parts of the notebook
    used different attribute names.
    """
    attrs = G.nodes[node] if node in G.nodes else {}
    for key in possible_keys:
        if key in attrs:
            return attrs[key]
    return default


def infer_node_type(G, node):
    """
    Robust node type detector.
    Works with either node attributes or string prefixes.
    """
    node_type = safe_get_node_attr(
        G,
        node,
        ["node_type", "type", "category", "kind"],
        default=None
    )

    if node_type is not None:
        node_type = str(node_type).lower()
        if "user" in node_type or "author" in node_type:
            return "user"
        if "tweet" in node_type or "message" in node_type or "rumor" in node_type or "nonrumor" in node_type:
            return "message"
        if "comment" in node_type or "review" in node_type:
            return "comment"

    node_str = str(node).lower()

    if node_str.startswith("user") or "user" in node_str or "author" in node_str:
        return "user"
    if node_str.startswith("tweet") or "tweet" in node_str or "message" in node_str:
        return "message"
    if node_str.startswith("comment") or "comment" in node_str or "review" in node_str:
        return "comment"

    return "unknown"


def infer_is_rumor(G, node):
    """
    Robust rumor-label detector for message/tweet nodes.
    Returns:
        1 if rumor
        0 if non-rumor
        np.nan if unknown / not message
    """
    attrs = G.nodes[node] if node in G.nodes else {}

    for key in ["label", "y", "rumor_label", "is_rumor", "class"]:
        if key in attrs:
            value = attrs[key]

            if isinstance(value, str):
                value_lower = value.lower()
                if value_lower in ["rumor", "true", "1"]:
                    return 1
                if value_lower in ["nonrumor", "non-rumor", "false", "0"]:
                    return 0

            if value in [0, 1]:
                return int(value)

    node_str = str(node).lower()
    if "nonrumor" in node_str or "non-rumor" in node_str:
        return 0
    if "rumor" in node_str:
        return 1

    return np.nan


def get_model_rumor_probabilities(model, data, node_to_idx, device):
    """
    Returns a dictionary:
        node_name -> probability of rumor class
    Assumes class 1 is rumor, consistent with your current attack code.
    """
    model.eval()

    data_device = data.to(device)
    model = model.to(device)

    with torch.no_grad():
        out = model(data_device.x, data_device.edge_index)
        probs = F.softmax(out, dim=1)[:, 1].detach().cpu().numpy()

    idx_to_node = {idx: node for node, idx in node_to_idx.items()}

    prob_by_node = {}
    for idx, prob in enumerate(probs):
        node = idx_to_node[idx]
        prob_by_node[node] = float(prob)

    return prob_by_node


def count_neighbor_types(G, node):
    """
    Counts basic neighbor composition around a node.
    """
    if node not in G:
        return {
            "n_neighbors": 0,
            "n_user_neighbors": 0,
            "n_message_neighbors": 0,
            "n_comment_neighbors": 0,
            "n_rumor_neighbors": 0,
            "n_nonrumor_neighbors": 0
        }

    neighbors = list(G.neighbors(node))

    n_user = 0
    n_message = 0
    n_comment = 0
    n_rumor = 0
    n_nonrumor = 0

    for nb in neighbors:
        nb_type = infer_node_type(G, nb)

        if nb_type == "user":
            n_user += 1
        elif nb_type == "message":
            n_message += 1
            rumor_value = infer_is_rumor(G, nb)
            if rumor_value == 1:
                n_rumor += 1
            elif rumor_value == 0:
                n_nonrumor += 1
        elif nb_type == "comment":
            n_comment += 1

    return {
        "n_neighbors": len(neighbors),
        "n_user_neighbors": n_user,
        "n_message_neighbors": n_message,
        "n_comment_neighbors": n_comment,
        "n_rumor_neighbors": n_rumor,
        "n_nonrumor_neighbors": n_nonrumor
    }


print("Feature helper functions are ready.")

Feature helper functions are ready.


In [215]:
def build_expert_action_feature_table(
    expert_trajectory_df,
    G_base,
    data,
    node_to_idx,
    model,
    device
):
    model.eval()

    data_device = data.to(device)
    model = model.to(device)

    with torch.no_grad():
        try:
            out = model(data_device)
        except TypeError:
            out = model(data_device.x, data_device.edge_index)

        probs = F.softmax(out, dim=1)[:, 1].detach().cpu().numpy()

    idx_to_node = {idx: node for node, idx in node_to_idx.items()}

    clean_prob_by_node = {}
    for idx, prob in enumerate(probs):
        node = idx_to_node[idx]
        clean_prob_by_node[node] = float(prob)

    pagerank_dict = nx.pagerank(G_base.to_undirected())

    feature_rows = []

    grouped = expert_trajectory_df.sort_values(
        ["attack_name", "step"]
    ).groupby("attack_name")

    for attack_name, traj_df in grouped:
        G_state = G_base.copy()

        T_value = int(traj_df["T"].iloc[0])

        user_attack_counter = {}
        rumor_attack_counter = {}

        for _, row in traj_df.iterrows():
            step = int(row["step"])
            user_node = row["user_node"]
            rumor_node = row["rumor_node"]

            user_degree = G_state.degree(user_node) if user_node in G_state else 0
            rumor_degree = G_state.degree(rumor_node) if rumor_node in G_state else 0

            user_pr = pagerank_dict.get(user_node, 0.0)
            rumor_pr = pagerank_dict.get(rumor_node, 0.0)

            user_neighbor_counts = count_neighbor_types(G_state, user_node)
            rumor_neighbor_counts = count_neighbor_types(G_state, rumor_node)

            edge_exists_before = int(G_state.has_edge(user_node, rumor_node))

            user_attack_degree = user_attack_counter.get(user_node, 0)
            rumor_attack_degree = rumor_attack_counter.get(rumor_node, 0)

            user_type = infer_node_type(G_state, user_node)
            rumor_type = infer_node_type(G_state, rumor_node)

            rumor_label = infer_is_rumor(G_state, rumor_node)

            action_features = {
                "attack_name": attack_name,
                "T": T_value,
                "step": step,
                "step_norm": step / max(T_value, 1),

                "user_node": user_node,
                "rumor_node": rumor_node,

                "is_expert": 1,

                "user_degree": user_degree,
                "rumor_degree": rumor_degree,
                "degree_sum": user_degree + rumor_degree,
                "degree_abs_diff": abs(user_degree - rumor_degree),

                "user_pagerank": user_pr,
                "rumor_pagerank": rumor_pr,
                "pagerank_sum": user_pr + rumor_pr,
                "pagerank_abs_diff": abs(user_pr - rumor_pr),

                "edge_exists_before": edge_exists_before,
                "user_attack_degree": user_attack_degree,
                "rumor_attack_degree": rumor_attack_degree,
                "user_attack_degree_norm": user_attack_degree / max(T_value, 1),
                "rumor_attack_degree_norm": rumor_attack_degree / max(T_value, 1),

                "user_n_neighbors": user_neighbor_counts["n_neighbors"],
                "user_n_message_neighbors": user_neighbor_counts["n_message_neighbors"],
                "user_n_rumor_neighbors": user_neighbor_counts["n_rumor_neighbors"],
                "user_n_nonrumor_neighbors": user_neighbor_counts["n_nonrumor_neighbors"],

                "rumor_n_neighbors": rumor_neighbor_counts["n_neighbors"],
                "rumor_n_user_neighbors": rumor_neighbor_counts["n_user_neighbors"],
                "rumor_n_comment_neighbors": rumor_neighbor_counts["n_comment_neighbors"],

                "rumor_clean_prob": clean_prob_by_node.get(rumor_node, 0.0),
                "before_avg_target_rumor_prob": row["before_avg_target_rumor_prob"],
                "after_avg_target_rumor_prob": row["after_avg_target_rumor_prob"],
                "avg_target_rumor_prob_change": row["avg_target_rumor_prob_change"],

                "before_target_rumor_loss": row["before_target_rumor_loss"],
                "after_target_rumor_loss": row["after_target_rumor_loss"],
                "target_rumor_loss_change": row["target_rumor_loss_change"],

                "before_accuracy": row["before_accuracy"],
                "after_accuracy": row["after_accuracy"],
                "accuracy_change": row["accuracy_change"],

                "before_rumor_f1": row["before_rumor_f1"],
                "after_rumor_f1": row["after_rumor_f1"],
                "rumor_f1_change": row["rumor_f1_change"],

                "user_type_is_user": int(user_type == "user"),
                "rumor_type_is_message": int(rumor_type == "message"),
                "rumor_label_is_rumor": 0 if pd.isna(rumor_label) else int(rumor_label)
            }

            feature_rows.append(action_features)

            if user_node in G_state and rumor_node in G_state:
                G_state.add_edge(user_node, rumor_node)

            user_attack_counter[user_node] = user_attack_counter.get(user_node, 0) + 1
            rumor_attack_counter[rumor_node] = rumor_attack_counter.get(rumor_node, 0) + 1

    return pd.DataFrame(feature_rows)


expert_action_feature_df = build_expert_action_feature_table(
    expert_trajectory_df=expert_trajectory_df,
    G_base=G_processed,
    data=data,
    node_to_idx=node_to_idx,
    model=model,
    device=device
)

print("Expert action feature table shape:", expert_action_feature_df.shape)
display(expert_action_feature_df.head())

Expert action feature table shape: (75, 43)


,attack_name,T,step,step_norm,user_node,rumor_node,is_expert,user_degree,rumor_degree,degree_sum,...,target_rumor_loss_change,before_accuracy,after_accuracy,accuracy_change,before_rumor_f1,after_rumor_f1,rumor_f1_change,user_type_is_user,rumor_type_is_message,rumor_label_is_rumor
0,Full_PR_BCD_T20,20,1,0.05,user_1546896846,message_500263680026869760,1,4,1,5,...,0.000447,0.741279,0.741279,0.0,0.621277,0.621277,0.0,1,1,1
1,Full_PR_BCD_T20,20,2,0.10,user_13393052,message_500360095738626049,1,19,1,20,...,0.000426,0.741279,0.741279,0.0,0.621277,0.621277,0.0,1,1,1
2,Full_PR_BCD_T20,20,3,0.15,user_97474887,message_500283291493494784,1,9,1,10,...,0.000383,0.741279,0.741279,0.0,0.621277,0.621277,0.0,1,1,1
3,Full_PR_BCD_T20,20,4,0.20,user_17974481,message_500283877567770624,1,2,1,3,...,0.000373,0.741279,0.741279,0.0,0.621277,0.621277,0.0,1,1,1
4,Full_PR_BCD_T20,20,5,0.25,user_15446531,message_500364250461003777,1,2,1,3,...,0.000298,0.741279,0.741279,0.0,0.621277,0.621277,0.0,1,1,1


In [216]:
print("Shape:", expert_action_feature_df.shape)

print("\nSamples per attack:")
display(
    expert_action_feature_df
    .groupby("attack_name")
    .size()
    .reset_index(name="num_feature_rows")
)

print("\nMissing values per column:")
missing_summary = expert_action_feature_df.isna().sum()
display(missing_summary[missing_summary > 0])

print("\nAverage target loss change by attack:")
display(
    expert_action_feature_df
    .groupby("attack_name")["target_rumor_loss_change"]
    .mean()
    .reset_index()
    .sort_values("target_rumor_loss_change", ascending=False)
)

print("\nFeature preview:")
display(expert_action_feature_df.head(10))

Shape: (75, 43)

Samples per attack:


,attack_name,num_feature_rows
0,Full_PR_BCD_T20,20
1,Full_PR_BCD_T5,5
2,Greedy_GC_RWCS_T20,20
3,Greedy_GC_RWCS_T5,5
4,PageRank_T20,20
5,PageRank_T5,5



Missing values per column:


,0



Average target loss change by attack:


,attack_name,target_rumor_loss_change
3,Greedy_GC_RWCS_T5,0.000451
1,Full_PR_BCD_T5,0.000377
0,Full_PR_BCD_T20,0.000317
2,Greedy_GC_RWCS_T20,0.000263
4,PageRank_T20,-0.000015
5,PageRank_T5,-0.000037



Feature preview:


,attack_name,T,step,step_norm,user_node,rumor_node,is_expert,user_degree,rumor_degree,degree_sum,...,target_rumor_loss_change,before_accuracy,after_accuracy,accuracy_change,before_rumor_f1,after_rumor_f1,rumor_f1_change,user_type_is_user,rumor_type_is_message,rumor_label_is_rumor
0,Full_PR_BCD_T20,20,1,0.05,user_1546896846,message_500263680026869760,1,4,1,5,...,0.000447,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,1,1,1
1,Full_PR_BCD_T20,20,2,0.10,user_13393052,message_500360095738626049,1,19,1,20,...,0.000426,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,1,1,1
2,Full_PR_BCD_T20,20,3,0.15,user_97474887,message_500283291493494784,1,9,1,10,...,0.000383,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,1,1,1
3,Full_PR_BCD_T20,20,4,0.20,user_17974481,message_500283877567770624,1,2,1,3,...,0.000373,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,1,1,1
4,Full_PR_BCD_T20,20,5,0.25,user_15446531,message_500364250461003777,1,2,1,3,...,0.000298,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,1,1,1
5,Full_PR_BCD_T20,20,6,0.30,user_1546896846,message_500284091305299968,1,5,1,6,...,0.000301,0.741279,0.738372,-0.002907,0.621277,0.615385,-0.005892,1,1,1
6,Full_PR_BCD_T20,20,7,0.35,user_622969430,message_500280847912955904,1,1,1,2,...,0.000349,0.738372,0.735465,-0.002907,0.615385,0.609442,-0.005943,1,1,1
7,Full_PR_BCD_T20,20,8,0.40,user_14606079,message_500278944881725440,1,2,1,3,...,0.000437,0.735465,0.735465,0.000000,0.609442,0.609442,0.000000,1,1,1
8,Full_PR_BCD_T20,20,9,0.45,user_69940000,message_500280249629036544,1,5,1,6,...,0.000379,0.735465,0.735465,0.000000,0.609442,0.609442,0.000000,1,1,1
9,Full_PR_BCD_T20,20,10,0.50,user_15907183,message_500297358908071936,1,9,1,10,...,0.000337,0.735465,0.735465,0.000000,0.609442,0.609442,0.000000,1,1,1


In [217]:
feature_dir = os.path.join(results_dir, "irl_features")
os.makedirs(feature_dir, exist_ok=True)

expert_action_feature_path = os.path.join(
    feature_dir,
    "expert_action_features.csv"
)

expert_action_feature_df.to_csv(expert_action_feature_path, index=False)

print("Saved expert action features:")
print(expert_action_feature_path)

Saved expert action features:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/irl_features/expert_action_features.csv


In [218]:
def get_candidate_users_and_rumors(G):
    candidate_users = []
    candidate_rumors = []

    for node in G.nodes():
        node_type = infer_node_type(G, node)
        rumor_value = infer_is_rumor(G, node)

        if node_type == "user":
            candidate_users.append(node)

        if node_type == "message" and rumor_value == 1:
            candidate_rumors.append(node)

    return candidate_users, candidate_rumors


def build_action_feature_row(
    attack_name,
    T_value,
    step,
    user_node,
    rumor_node,
    is_expert,
    G_state,
    pagerank_dict,
    clean_prob_by_node,
    user_attack_counter,
    rumor_attack_counter,
    source_row=None
):
    user_degree = G_state.degree(user_node) if user_node in G_state else 0
    rumor_degree = G_state.degree(rumor_node) if rumor_node in G_state else 0

    user_pr = pagerank_dict.get(user_node, 0.0)
    rumor_pr = pagerank_dict.get(rumor_node, 0.0)

    user_neighbor_counts = count_neighbor_types(G_state, user_node)
    rumor_neighbor_counts = count_neighbor_types(G_state, rumor_node)

    edge_exists_before = int(G_state.has_edge(user_node, rumor_node))

    user_attack_degree = user_attack_counter.get(user_node, 0)
    rumor_attack_degree = rumor_attack_counter.get(rumor_node, 0)

    user_type = infer_node_type(G_state, user_node)
    rumor_type = infer_node_type(G_state, rumor_node)

    rumor_label = infer_is_rumor(G_state, rumor_node)

    if source_row is None:
        before_avg_target_rumor_prob = np.nan
        after_avg_target_rumor_prob = np.nan
        avg_target_rumor_prob_change = np.nan
        before_target_rumor_loss = np.nan
        after_target_rumor_loss = np.nan
        target_rumor_loss_change = np.nan
        before_accuracy = np.nan
        after_accuracy = np.nan
        accuracy_change = np.nan
        before_rumor_f1 = np.nan
        after_rumor_f1 = np.nan
        rumor_f1_change = np.nan
    else:
        before_avg_target_rumor_prob = source_row["before_avg_target_rumor_prob"]
        after_avg_target_rumor_prob = np.nan
        avg_target_rumor_prob_change = np.nan
        before_target_rumor_loss = source_row["before_target_rumor_loss"]
        after_target_rumor_loss = np.nan
        target_rumor_loss_change = np.nan
        before_accuracy = source_row["before_accuracy"]
        after_accuracy = np.nan
        accuracy_change = np.nan
        before_rumor_f1 = source_row["before_rumor_f1"]
        after_rumor_f1 = np.nan
        rumor_f1_change = np.nan

    return {
        "attack_name": attack_name,
        "T": T_value,
        "step": step,
        "step_norm": step / max(T_value, 1),

        "user_node": user_node,
        "rumor_node": rumor_node,

        "is_expert": is_expert,

        "user_degree": user_degree,
        "rumor_degree": rumor_degree,
        "degree_sum": user_degree + rumor_degree,
        "degree_abs_diff": abs(user_degree - rumor_degree),

        "user_pagerank": user_pr,
        "rumor_pagerank": rumor_pr,
        "pagerank_sum": user_pr + rumor_pr,
        "pagerank_abs_diff": abs(user_pr - rumor_pr),

        "edge_exists_before": edge_exists_before,
        "user_attack_degree": user_attack_degree,
        "rumor_attack_degree": rumor_attack_degree,
        "user_attack_degree_norm": user_attack_degree / max(T_value, 1),
        "rumor_attack_degree_norm": rumor_attack_degree / max(T_value, 1),

        "user_n_neighbors": user_neighbor_counts["n_neighbors"],
        "user_n_message_neighbors": user_neighbor_counts["n_message_neighbors"],
        "user_n_rumor_neighbors": user_neighbor_counts["n_rumor_neighbors"],
        "user_n_nonrumor_neighbors": user_neighbor_counts["n_nonrumor_neighbors"],

        "rumor_n_neighbors": rumor_neighbor_counts["n_neighbors"],
        "rumor_n_user_neighbors": rumor_neighbor_counts["n_user_neighbors"],
        "rumor_n_comment_neighbors": rumor_neighbor_counts["n_comment_neighbors"],

        "rumor_clean_prob": clean_prob_by_node.get(rumor_node, 0.0),
        "before_avg_target_rumor_prob": before_avg_target_rumor_prob,
        "after_avg_target_rumor_prob": after_avg_target_rumor_prob,
        "avg_target_rumor_prob_change": avg_target_rumor_prob_change,

        "before_target_rumor_loss": before_target_rumor_loss,
        "after_target_rumor_loss": after_target_rumor_loss,
        "target_rumor_loss_change": target_rumor_loss_change,

        "before_accuracy": before_accuracy,
        "after_accuracy": after_accuracy,
        "accuracy_change": accuracy_change,

        "before_rumor_f1": before_rumor_f1,
        "after_rumor_f1": after_rumor_f1,
        "rumor_f1_change": rumor_f1_change,

        "user_type_is_user": int(user_type == "user"),
        "rumor_type_is_message": int(rumor_type == "message"),
        "rumor_label_is_rumor": 0 if pd.isna(rumor_label) else int(rumor_label)
    }


def get_similarity_feature_columns(feature_df):
    excluded_columns = [
        "attack_name",
        "user_node",
        "rumor_node",
        "is_expert",
        "after_avg_target_rumor_prob",
        "avg_target_rumor_prob_change",
        "after_target_rumor_loss",
        "target_rumor_loss_change",
        "after_accuracy",
        "accuracy_change",
        "after_rumor_f1",
        "rumor_f1_change"
    ]

    similarity_columns = [
        col for col in feature_df.columns
        if col not in excluded_columns
    ]

    similarity_columns = list(dict.fromkeys(similarity_columns))

    return similarity_columns


def cosine_similarity_between_rows(row_a, row_b, similarity_columns):
    vector_a = pd.to_numeric(row_a[similarity_columns], errors="coerce").fillna(0.0).values.astype(np.float32)
    vector_b = pd.to_numeric(row_b[similarity_columns], errors="coerce").fillna(0.0).values.astype(np.float32)

    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return float(np.dot(vector_a, vector_b) / (norm_a * norm_b))


def build_negative_action_feature_table(
    expert_trajectory_df,
    expert_action_feature_df,
    G_base,
    data,
    node_to_idx,
    model,
    device,
    negatives_per_expert=100,
    similarity_mu=0.8,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)

    model.eval()
    data_device = data.to(device)
    model = model.to(device)

    with torch.no_grad():
        try:
            out = model(data_device)
        except TypeError:
            out = model(data_device.x, data_device.edge_index)

        probs = F.softmax(out, dim=1)[:, 1].detach().cpu().numpy()

    idx_to_node = {idx: node for node, idx in node_to_idx.items()}

    clean_prob_by_node = {}
    for idx, prob in enumerate(probs):
        node = idx_to_node[idx]
        clean_prob_by_node[node] = float(prob)

    pagerank_dict = nx.pagerank(G_base.to_undirected())

    candidate_users, candidate_rumors = get_candidate_users_and_rumors(G_base)

    similarity_columns = get_similarity_feature_columns(expert_action_feature_df)

    negative_rows = []
    sampling_summary_rows = []

    grouped = expert_trajectory_df.sort_values(
        ["attack_name", "step"]
    ).groupby("attack_name")

    for attack_name, traj_df in grouped:
        G_state = G_base.copy()

        T_value = int(traj_df["T"].iloc[0])

        user_attack_counter = {}
        rumor_attack_counter = {}

        for _, row in traj_df.iterrows():
            step = int(row["step"])
            expert_user_node = row["user_node"]
            expert_rumor_node = row["rumor_node"]

            expert_match = expert_action_feature_df[
                (expert_action_feature_df["attack_name"] == attack_name)
                & (expert_action_feature_df["step"] == step)
                & (expert_action_feature_df["user_node"] == expert_user_node)
                & (expert_action_feature_df["rumor_node"] == expert_rumor_node)
            ]

            if len(expert_match) > 0:
                expert_feature_row = expert_match.iloc[0]
            else:
                expert_feature_row = pd.Series(
                    build_action_feature_row(
                        attack_name=attack_name,
                        T_value=T_value,
                        step=step,
                        user_node=expert_user_node,
                        rumor_node=expert_rumor_node,
                        is_expert=1,
                        G_state=G_state,
                        pagerank_dict=pagerank_dict,
                        clean_prob_by_node=clean_prob_by_node,
                        user_attack_counter=user_attack_counter,
                        rumor_attack_counter=rumor_attack_counter,
                        source_row=row
                    )
                )

            used_pairs = set()
            used_pairs.add((expert_user_node, expert_rumor_node))

            collected = 0
            attempts = 0
            max_attempts = negatives_per_expert * 2000
            accepted_similarities = []

            while collected < negatives_per_expert and attempts < max_attempts:
                attempts += 1

                sampling_mode = rng.integers(0, 3)

                if sampling_mode == 0:
                    user_node = str(rng.choice(candidate_users))
                    rumor_node = expert_rumor_node
                elif sampling_mode == 1:
                    user_node = expert_user_node
                    rumor_node = str(rng.choice(candidate_rumors))
                else:
                    user_node = str(rng.choice(candidate_users))
                    rumor_node = str(rng.choice(candidate_rumors))

                if (user_node, rumor_node) in used_pairs:
                    continue

                if user_node not in G_state or rumor_node not in G_state:
                    continue

                if G_state.has_edge(user_node, rumor_node):
                    continue

                candidate_row = build_action_feature_row(
                    attack_name=attack_name,
                    T_value=T_value,
                    step=step,
                    user_node=user_node,
                    rumor_node=rumor_node,
                    is_expert=0,
                    G_state=G_state,
                    pagerank_dict=pagerank_dict,
                    clean_prob_by_node=clean_prob_by_node,
                    user_attack_counter=user_attack_counter,
                    rumor_attack_counter=rumor_attack_counter,
                    source_row=row
                )

                candidate_series = pd.Series(candidate_row)

                similarity = cosine_similarity_between_rows(
                    expert_feature_row,
                    candidate_series,
                    similarity_columns
                )

                if similarity >= similarity_mu:
                    continue

                used_pairs.add((user_node, rumor_node))
                negative_rows.append(candidate_row)
                accepted_similarities.append(similarity)
                collected += 1

            sampling_summary_rows.append({
                "attack_name": attack_name,
                "step": step,
                "requested_negatives": negatives_per_expert,
                "collected_negatives": collected,
                "attempts": attempts,
                "mean_similarity": np.mean(accepted_similarities) if len(accepted_similarities) > 0 else np.nan,
                "max_similarity": np.max(accepted_similarities) if len(accepted_similarities) > 0 else np.nan
            })

            if collected < negatives_per_expert:
                print(
                    "Warning:",
                    attack_name,
                    "step",
                    step,
                    "collected",
                    collected,
                    "negative samples instead of",
                    negatives_per_expert
                )

            if expert_user_node in G_state and expert_rumor_node in G_state:
                G_state.add_edge(expert_user_node, expert_rumor_node)

            user_attack_counter[expert_user_node] = user_attack_counter.get(expert_user_node, 0) + 1
            rumor_attack_counter[expert_rumor_node] = rumor_attack_counter.get(expert_rumor_node, 0) + 1

    negative_df = pd.DataFrame(negative_rows)
    negative_df = negative_df.reindex(columns=expert_action_feature_df.columns)

    sampling_summary_df = pd.DataFrame(sampling_summary_rows)

    return negative_df, sampling_summary_df


negative_action_feature_df, negative_sampling_summary_df = build_negative_action_feature_table(
    expert_trajectory_df=expert_trajectory_df,
    expert_action_feature_df=expert_action_feature_df,
    G_base=G_processed,
    data=data,
    node_to_idx=node_to_idx,
    model=model,
    device=device,
    negatives_per_expert=PAPER_NEGATIVES_PER_EXPERT,
    similarity_mu=PAPER_NEGATIVE_SIMILARITY_MU,
    random_seed=42
)

print("Negative action feature table shape:", negative_action_feature_df.shape)
display(negative_action_feature_df.head())

print("Negative sampling summary:")
display(negative_sampling_summary_df)

print("Minimum collected negatives per expert action:", negative_sampling_summary_df["collected_negatives"].min())
print("Maximum accepted similarity:", negative_sampling_summary_df["max_similarity"].max())

Negative action feature table shape: (7500, 43)


,attack_name,T,step,step_norm,user_node,rumor_node,is_expert,user_degree,rumor_degree,degree_sum,...,target_rumor_loss_change,before_accuracy,after_accuracy,accuracy_change,before_rumor_f1,after_rumor_f1,rumor_f1_change,user_type_is_user,rumor_type_is_message,rumor_label_is_rumor
0,Full_PR_BCD_T20,20,1,0.05,user_124227622,message_500308574703452160,0,32,1,33,...,NaN,0.741279,NaN,NaN,0.621277,NaN,NaN,1,1,1
1,Full_PR_BCD_T20,20,1,0.05,user_119972954,message_500263680026869760,0,20,1,21,...,NaN,0.741279,NaN,NaN,0.621277,NaN,NaN,1,1,1
2,Full_PR_BCD_T20,20,1,0.05,user_23831448,message_500263680026869760,0,19,1,20,...,NaN,0.741279,NaN,NaN,0.621277,NaN,NaN,1,1,1
3,Full_PR_BCD_T20,20,1,0.05,user_153919561,message_500266427840864256,0,17,1,18,...,NaN,0.741279,NaN,NaN,0.621277,NaN,NaN,1,1,1
4,Full_PR_BCD_T20,20,1,0.05,user_14849562,message_500263680026869760,0,39,1,40,...,NaN,0.741279,NaN,NaN,0.621277,NaN,NaN,1,1,1


Negative sampling summary:


,attack_name,step,requested_negatives,collected_negatives,attempts,mean_similarity,max_similarity
0,Full_PR_BCD_T20,1,100,100,10591,0.699196,0.784295
1,Full_PR_BCD_T20,2,100,100,231,0.702786,0.791880
2,Full_PR_BCD_T20,3,100,100,46739,0.756239,0.796694
3,Full_PR_BCD_T20,4,100,100,4237,0.657978,0.798014
4,Full_PR_BCD_T20,5,100,100,3766,0.641533,0.798544
...,...,...,...,...,...,...,...
70,PageRank_T5,1,100,100,241,0.635831,0.797774
71,PageRank_T5,2,100,100,247,0.669502,0.794665
72,PageRank_T5,3,100,100,230,0.653789,0.799961
73,PageRank_T5,4,100,100,274,0.732874,0.798457


Minimum collected negatives per expert action: 100
Maximum accepted similarity: 0.7999808192253113


In [219]:
irl_action_feature_df = pd.concat(
    [expert_action_feature_df, negative_action_feature_df],
    ignore_index=True
)

irl_action_feature_df = irl_action_feature_df.sort_values(
    ["attack_name", "step", "is_expert"],
    ascending=[True, True, False]
).reset_index(drop=True)

print("Combined IRL action feature table shape:", irl_action_feature_df.shape)

print("\nClass counts:")
display(
    irl_action_feature_df["is_expert"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "is_expert", "is_expert": "count"})
)

print("\nSamples per attack and class:")
display(
    irl_action_feature_df
    .groupby(["attack_name", "is_expert"])
    .size()
    .reset_index(name="num_rows")
)

Combined IRL action feature table shape: (7575, 43)

Class counts:


,count,count
0,0,7500
1,1,75



Samples per attack and class:


,attack_name,is_expert,num_rows
0,Full_PR_BCD_T20,0,2000
1,Full_PR_BCD_T20,1,20
2,Full_PR_BCD_T5,0,500
3,Full_PR_BCD_T5,1,5
4,Greedy_GC_RWCS_T20,0,2000
5,Greedy_GC_RWCS_T20,1,20
6,Greedy_GC_RWCS_T5,0,500
7,Greedy_GC_RWCS_T5,1,5
8,PageRank_T20,0,2000
9,PageRank_T20,1,20


In [220]:
non_training_columns = [
    "attack_name",
    "step",
    "user_node",
    "rumor_node",
    "is_expert",

    "after_avg_target_rumor_prob",
    "avg_target_rumor_prob_change",

    "after_target_rumor_loss",
    "target_rumor_loss_change",

    "after_accuracy",
    "accuracy_change",

    "after_rumor_f1",
    "rumor_f1_change"
]

irl_feature_columns = [
    col for col in irl_action_feature_df.columns
    if col not in non_training_columns
]

irl_feature_columns = list(dict.fromkeys(irl_feature_columns))

irl_training_df = irl_action_feature_df[
    ["attack_name", "step", "user_node", "rumor_node", "is_expert"] + irl_feature_columns
].copy()

irl_training_df = irl_training_df.loc[:, ~irl_training_df.columns.duplicated()].copy()

for col in irl_feature_columns:
    irl_training_df[col] = pd.to_numeric(irl_training_df[col], errors="coerce")

irl_training_df[irl_feature_columns] = irl_training_df[irl_feature_columns].fillna(0.0)

print("Number of IRL training features:", len(irl_feature_columns))
print("IRL training dataframe shape:", irl_training_df.shape)

print("\nIRL feature columns:")
print(irl_feature_columns)

print("\nMissing values after fill:")
display(irl_training_df.isna().sum()[irl_training_df.isna().sum() > 0])

display(irl_training_df.head())

Number of IRL training features: 30
IRL training dataframe shape: (7575, 35)

IRL feature columns:
['T', 'step_norm', 'user_degree', 'rumor_degree', 'degree_sum', 'degree_abs_diff', 'user_pagerank', 'rumor_pagerank', 'pagerank_sum', 'pagerank_abs_diff', 'edge_exists_before', 'user_attack_degree', 'rumor_attack_degree', 'user_attack_degree_norm', 'rumor_attack_degree_norm', 'user_n_neighbors', 'user_n_message_neighbors', 'user_n_rumor_neighbors', 'user_n_nonrumor_neighbors', 'rumor_n_neighbors', 'rumor_n_user_neighbors', 'rumor_n_comment_neighbors', 'rumor_clean_prob', 'before_avg_target_rumor_prob', 'before_target_rumor_loss', 'before_accuracy', 'before_rumor_f1', 'user_type_is_user', 'rumor_type_is_message', 'rumor_label_is_rumor']

Missing values after fill:


,0


,attack_name,step,user_node,rumor_node,is_expert,T,step_norm,user_degree,rumor_degree,degree_sum,...,rumor_n_user_neighbors,rumor_n_comment_neighbors,rumor_clean_prob,before_avg_target_rumor_prob,before_target_rumor_loss,before_accuracy,before_rumor_f1,user_type_is_user,rumor_type_is_message,rumor_label_is_rumor
0,Full_PR_BCD_T20,1,user_1546896846,message_500263680026869760,1,20,0.05,4,1,5,...,0,0,0.525677,0.515993,0.662006,0.741279,0.621277,1,1,1
1,Full_PR_BCD_T20,1,user_124227622,message_500308574703452160,0,20,0.05,32,1,33,...,0,0,0.512724,0.515993,0.662006,0.741279,0.621277,1,1,1
2,Full_PR_BCD_T20,1,user_119972954,message_500263680026869760,0,20,0.05,20,1,21,...,0,0,0.525677,0.515993,0.662006,0.741279,0.621277,1,1,1
3,Full_PR_BCD_T20,1,user_23831448,message_500263680026869760,0,20,0.05,19,1,20,...,0,0,0.525677,0.515993,0.662006,0.741279,0.621277,1,1,1
4,Full_PR_BCD_T20,1,user_153919561,message_500266427840864256,0,20,0.05,17,1,18,...,0,0,0.520556,0.515993,0.662006,0.741279,0.621277,1,1,1


In [221]:
irl_data_dir = os.path.join(results_dir, "irl_features")
os.makedirs(irl_data_dir, exist_ok=True)

negative_action_feature_path = os.path.join(
    irl_data_dir,
    "negative_action_features.csv"
)

irl_action_feature_path = os.path.join(
    irl_data_dir,
    "irl_action_features_expert_and_negative.csv"
)

irl_training_path = os.path.join(
    irl_data_dir,
    "irl_training_features.csv"
)

feature_columns_path = os.path.join(
    irl_data_dir,
    "irl_feature_columns.pkl"
)

negative_action_feature_df.to_csv(negative_action_feature_path, index=False)
irl_action_feature_df.to_csv(irl_action_feature_path, index=False)
irl_training_df.to_csv(irl_training_path, index=False)

with open(feature_columns_path, "wb") as f:
    pickle.dump(irl_feature_columns, f)

print("Saved files:")
print(negative_action_feature_path)
print(irl_action_feature_path)
print(irl_training_path)
print(feature_columns_path)

Saved files:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/irl_features/negative_action_features.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/irl_features/irl_action_features_expert_and_negative.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/irl_features/irl_training_features.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/irl_features/irl_feature_columns.pkl


In [222]:



#EntIRL Baseline



In [223]:
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import pickle

entirl_df = irl_training_df.copy()

entirl_df["state_id"] = (
    entirl_df["attack_name"].astype(str)
    + "_step_"
    + entirl_df["step"].astype(str)
)

state_ids = entirl_df["state_id"].unique().tolist()
state_to_group = {state_id: i for i, state_id in enumerate(state_ids)}

entirl_df["group_id"] = entirl_df["state_id"].map(state_to_group)

X = entirl_df[irl_feature_columns].values.astype(np.float32)
y = entirl_df["is_expert"].values.astype(np.float32)
group_ids = entirl_df["group_id"].values.astype(np.int64)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)
group_tensor = torch.tensor(group_ids, dtype=torch.long)

num_features = X_tensor.shape[1]
num_groups = len(state_ids)

print("X_tensor shape:", X_tensor.shape)
print("y_tensor shape:", y_tensor.shape)
print("Number of state-action groups:", num_groups)
print("Number of features:", num_features)

display(
    entirl_df
    .groupby(["state_id", "is_expert"])
    .size()
    .reset_index(name="count")
    .head(10)
)

X_tensor shape: torch.Size([7575, 30])
y_tensor shape: torch.Size([7575])
Number of state-action groups: 75
Number of features: 30


,state_id,is_expert,count
0,Full_PR_BCD_T20_step_1,0,100
1,Full_PR_BCD_T20_step_1,1,1
2,Full_PR_BCD_T20_step_10,0,100
3,Full_PR_BCD_T20_step_10,1,1
4,Full_PR_BCD_T20_step_11,0,100
5,Full_PR_BCD_T20_step_11,1,1
6,Full_PR_BCD_T20_step_12,0,100
7,Full_PR_BCD_T20_step_12,1,1
8,Full_PR_BCD_T20_step_13,0,100
9,Full_PR_BCD_T20_step_13,1,1


In [224]:
class LinearEntIRLReward(torch.nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.theta = torch.nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        return x @ self.theta


def entirl_group_softmax_loss(rewards, y, group_ids):
    losses = []

    unique_groups = torch.unique(group_ids)

    for group_id in unique_groups:
        mask = group_ids == group_id
        group_rewards = rewards[mask]
        group_y = y[mask]

        expert_positions = torch.where(group_y == 1)[0]

        if len(expert_positions) == 0:
            continue

        log_probs = F.log_softmax(group_rewards, dim=0)
        expert_log_probs = log_probs[expert_positions]
        losses.append(-expert_log_probs.mean())

    if len(losses) == 0:
        return torch.tensor(0.0, requires_grad=True)

    return torch.stack(losses).mean()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = X_tensor.to(device)
y_tensor = y_tensor.to(device)
group_tensor = group_tensor.to(device)

entirl_model = LinearEntIRLReward(num_features=num_features).to(device)

optimizer = torch.optim.Adam(entirl_model.parameters(), lr=0.05, weight_decay=1e-4)

entirl_losses = []

num_epochs = 500

for epoch in range(1, num_epochs + 1):
    entirl_model.train()
    optimizer.zero_grad()

    rewards = entirl_model(X_tensor)
    loss = entirl_group_softmax_loss(rewards, y_tensor, group_tensor)

    loss.backward()
    optimizer.step()

    entirl_losses.append(float(loss.detach().cpu()))

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:04d} | EntIRL loss: {entirl_losses[-1]:.6f}")

print("Finished EntIRL training.")

Epoch 0001 | EntIRL loss: 4.615120
Epoch 0050 | EntIRL loss: 3.052392
Epoch 0100 | EntIRL loss: 2.970189
Epoch 0150 | EntIRL loss: 2.968096
Epoch 0200 | EntIRL loss: 2.967071
Epoch 0250 | EntIRL loss: 2.965907
Epoch 0300 | EntIRL loss: 2.964612
Epoch 0350 | EntIRL loss: 2.963196
Epoch 0400 | EntIRL loss: 2.961666
Epoch 0450 | EntIRL loss: 2.960031
Epoch 0500 | EntIRL loss: 2.958293
Finished EntIRL training.


In [225]:
entirl_model.eval()

with torch.no_grad():
    entirl_rewards = entirl_model(X_tensor).detach().cpu().numpy()

entirl_df["entirl_reward"] = entirl_rewards

ranking_rows = []

for state_id, group_df in entirl_df.groupby("state_id"):
    group_df_sorted = group_df.sort_values("entirl_reward", ascending=False).reset_index(drop=True)

    expert_positions = group_df_sorted.index[group_df_sorted["is_expert"] == 1].tolist()

    if len(expert_positions) == 0:
        continue

    expert_rank = expert_positions[0] + 1
    top_is_expert = int(group_df_sorted.iloc[0]["is_expert"] == 1)
    expert_reward = group_df_sorted[group_df_sorted["is_expert"] == 1]["entirl_reward"].iloc[0]
    best_negative_reward = group_df_sorted[group_df_sorted["is_expert"] == 0]["entirl_reward"].max()

    group_rewards = torch.tensor(group_df["entirl_reward"].values, dtype=torch.float32)
    group_probs = F.softmax(group_rewards, dim=0).numpy()
    expert_index_in_group = group_df.index[group_df["is_expert"] == 1].tolist()[0]
    expert_local_position = list(group_df.index).index(expert_index_in_group)
    expert_probability = group_probs[expert_local_position]

    ranking_rows.append({
        "state_id": state_id,
        "attack_name": group_df["attack_name"].iloc[0],
        "step": int(group_df["step"].iloc[0]),
        "num_actions": len(group_df),
        "expert_rank": expert_rank,
        "top_is_expert": top_is_expert,
        "expert_reward": expert_reward,
        "best_negative_reward": best_negative_reward,
        "reward_margin": expert_reward - best_negative_reward,
        "expert_probability": expert_probability
    })

entirl_ranking_df = pd.DataFrame(ranking_rows)

top1_accuracy = entirl_ranking_df["top_is_expert"].mean()
mean_expert_rank = entirl_ranking_df["expert_rank"].mean()
mean_reciprocal_rank = (1.0 / entirl_ranking_df["expert_rank"]).mean()
mean_expert_probability = entirl_ranking_df["expert_probability"].mean()

print("EntIRL ranking results")
print("Top-1 expert accuracy:", top1_accuracy)
print("Mean expert rank:", mean_expert_rank)
print("Mean reciprocal rank:", mean_reciprocal_rank)
print("Mean expert probability:", mean_expert_probability)

display(
    entirl_ranking_df
    .groupby("attack_name")
    .agg(
        top1_accuracy=("top_is_expert", "mean"),
        mean_expert_rank=("expert_rank", "mean"),
        mean_reward_margin=("reward_margin", "mean"),
        mean_expert_probability=("expert_probability", "mean")
    )
    .reset_index()
)

display(entirl_ranking_df.head())

EntIRL ranking results
Top-1 expert accuracy: 0.5466666666666666
Mean expert rank: 15.693333333333333
Mean reciprocal rank: 0.5962953566419943
Mean expert probability: 0.26428783


,attack_name,top1_accuracy,mean_expert_rank,mean_reward_margin,mean_expert_probability
0,Full_PR_BCD_T20,0.10,35.8,-3.985830,0.020493
1,Full_PR_BCD_T5,0.00,9.4,-2.321747,0.043927
2,Greedy_GC_RWCS_T20,0.60,17.3,1.332708,0.263647
3,Greedy_GC_RWCS_T5,0.60,1.4,1.695141,0.252190
4,PageRank_T20,0.95,2.8,6.207194,0.562508
5,PageRank_T5,1.00,1.0,2.463959,0.281608


,state_id,attack_name,step,num_actions,expert_rank,top_is_expert,expert_reward,best_negative_reward,reward_margin,expert_probability
0,Full_PR_BCD_T20_step_1,Full_PR_BCD_T20,1,101,34,0,0.170401,4.861038,-4.690638,0.002558
1,Full_PR_BCD_T20_step_10,Full_PR_BCD_T20,10,101,55,0,-0.400401,4.860364,-5.260765,0.001082
2,Full_PR_BCD_T20_step_11,Full_PR_BCD_T20,11,101,1,1,2.814560,1.318904,1.495656,0.238196
3,Full_PR_BCD_T20_step_12,Full_PR_BCD_T20,12,101,7,0,1.157080,3.279275,-2.122195,0.025477
4,Full_PR_BCD_T20_step_13,Full_PR_BCD_T20,13,101,46,0,-0.110930,4.699583,-4.810513,0.001668


In [226]:
theta_values = entirl_model.theta.detach().cpu().numpy()

entirl_feature_importance_df = pd.DataFrame({
    "feature": irl_feature_columns,
    "theta": theta_values,
    "abs_theta": np.abs(theta_values)
}).sort_values("abs_theta", ascending=False)

display(entirl_feature_importance_df)

print("Top 10 positive reward features:")
display(entirl_feature_importance_df.sort_values("theta", ascending=False).head(10))

print("Top 10 negative reward features:")
display(entirl_feature_importance_df.sort_values("theta", ascending=True).head(10))

,feature,theta,abs_theta
18,user_n_nonrumor_neighbors,1.702379,1.702379
16,user_n_message_neighbors,1.587773,1.587773
5,degree_abs_diff,-1.524454,1.524454
15,user_n_neighbors,-0.956787,0.956787
17,user_n_rumor_neighbors,-0.920873,0.920873
14,rumor_attack_degree_norm,0.792678,0.792678
22,rumor_clean_prob,0.773862,0.773862
12,rumor_attack_degree,0.671476,0.671476
3,rumor_degree,0.640237,0.640237
7,rumor_pagerank,0.380636,0.380636


Top 10 positive reward features:


,feature,theta,abs_theta
18,user_n_nonrumor_neighbors,1.702379,1.702379
16,user_n_message_neighbors,1.587773,1.587773
14,rumor_attack_degree_norm,0.792678,0.792678
22,rumor_clean_prob,0.773862,0.773862
12,rumor_attack_degree,0.671476,0.671476
3,rumor_degree,0.640237,0.640237
7,rumor_pagerank,0.380636,0.380636
11,user_attack_degree,0.210536,0.210536
4,degree_sum,0.184679,0.184679
2,user_degree,0.174674,0.174674


Top 10 negative reward features:


,feature,theta,abs_theta
5,degree_abs_diff,-1.524454,1.524454
15,user_n_neighbors,-0.956787,0.956787
17,user_n_rumor_neighbors,-0.920873,0.920873
9,pagerank_abs_diff,-0.188083,0.188083
19,rumor_n_neighbors,-0.125988,0.125988
21,rumor_n_comment_neighbors,-0.125988,0.125988
6,user_pagerank,-0.010952,0.010952
8,pagerank_sum,-0.010422,0.010422
1,step_norm,-0.000330,0.000330
26,before_rumor_f1,-0.000115,0.000115


In [227]:
entirl_dir = os.path.join(results_dir, "entirl_outputs")
os.makedirs(entirl_dir, exist_ok=True)

entirl_training_with_rewards_path = os.path.join(
    entirl_dir,
    "entirl_training_with_rewards.csv"
)

entirl_ranking_path = os.path.join(
    entirl_dir,
    "entirl_ranking_results.csv"
)

entirl_feature_importance_path = os.path.join(
    entirl_dir,
    "entirl_feature_importance.csv"
)

entirl_model_path = os.path.join(
    entirl_dir,
    "entirl_linear_reward_model.pt"
)

entirl_scaler_path = os.path.join(
    entirl_dir,
    "entirl_feature_scaler.pkl"
)

entirl_df.to_csv(entirl_training_with_rewards_path, index=False)
entirl_ranking_df.to_csv(entirl_ranking_path, index=False)
entirl_feature_importance_df.to_csv(entirl_feature_importance_path, index=False)

torch.save(entirl_model.state_dict(), entirl_model_path)

with open(entirl_scaler_path, "wb") as f:
    pickle.dump(scaler, f)

print("Saved EntIRL outputs:")
print(entirl_training_with_rewards_path)
print(entirl_ranking_path)
print(entirl_feature_importance_path)
print(entirl_model_path)
print(entirl_scaler_path)

Saved EntIRL outputs:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/entirl_outputs/entirl_training_with_rewards.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/entirl_outputs/entirl_ranking_results.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/entirl_outputs/entirl_feature_importance.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/entirl_outputs/entirl_linear_reward_model.pt
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/entirl_outputs/entirl_feature_scaler.pkl


In [228]:



#MoE-EntIRL




In [229]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import pickle

moe_entirl_df = entirl_df.copy()

if "group_id" not in moe_entirl_df.columns:
    moe_entirl_df["state_id"] = (
        moe_entirl_df["attack_name"].astype(str)
        + "_step_"
        + moe_entirl_df["step"].astype(str)
    )
    state_ids = moe_entirl_df["state_id"].unique().tolist()
    state_to_group = {state_id: i for i, state_id in enumerate(state_ids)}
    moe_entirl_df["group_id"] = moe_entirl_df["state_id"].map(state_to_group)

X_moe = moe_entirl_df[irl_feature_columns].values.astype(np.float32)
y_moe = moe_entirl_df["is_expert"].values.astype(np.float32)
group_ids_moe = moe_entirl_df["group_id"].values.astype(np.int64)

X_moe_scaled = scaler.transform(X_moe).astype(np.float32)

group_state_rows = []

for group_id, group_df in moe_entirl_df.groupby("group_id"):
    group_X = group_df[irl_feature_columns].values.astype(np.float32)
    group_X_scaled = scaler.transform(group_X).astype(np.float32)
    group_state_rows.append(group_X_scaled.mean(axis=0))

group_state_matrix = np.vstack(group_state_rows).astype(np.float32)

X_moe_tensor = torch.tensor(X_moe_scaled, dtype=torch.float32)
y_moe_tensor = torch.tensor(y_moe, dtype=torch.float32)
group_moe_tensor = torch.tensor(group_ids_moe, dtype=torch.long)
group_state_tensor = torch.tensor(group_state_matrix, dtype=torch.float32)

num_moe_features = X_moe_tensor.shape[1]
num_moe_groups = group_state_tensor.shape[0]

print("X_moe_tensor shape:", X_moe_tensor.shape)
print("y_moe_tensor shape:", y_moe_tensor.shape)
print("group_moe_tensor shape:", group_moe_tensor.shape)
print("group_state_tensor shape:", group_state_tensor.shape)
print("Number of MoE features:", num_moe_features)
print("Number of MoE groups:", num_moe_groups)

X_moe_tensor shape: torch.Size([7575, 30])
y_moe_tensor shape: torch.Size([7575])
group_moe_tensor shape: torch.Size([7575])
group_state_tensor shape: torch.Size([75, 30])
Number of MoE features: 30
Number of MoE groups: 75


In [230]:
class MoEEntIRLReward(torch.nn.Module):
    def __init__(self, num_features, num_experts=3, gate_hidden_dim=32):
        super().__init__()

        self.num_features = num_features
        self.num_experts = num_experts

        self.theta = torch.nn.Parameter(
            torch.zeros(num_experts, num_features)
        )

        self.gate = torch.nn.Sequential(
            torch.nn.Linear(num_features, gate_hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(gate_hidden_dim, num_experts)
        )

    def expert_rewards(self, x):
        return x @ self.theta.T

    def gate_probs(self, group_state_x):
        gate_logits = self.gate(group_state_x)
        return F.softmax(gate_logits, dim=1)

    def forward(self, x, group_state_x):
        rewards = self.expert_rewards(x)
        gate_probs = self.gate_probs(group_state_x)
        return rewards, gate_probs


def moe_entirl_loss(expert_rewards, gate_probs, y, group_ids):
    losses = []

    unique_groups = torch.unique(group_ids)

    for group_id in unique_groups:
        mask = group_ids == group_id

        group_rewards = expert_rewards[mask]
        group_y = y[mask]

        expert_positions = torch.where(group_y == 1)[0]

        if len(expert_positions) == 0:
            continue

        expert_position = expert_positions[0]

        group_gate_probs = gate_probs[group_id]
        log_gate_probs = torch.log(group_gate_probs + 1e-12)

        expert_log_probs_by_expert = []

        for k in range(group_rewards.shape[1]):
            expert_log_probs_by_expert.append(
                F.log_softmax(group_rewards[:, k], dim=0)[expert_position]
            )

        expert_log_probs_by_expert = torch.stack(expert_log_probs_by_expert)

        log_mixture_prob = torch.logsumexp(
            log_gate_probs + expert_log_probs_by_expert,
            dim=0
        )

        losses.append(-log_mixture_prob)

    if len(losses) == 0:
        return torch.tensor(0.0, requires_grad=True)

    return torch.stack(losses).mean()


print("MoE-EntIRL model is ready.")

MoE-EntIRL model is ready.


In [231]:
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

def estimate_num_experts_with_dbscan(group_state_matrix, min_k=1, max_k=25):
    eps_candidates = np.linspace(0.5, 8.0, 31)

    best_k = None
    best_labels = None
    best_score = -np.inf
    best_eps = None

    for eps in eps_candidates:
        dbscan = DBSCAN(eps=eps, min_samples=2)
        labels = dbscan.fit_predict(group_state_matrix)

        non_noise_labels = sorted(set(labels) - {-1})
        k = len(non_noise_labels)

        if k < min_k or k > max_k:
            continue

        valid_mask = labels != -1

        if valid_mask.sum() <= k or k <= 1:
            score = valid_mask.sum() / len(labels)
        else:
            try:
                score = silhouette_score(
                    group_state_matrix[valid_mask],
                    labels[valid_mask]
                )
            except Exception:
                score = valid_mask.sum() / len(labels)

        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels
            best_eps = eps

    if best_k is None:
        best_k = 3
        best_eps = None
        best_labels = np.zeros(group_state_matrix.shape[0], dtype=int)

    best_k = int(max(min(best_k, max_k), min_k))

    return best_k, best_labels, best_eps, best_score


num_experts, dbscan_group_labels, dbscan_eps, dbscan_score = estimate_num_experts_with_dbscan(
    group_state_matrix=group_state_matrix,
    min_k=1,
    max_k=25
)

gmm = GaussianMixture(
    n_components=num_experts,
    covariance_type="full",
    random_state=42,
    reg_covar=1e-6
)

gmm_group_labels = gmm.fit_predict(group_state_matrix)

gmm_group_labels_tensor = torch.tensor(
    gmm_group_labels,
    dtype=torch.long
)

print("Estimated number of experts K:", num_experts)
print("DBSCAN eps:", dbscan_eps)
print("DBSCAN score:", dbscan_score)
print("GMM label counts:")
display(
    pd.Series(gmm_group_labels)
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "expert_id", 0: "num_groups"})
)

Estimated number of experts K: 1
DBSCAN eps: 2.75
DBSCAN score: 1.0
GMM label counts:


,expert_id,count
0,0,75


In [232]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_moe_tensor = X_moe_tensor.to(device)
y_moe_tensor = y_moe_tensor.to(device)
group_moe_tensor = group_moe_tensor.to(device)
group_state_tensor = group_state_tensor.to(device)
gmm_group_labels_tensor = gmm_group_labels_tensor.to(device)

moe_entirl_model = MoEEntIRLReward(
    num_features=num_moe_features,
    num_experts=num_experts,
    gate_hidden_dim=32
).to(device)

gate_optimizer = torch.optim.Adam(
    moe_entirl_model.gate.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

gate_pretrain_epochs = 200
gate_pretraining_losses = []

for epoch in range(1, gate_pretrain_epochs + 1):
    moe_entirl_model.train()
    gate_optimizer.zero_grad()

    gate_logits = moe_entirl_model.gate(group_state_tensor)
    gate_loss = F.cross_entropy(gate_logits, gmm_group_labels_tensor)

    gate_loss.backward()
    gate_optimizer.step()

    gate_pretraining_losses.append(float(gate_loss.detach().cpu()))

    if epoch % 50 == 0 or epoch == 1:
        print(f"Gate pretrain epoch {epoch:04d} | loss: {gate_pretraining_losses[-1]:.6f}")

optimizer = torch.optim.Adam(
    moe_entirl_model.parameters(),
    lr=0.03,
    weight_decay=1e-4
)

moe_entirl_losses = []

num_epochs = PAPER_IRL_EPISODES
gate_label_mix = 0.5

one_hot_gmm_labels = F.one_hot(
    gmm_group_labels_tensor,
    num_classes=num_experts
).float()

for epoch in range(1, num_epochs + 1):
    moe_entirl_model.train()
    optimizer.zero_grad()

    expert_rewards, gate_probs = moe_entirl_model(
        X_moe_tensor,
        group_state_tensor
    )

    mixed_gate_probs = (
        gate_label_mix * one_hot_gmm_labels
        + (1.0 - gate_label_mix) * gate_probs
    )

    loss = moe_entirl_loss(
        expert_rewards=expert_rewards,
        gate_probs=mixed_gate_probs,
        y=y_moe_tensor,
        group_ids=group_moe_tensor
    )

    loss.backward()
    optimizer.step()

    moe_entirl_losses.append(float(loss.detach().cpu()))

    if epoch % 100 == 0 or epoch == 1:
        print(f"Epoch {epoch:04d} | MoE-EntIRL loss: {moe_entirl_losses[-1]:.6f}")

print("Finished MoE-EntIRL training.")
print("Number of experts used:", num_experts)

Gate pretrain epoch 0001 | loss: 0.000000
Gate pretrain epoch 0050 | loss: 0.000000
Gate pretrain epoch 0100 | loss: 0.000000
Gate pretrain epoch 0150 | loss: 0.000000
Gate pretrain epoch 0200 | loss: 0.000000
Epoch 0001 | MoE-EntIRL loss: 4.615120
Epoch 0100 | MoE-EntIRL loss: 2.968617
Epoch 0200 | MoE-EntIRL loss: 2.964520
Epoch 0300 | MoE-EntIRL loss: 2.960219
Epoch 0400 | MoE-EntIRL loss: 2.955164
Epoch 0500 | MoE-EntIRL loss: 2.949461
Epoch 0600 | MoE-EntIRL loss: 2.943212
Epoch 0700 | MoE-EntIRL loss: 2.936506
Epoch 0800 | MoE-EntIRL loss: 2.929423
Epoch 0900 | MoE-EntIRL loss: 2.922040
Epoch 1000 | MoE-EntIRL loss: 2.914429
Finished MoE-EntIRL training.
Number of experts used: 1


In [233]:
moe_entirl_model.eval()

with torch.no_grad():
    expert_rewards_tensor, gate_probs_tensor = moe_entirl_model(
        X_moe_tensor,
        group_state_tensor
    )

    expert_rewards_np = expert_rewards_tensor.detach().cpu().numpy()
    gate_probs_np = gate_probs_tensor.detach().cpu().numpy()

moe_entirl_df = moe_entirl_df.copy()

for k in range(num_experts):
    moe_entirl_df[f"moe_expert_{k}_reward"] = expert_rewards_np[:, k]

mixture_rewards = []

for row_idx, row in moe_entirl_df.iterrows():
    group_id = int(row["group_id"])

    action_expert_rewards = expert_rewards_np[row_idx]
    action_gate_probs = gate_probs_np[group_id]

    mixture_reward = np.sum(action_gate_probs * action_expert_rewards)
    mixture_rewards.append(mixture_reward)

moe_entirl_df["moe_entirl_reward"] = mixture_rewards

ranking_rows = []

for state_id, group_df in moe_entirl_df.groupby("state_id"):
    group_df_sorted = group_df.sort_values("moe_entirl_reward", ascending=False).reset_index(drop=True)

    expert_positions = group_df_sorted.index[group_df_sorted["is_expert"] == 1].tolist()

    if len(expert_positions) == 0:
        continue

    expert_rank = expert_positions[0] + 1
    top_is_expert = int(group_df_sorted.iloc[0]["is_expert"] == 1)

    expert_reward = group_df_sorted[group_df_sorted["is_expert"] == 1]["moe_entirl_reward"].iloc[0]
    best_negative_reward = group_df_sorted[group_df_sorted["is_expert"] == 0]["moe_entirl_reward"].max()

    group_rewards = torch.tensor(group_df["moe_entirl_reward"].values, dtype=torch.float32)
    group_probs = F.softmax(group_rewards, dim=0).numpy()

    expert_index_in_group = group_df.index[group_df["is_expert"] == 1].tolist()[0]
    expert_local_position = list(group_df.index).index(expert_index_in_group)
    expert_probability = group_probs[expert_local_position]

    group_id = int(group_df["group_id"].iloc[0])
    assigned_expert = int(np.argmax(gate_probs_np[group_id]))

    ranking_rows.append({
        "state_id": state_id,
        "attack_name": group_df["attack_name"].iloc[0],
        "step": int(group_df["step"].iloc[0]),
        "group_id": group_id,
        "assigned_expert": assigned_expert,
        "num_actions": len(group_df),
        "expert_rank": expert_rank,
        "top_is_expert": top_is_expert,
        "expert_reward": expert_reward,
        "best_negative_reward": best_negative_reward,
        "reward_margin": expert_reward - best_negative_reward,
        "expert_probability": expert_probability
    })

moe_entirl_ranking_df = pd.DataFrame(ranking_rows)

moe_top1_accuracy = moe_entirl_ranking_df["top_is_expert"].mean()
moe_mean_expert_rank = moe_entirl_ranking_df["expert_rank"].mean()
moe_mean_reciprocal_rank = (1.0 / moe_entirl_ranking_df["expert_rank"]).mean()
moe_mean_expert_probability = moe_entirl_ranking_df["expert_probability"].mean()

print("MoE-EntIRL ranking results")
print("Top-1 expert accuracy:", moe_top1_accuracy)
print("Mean expert rank:", moe_mean_expert_rank)
print("Mean reciprocal rank:", moe_mean_reciprocal_rank)
print("Mean expert probability:", moe_mean_expert_probability)

display(
    moe_entirl_ranking_df
    .groupby("attack_name")
    .agg(
        top1_accuracy=("top_is_expert", "mean"),
        mean_expert_rank=("expert_rank", "mean"),
        mean_reward_margin=("reward_margin", "mean"),
        mean_expert_probability=("expert_probability", "mean")
    )
    .reset_index()
)

display(
    moe_entirl_ranking_df
    .groupby(["attack_name", "assigned_expert"])
    .size()
    .reset_index(name="num_groups")
)

display(moe_entirl_ranking_df.head())

MoE-EntIRL ranking results
Top-1 expert accuracy: 0.56
Mean expert rank: 15.16
Mean reciprocal rank: 0.6092651482796713
Mean expert probability: 0.2697008


,attack_name,top1_accuracy,mean_expert_rank,mean_reward_margin,mean_expert_probability
0,Full_PR_BCD_T20,0.10,35.70,-3.924553,0.020240
1,Full_PR_BCD_T5,0.00,9.60,-2.317026,0.044026
2,Greedy_GC_RWCS_T20,0.65,17.10,1.319865,0.263761
3,Greedy_GC_RWCS_T5,0.60,1.40,1.696172,0.254283
4,PageRank_T20,0.95,1.05,6.380105,0.579648
5,PageRank_T5,1.00,1.00,2.480674,0.292609


,attack_name,assigned_expert,num_groups
0,Full_PR_BCD_T20,0,20
1,Full_PR_BCD_T5,0,5
2,Greedy_GC_RWCS_T20,0,20
3,Greedy_GC_RWCS_T5,0,5
4,PageRank_T20,0,20
5,PageRank_T5,0,5


,state_id,attack_name,step,group_id,assigned_expert,num_actions,expert_rank,top_is_expert,expert_reward,best_negative_reward,reward_margin,expert_probability
0,Full_PR_BCD_T20_step_1,Full_PR_BCD_T20,1,0,0,101,34,0,0.160519,4.773907,-4.613388,0.002674
1,Full_PR_BCD_T20_step_10,Full_PR_BCD_T20,10,9,0,101,54,0,-0.365929,4.776243,-5.142172,0.001196
2,Full_PR_BCD_T20_step_11,Full_PR_BCD_T20,11,10,0,101,1,1,2.801342,1.325160,1.476182,0.232735
3,Full_PR_BCD_T20_step_12,Full_PR_BCD_T20,12,11,0,101,7,0,1.168864,3.245484,-2.076620,0.025798
4,Full_PR_BCD_T20_step_13,Full_PR_BCD_T20,13,12,0,101,46,0,-0.102856,4.678099,-4.780955,0.001721


In [234]:
comparison_rows = [
    {
        "model": "EntIRL",
        "top1_accuracy": top1_accuracy,
        "mean_expert_rank": mean_expert_rank,
        "mean_reciprocal_rank": mean_reciprocal_rank,
        "mean_expert_probability": mean_expert_probability
    },
    {
        "model": "MoE-EntIRL",
        "top1_accuracy": moe_top1_accuracy,
        "mean_expert_rank": moe_mean_expert_rank,
        "mean_reciprocal_rank": moe_mean_reciprocal_rank,
        "mean_expert_probability": moe_mean_expert_probability
    }
]

entirl_vs_moe_comparison_df = pd.DataFrame(comparison_rows)

display(entirl_vs_moe_comparison_df)

,model,top1_accuracy,mean_expert_rank,mean_reciprocal_rank,mean_expert_probability
0,EntIRL,0.546667,15.693333,0.596295,0.264288
1,MoE-EntIRL,0.560000,15.160000,0.609265,0.269701


In [235]:
theta_matrix = moe_entirl_model.theta.detach().cpu().numpy()

moe_feature_importance_rows = []

for k in range(num_experts):
    for feature_idx, feature_name in enumerate(irl_feature_columns):
        theta_value = theta_matrix[k, feature_idx]

        moe_feature_importance_rows.append({
            "expert_id": k,
            "feature": feature_name,
            "theta": theta_value,
            "abs_theta": abs(theta_value)
        })

moe_feature_importance_df = pd.DataFrame(moe_feature_importance_rows)

print("Top features by expert:")

for k in range(num_experts):
    print("\nExpert", k, "- top absolute features")
    display(
        moe_feature_importance_df[moe_feature_importance_df["expert_id"] == k]
        .sort_values("abs_theta", ascending=False)
        .head(10)
    )

    print("Expert", k, "- top positive features")
    display(
        moe_feature_importance_df[moe_feature_importance_df["expert_id"] == k]
        .sort_values("theta", ascending=False)
        .head(10)
    )

    print("Expert", k, "- top negative features")
    display(
        moe_feature_importance_df[moe_feature_importance_df["expert_id"] == k]
        .sort_values("theta", ascending=True)
        .head(10)
    )

Top features by expert:

Expert 0 - top absolute features


,expert_id,feature,theta,abs_theta
5,0,degree_abs_diff,-6.097491,6.097491
15,0,user_n_neighbors,-2.816224,2.816224
4,0,degree_sum,2.303143,2.303143
2,0,user_degree,2.253340,2.253340
18,0,user_n_nonrumor_neighbors,1.539268,1.539268
16,0,user_n_message_neighbors,1.487778,1.487778
8,0,pagerank_sum,1.060847,1.060847
6,0,user_pagerank,1.057898,1.057898
17,0,user_n_rumor_neighbors,-0.938916,0.938916
14,0,rumor_attack_degree_norm,0.798770,0.798770


Expert 0 - top positive features


,expert_id,feature,theta,abs_theta
4,0,degree_sum,2.303143,2.303143
2,0,user_degree,2.253340,2.253340
18,0,user_n_nonrumor_neighbors,1.539268,1.539268
16,0,user_n_message_neighbors,1.487778,1.487778
8,0,pagerank_sum,1.060847,1.060847
6,0,user_pagerank,1.057898,1.057898
14,0,rumor_attack_degree_norm,0.798770,0.798770
22,0,rumor_clean_prob,0.769678,0.769678
12,0,rumor_attack_degree,0.640492,0.640492
3,0,rumor_degree,0.610927,0.610927


Expert 0 - top negative features


,expert_id,feature,theta,abs_theta
5,0,degree_abs_diff,-6.097491,6.097491
15,0,user_n_neighbors,-2.816224,2.816224
17,0,user_n_rumor_neighbors,-0.938916,0.938916
19,0,rumor_n_neighbors,-0.136054,0.136054
21,0,rumor_n_comment_neighbors,-0.136054,0.136054
26,0,before_rumor_f1,-0.000590,0.000590
23,0,before_avg_target_rumor_prob,-0.000240,0.000240
24,0,before_target_rumor_loss,-0.000108,0.000108
27,0,user_type_is_user,0.000000,0.000000
20,0,rumor_n_user_neighbors,0.000000,0.000000


In [236]:
moe_entirl_dir = os.path.join(results_dir, "moe_entirl_outputs")
os.makedirs(moe_entirl_dir, exist_ok=True)

moe_entirl_training_with_rewards_path = os.path.join(
    moe_entirl_dir,
    "moe_entirl_training_with_rewards.csv"
)

moe_entirl_ranking_path = os.path.join(
    moe_entirl_dir,
    "moe_entirl_ranking_results.csv"
)

moe_feature_importance_path = os.path.join(
    moe_entirl_dir,
    "moe_entirl_feature_importance.csv"
)

entirl_vs_moe_comparison_path = os.path.join(
    moe_entirl_dir,
    "entirl_vs_moe_comparison.csv"
)

moe_entirl_model_path = os.path.join(
    moe_entirl_dir,
    "moe_entirl_reward_model.pt"
)

moe_entirl_df.to_csv(moe_entirl_training_with_rewards_path, index=False)
moe_entirl_ranking_df.to_csv(moe_entirl_ranking_path, index=False)
moe_feature_importance_df.to_csv(moe_feature_importance_path, index=False)
entirl_vs_moe_comparison_df.to_csv(entirl_vs_moe_comparison_path, index=False)

torch.save(moe_entirl_model.state_dict(), moe_entirl_model_path)

print("Saved MoE-EntIRL outputs:")
print(moe_entirl_training_with_rewards_path)
print(moe_entirl_ranking_path)
print(moe_feature_importance_path)
print(entirl_vs_moe_comparison_path)
print(moe_entirl_model_path)

Saved MoE-EntIRL outputs:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_entirl_outputs/moe_entirl_training_with_rewards.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_entirl_outputs/moe_entirl_ranking_results.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_entirl_outputs/moe_entirl_feature_importance.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_entirl_outputs/entirl_vs_moe_comparison.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_entirl_outputs/moe_entirl_reward_model.pt


In [237]:
def moe_bientirl_loss(
    expert_rewards,
    gate_probs,
    y,
    group_ids,
    apply_guidance=False,
    max_reward_value=2.0,
    guidance_weight=0.15,
    inverse_weight=0.02,
    margin_value=0.3
):
    likelihood_losses = []
    guidance_losses = []
    inverse_losses = []

    unique_groups = torch.unique(group_ids)

    for group_id in unique_groups:
        mask = group_ids == group_id

        group_rewards = expert_rewards[mask]
        group_y = y[mask]

        expert_positions = torch.where(group_y == 1)[0]

        if len(expert_positions) == 0:
            continue

        expert_position = expert_positions[0]

        group_gate_probs = gate_probs[group_id]
        log_gate_probs = torch.log(group_gate_probs + 1e-12)

        expert_log_probs_by_expert = []

        for k in range(group_rewards.shape[1]):
            expert_log_probs_by_expert.append(
                F.log_softmax(group_rewards[:, k], dim=0)[expert_position]
            )

        expert_log_probs_by_expert = torch.stack(expert_log_probs_by_expert)

        log_mixture_prob = torch.logsumexp(
            log_gate_probs + expert_log_probs_by_expert,
            dim=0
        )

        likelihood_losses.append(-log_mixture_prob)

        if apply_guidance:
            mixture_action_rewards = torch.sum(
                group_rewards * group_gate_probs.unsqueeze(0),
                dim=1
            )

            expert_mixture_reward = mixture_action_rewards[expert_position]

            negative_mask = group_y == 0

            if negative_mask.sum() > 0:
                negative_rewards = mixture_action_rewards[negative_mask]
                hardest_negative_reward = torch.max(negative_rewards)

                margin_loss = F.relu(
                    margin_value - expert_mixture_reward + hardest_negative_reward
                )

                guidance_losses.append(margin_loss)

            expert_rewards_by_expert = group_rewards[expert_position]
            target_reward = torch.clamp(
                expert_rewards_by_expert.detach() + 0.2,
                max=max_reward_value
            )

            inverse_loss = torch.sum(
                group_gate_probs * ((expert_rewards_by_expert - target_reward) ** 2)
            )

            inverse_losses.append(inverse_loss)

    if len(likelihood_losses) == 0:
        zero_loss = torch.tensor(0.0, requires_grad=True, device=expert_rewards.device)
        return zero_loss, zero_loss, zero_loss, zero_loss

    likelihood_loss = torch.stack(likelihood_losses).mean()

    if len(guidance_losses) > 0:
        guidance_loss = torch.stack(guidance_losses).mean()
    else:
        guidance_loss = torch.tensor(0.0, device=expert_rewards.device)

    if len(inverse_losses) > 0:
        inverse_loss = torch.stack(inverse_losses).mean()
    else:
        inverse_loss = torch.tensor(0.0, device=expert_rewards.device)

    total_loss = likelihood_loss

    if apply_guidance:
        total_loss = total_loss + guidance_weight * guidance_loss + inverse_weight * inverse_loss

    return total_loss, likelihood_loss, guidance_loss, inverse_loss


print("MoE-BiEntIRL loss is ready.")

MoE-BiEntIRL loss is ready.


In [238]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_moe_tensor = X_moe_tensor.to(device)
y_moe_tensor = y_moe_tensor.to(device)
group_moe_tensor = group_moe_tensor.to(device)
group_state_tensor = group_state_tensor.to(device)

moe_bientirl_model = MoEEntIRLReward(
    num_features=num_moe_features,
    num_experts=num_experts,
    gate_hidden_dim=32
).to(device)

moe_bientirl_model.load_state_dict(moe_entirl_model.state_dict())

optimizer = torch.optim.Adam(
    moe_bientirl_model.parameters(),
    lr=0.003,
    weight_decay=1e-4
)

moe_bientirl_losses = []
moe_bientirl_likelihood_losses = []
moe_bientirl_guidance_losses = []
moe_bientirl_inverse_losses = []
moe_bientirl_guidance_flags = []

best_state_dict = None
best_top1_accuracy = -1
best_epoch = 0

num_epochs = PAPER_IRL_EPISODES

for epoch in range(1, num_epochs + 1):
    moe_bientirl_model.train()
    optimizer.zero_grad()

    expert_rewards, gate_probs = moe_bientirl_model(
        X_moe_tensor,
        group_state_tensor
    )

    apply_guidance = (
        epoch <= PAPER_GUIDANCE_EPISODE_LIMIT
        and epoch % PAPER_GUIDANCE_FREQUENCY == 0
    )

    loss, likelihood_loss, guidance_loss, inverse_loss = moe_bientirl_loss(
        expert_rewards=expert_rewards,
        gate_probs=gate_probs,
        y=y_moe_tensor,
        group_ids=group_moe_tensor,
        apply_guidance=apply_guidance,
        max_reward_value=2.0,
        guidance_weight=0.15,
        inverse_weight=0.02,
        margin_value=0.3
    )

    loss.backward()
    optimizer.step()

    moe_bientirl_losses.append(float(loss.detach().cpu()))
    moe_bientirl_likelihood_losses.append(float(likelihood_loss.detach().cpu()))
    moe_bientirl_guidance_losses.append(float(guidance_loss.detach().cpu()))
    moe_bientirl_inverse_losses.append(float(inverse_loss.detach().cpu()))
    moe_bientirl_guidance_flags.append(int(apply_guidance))

    if epoch % 50 == 0 or epoch == 1:
        moe_bientirl_model.eval()

        with torch.no_grad():
            eval_expert_rewards, eval_gate_probs = moe_bientirl_model(
                X_moe_tensor,
                group_state_tensor
            )

            eval_expert_rewards_np = eval_expert_rewards.detach().cpu().numpy()
            eval_gate_probs_np = eval_gate_probs.detach().cpu().numpy()

        eval_df = moe_entirl_df.copy()
        eval_rewards = []

        for row_idx, row in eval_df.iterrows():
            group_id = int(row["group_id"])
            action_expert_rewards = eval_expert_rewards_np[row_idx]
            action_gate_probs = eval_gate_probs_np[group_id]
            eval_rewards.append(np.sum(action_gate_probs * action_expert_rewards))

        eval_df["eval_reward"] = eval_rewards

        eval_top1_values = []

        for state_id, group_df in eval_df.groupby("state_id"):
            group_df_sorted = group_df.sort_values("eval_reward", ascending=False).reset_index(drop=True)
            eval_top1_values.append(int(group_df_sorted.iloc[0]["is_expert"] == 1))

        current_top1_accuracy = np.mean(eval_top1_values)

        if current_top1_accuracy > best_top1_accuracy:
            best_top1_accuracy = current_top1_accuracy
            best_epoch = epoch
            best_state_dict = {
                key: value.detach().cpu().clone()
                for key, value in moe_bientirl_model.state_dict().items()
            }

        print(
            f"Epoch {epoch:04d} | "
            f"Total: {moe_bientirl_losses[-1]:.6f} | "
            f"Likelihood: {moe_bientirl_likelihood_losses[-1]:.6f} | "
            f"Guidance: {moe_bientirl_guidance_losses[-1]:.6f} | "
            f"Inverse: {moe_bientirl_inverse_losses[-1]:.6f} | "
            f"Guided: {int(apply_guidance)} | "
            f"Top1: {current_top1_accuracy:.4f}"
        )

if best_state_dict is not None:
    moe_bientirl_model.load_state_dict(best_state_dict)

print("Finished MoE-BiEntIRL-style training.")
print("Best epoch:", best_epoch)
print("Best Top-1 expert accuracy during training:", best_top1_accuracy)

Epoch 0001 | Total: 2.914352 | Likelihood: 2.914352 | Guidance: 0.000000 | Inverse: 0.000000 | Guided: 0 | Top1: 0.5600
Epoch 0050 | Total: 3.487571 | Likelihood: 2.929434 | Guidance: 1.425640 | Inverse: 17.214531 | Guided: 1 | Top1: 0.5333
Epoch 0100 | Total: 3.365692 | Likelihood: 2.932709 | Guidance: 1.374154 | Inverse: 11.342978 | Guided: 1 | Top1: 0.5467
Epoch 0150 | Total: 3.296361 | Likelihood: 2.934846 | Guidance: 1.342927 | Inverse: 8.003825 | Guided: 1 | Top1: 0.5600
Epoch 0200 | Total: 3.259459 | Likelihood: 2.938942 | Guidance: 1.326594 | Inverse: 6.076415 | Guided: 1 | Top1: 0.5733
Epoch 0250 | Total: 3.242022 | Likelihood: 2.941988 | Guidance: 1.320198 | Inverse: 5.100201 | Guided: 1 | Top1: 0.5733
Epoch 0300 | Total: 3.233471 | Likelihood: 2.943288 | Guidance: 1.317671 | Inverse: 4.626584 | Guided: 1 | Top1: 0.5733
Epoch 0350 | Total: 3.228483 | Likelihood: 2.943238 | Guidance: 1.316706 | Inverse: 4.386940 | Guided: 1 | Top1: 0.5733
Epoch 0400 | Total: 3.224983 | Likelih

In [239]:
moe_bientirl_model.eval()

with torch.no_grad():
    bientirl_expert_rewards_tensor, bientirl_gate_probs_tensor = moe_bientirl_model(
        X_moe_tensor,
        group_state_tensor
    )

    bientirl_expert_rewards_np = bientirl_expert_rewards_tensor.detach().cpu().numpy()
    bientirl_gate_probs_np = bientirl_gate_probs_tensor.detach().cpu().numpy()

moe_bientirl_df = moe_entirl_df.copy()

for k in range(num_experts):
    moe_bientirl_df[f"bientirl_expert_{k}_reward"] = bientirl_expert_rewards_np[:, k]

bientirl_mixture_rewards = []

for row_idx, row in moe_bientirl_df.iterrows():
    group_id = int(row["group_id"])

    action_expert_rewards = bientirl_expert_rewards_np[row_idx]
    action_gate_probs = bientirl_gate_probs_np[group_id]

    mixture_reward = np.sum(action_gate_probs * action_expert_rewards)
    bientirl_mixture_rewards.append(mixture_reward)

moe_bientirl_df["moe_bientirl_reward"] = bientirl_mixture_rewards

bientirl_ranking_rows = []

for state_id, group_df in moe_bientirl_df.groupby("state_id"):
    group_df_sorted = group_df.sort_values("moe_bientirl_reward", ascending=False).reset_index(drop=True)

    expert_positions = group_df_sorted.index[group_df_sorted["is_expert"] == 1].tolist()

    if len(expert_positions) == 0:
        continue

    expert_rank = expert_positions[0] + 1
    top_is_expert = int(group_df_sorted.iloc[0]["is_expert"] == 1)

    expert_reward = group_df_sorted[group_df_sorted["is_expert"] == 1]["moe_bientirl_reward"].iloc[0]
    best_negative_reward = group_df_sorted[group_df_sorted["is_expert"] == 0]["moe_bientirl_reward"].max()

    group_rewards = torch.tensor(group_df["moe_bientirl_reward"].values, dtype=torch.float32)
    group_probs = F.softmax(group_rewards, dim=0).numpy()

    expert_index_in_group = group_df.index[group_df["is_expert"] == 1].tolist()[0]
    expert_local_position = list(group_df.index).index(expert_index_in_group)
    expert_probability = group_probs[expert_local_position]

    group_id = int(group_df["group_id"].iloc[0])
    assigned_expert = int(np.argmax(bientirl_gate_probs_np[group_id]))

    bientirl_ranking_rows.append({
        "state_id": state_id,
        "attack_name": group_df["attack_name"].iloc[0],
        "step": int(group_df["step"].iloc[0]),
        "group_id": group_id,
        "assigned_expert": assigned_expert,
        "num_actions": len(group_df),
        "expert_rank": expert_rank,
        "top_is_expert": top_is_expert,
        "expert_reward": expert_reward,
        "best_negative_reward": best_negative_reward,
        "reward_margin": expert_reward - best_negative_reward,
        "expert_probability": expert_probability
    })

moe_bientirl_ranking_df = pd.DataFrame(bientirl_ranking_rows)

bientirl_top1_accuracy = moe_bientirl_ranking_df["top_is_expert"].mean()
bientirl_mean_expert_rank = moe_bientirl_ranking_df["expert_rank"].mean()
bientirl_mean_reciprocal_rank = (1.0 / moe_bientirl_ranking_df["expert_rank"]).mean()
bientirl_mean_expert_probability = moe_bientirl_ranking_df["expert_probability"].mean()

print("MoE-BiEntIRL-style ranking results")
print("Top-1 expert accuracy:", bientirl_top1_accuracy)
print("Mean expert rank:", bientirl_mean_expert_rank)
print("Mean reciprocal rank:", bientirl_mean_reciprocal_rank)
print("Mean expert probability:", bientirl_mean_expert_probability)

display(
    moe_bientirl_ranking_df
    .groupby("attack_name")
    .agg(
        top1_accuracy=("top_is_expert", "mean"),
        mean_expert_rank=("expert_rank", "mean"),
        mean_reward_margin=("reward_margin", "mean"),
        mean_expert_probability=("expert_probability", "mean")
    )
    .reset_index()
)

display(
    moe_bientirl_ranking_df
    .groupby(["attack_name", "assigned_expert"])
    .size()
    .reset_index(name="num_groups")
)

display(moe_bientirl_ranking_df.head())

MoE-BiEntIRL-style ranking results
Top-1 expert accuracy: 0.5866666666666667
Mean expert rank: 15.36
Mean reciprocal rank: 0.6224689511677941
Mean expert probability: 0.25667784


,attack_name,top1_accuracy,mean_expert_rank,mean_reward_margin,mean_expert_probability
0,Full_PR_BCD_T20,0.10,36.30,-3.715068,0.018592
1,Full_PR_BCD_T5,0.00,9.60,-2.350943,0.047074
2,Greedy_GC_RWCS_T20,0.65,17.35,1.483890,0.251394
3,Greedy_GC_RWCS_T5,0.80,1.20,1.798340,0.242068
4,PageRank_T20,1.00,1.00,3.822303,0.555336
5,PageRank_T5,1.00,1.00,2.407809,0.259738


,attack_name,assigned_expert,num_groups
0,Full_PR_BCD_T20,0,20
1,Full_PR_BCD_T5,0,5
2,Greedy_GC_RWCS_T20,0,20
3,Greedy_GC_RWCS_T5,0,5
4,PageRank_T20,0,20
5,PageRank_T5,0,5


,state_id,attack_name,step,group_id,assigned_expert,num_actions,expert_rank,top_is_expert,expert_reward,best_negative_reward,reward_margin,expert_probability
0,Full_PR_BCD_T20_step_1,Full_PR_BCD_T20,1,0,0,101,34,0,0.236651,4.628148,-4.391498,0.002948
1,Full_PR_BCD_T20_step_10,Full_PR_BCD_T20,10,9,0,101,55,0,2.014156,6.915481,-4.901325,0.001391
2,Full_PR_BCD_T20_step_11,Full_PR_BCD_T20,11,10,0,101,1,1,5.061606,3.700461,1.361145,0.206375
3,Full_PR_BCD_T20_step_12,Full_PR_BCD_T20,12,11,0,101,7,0,3.470876,5.447580,-1.976705,0.025197
4,Full_PR_BCD_T20_step_13,Full_PR_BCD_T20,13,12,0,101,48,0,2.306114,6.895546,-4.589433,0.002010


In [240]:
model_comparison_rows = [
    {
        "model": "EntIRL",
        "top1_accuracy": top1_accuracy,
        "mean_expert_rank": mean_expert_rank,
        "mean_reciprocal_rank": mean_reciprocal_rank,
        "mean_expert_probability": mean_expert_probability
    },
    {
        "model": "MoE-EntIRL",
        "top1_accuracy": moe_top1_accuracy,
        "mean_expert_rank": moe_mean_expert_rank,
        "mean_reciprocal_rank": moe_mean_reciprocal_rank,
        "mean_expert_probability": moe_mean_expert_probability
    },
    {
        "model": "MoE-BiEntIRL-style",
        "top1_accuracy": bientirl_top1_accuracy,
        "mean_expert_rank": bientirl_mean_expert_rank,
        "mean_reciprocal_rank": bientirl_mean_reciprocal_rank,
        "mean_expert_probability": bientirl_mean_expert_probability
    }
]

final_irl_model_comparison_df = pd.DataFrame(model_comparison_rows)

display(final_irl_model_comparison_df)

,model,top1_accuracy,mean_expert_rank,mean_reciprocal_rank,mean_expert_probability
0,EntIRL,0.546667,15.693333,0.596295,0.264288
1,MoE-EntIRL,0.560000,15.160000,0.609265,0.269701
2,MoE-BiEntIRL-style,0.586667,15.360000,0.622469,0.256678


In [241]:
bientirl_theta_matrix = moe_bientirl_model.theta.detach().cpu().numpy()

bientirl_feature_importance_rows = []

for k in range(num_experts):
    for feature_idx, feature_name in enumerate(irl_feature_columns):
        theta_value = bientirl_theta_matrix[k, feature_idx]

        bientirl_feature_importance_rows.append({
            "expert_id": k,
            "feature": feature_name,
            "theta": theta_value,
            "abs_theta": abs(theta_value)
        })

bientirl_feature_importance_df = pd.DataFrame(bientirl_feature_importance_rows)

for k in range(num_experts):
    print("\nExpert", k, "- top absolute features")
    display(
        bientirl_feature_importance_df[bientirl_feature_importance_df["expert_id"] == k]
        .sort_values("abs_theta", ascending=False)
        .head(10)
    )

    print("Expert", k, "- top positive features")
    display(
        bientirl_feature_importance_df[bientirl_feature_importance_df["expert_id"] == k]
        .sort_values("theta", ascending=False)
        .head(10)
    )

    print("Expert", k, "- top negative features")
    display(
        bientirl_feature_importance_df[bientirl_feature_importance_df["expert_id"] == k]
        .sort_values("theta", ascending=True)
        .head(10)
    )


Expert 0 - top absolute features


,expert_id,feature,theta,abs_theta
5,0,degree_abs_diff,-6.765170,6.765170
15,0,user_n_neighbors,-3.063528,3.063528
2,0,user_degree,2.628760,2.628760
4,0,degree_sum,2.548898,2.548898
18,0,user_n_nonrumor_neighbors,1.422261,1.422261
16,0,user_n_message_neighbors,1.386325,1.386325
8,0,pagerank_sum,1.247085,1.247085
6,0,user_pagerank,1.244094,1.244094
17,0,user_n_rumor_neighbors,-0.871880,0.871880
1,0,step_norm,-0.822836,0.822836


Expert 0 - top positive features


,expert_id,feature,theta,abs_theta
2,0,user_degree,2.628760,2.628760
4,0,degree_sum,2.548898,2.548898
18,0,user_n_nonrumor_neighbors,1.422261,1.422261
16,0,user_n_message_neighbors,1.386325,1.386325
8,0,pagerank_sum,1.247085,1.247085
6,0,user_pagerank,1.244094,1.244094
22,0,rumor_clean_prob,0.719793,0.719793
14,0,rumor_attack_degree_norm,0.525731,0.525731
24,0,before_target_rumor_loss,0.473885,0.473885
7,0,rumor_pagerank,0.433865,0.433865


Expert 0 - top negative features


,expert_id,feature,theta,abs_theta
5,0,degree_abs_diff,-6.765170,6.765170
15,0,user_n_neighbors,-3.063528,3.063528
17,0,user_n_rumor_neighbors,-0.871880,0.871880
1,0,step_norm,-0.822836,0.822836
0,0,T,-0.543266,0.543266
26,0,before_rumor_f1,-0.536663,0.536663
25,0,before_accuracy,-0.468971,0.468971
23,0,before_avg_target_rumor_prob,-0.468424,0.468424
19,0,rumor_n_neighbors,-0.095392,0.095392
21,0,rumor_n_comment_neighbors,-0.095392,0.095392


In [242]:
bientirl_dir = os.path.join(results_dir, "moe_bientirl_outputs")
os.makedirs(bientirl_dir, exist_ok=True)

moe_bientirl_training_with_rewards_path = os.path.join(
    bientirl_dir,
    "moe_bientirl_training_with_rewards.csv"
)

moe_bientirl_ranking_path = os.path.join(
    bientirl_dir,
    "moe_bientirl_ranking_results.csv"
)

bientirl_feature_importance_path = os.path.join(
    bientirl_dir,
    "moe_bientirl_feature_importance.csv"
)

final_irl_model_comparison_path = os.path.join(
    bientirl_dir,
    "final_irl_model_comparison.csv"
)

moe_bientirl_model_path = os.path.join(
    bientirl_dir,
    "moe_bientirl_reward_model.pt"
)

moe_bientirl_losses_path = os.path.join(
    bientirl_dir,
    "moe_bientirl_training_losses.csv"
)

moe_bientirl_df.to_csv(moe_bientirl_training_with_rewards_path, index=False)
moe_bientirl_ranking_df.to_csv(moe_bientirl_ranking_path, index=False)
bientirl_feature_importance_df.to_csv(bientirl_feature_importance_path, index=False)
final_irl_model_comparison_df.to_csv(final_irl_model_comparison_path, index=False)

torch.save(moe_bientirl_model.state_dict(), moe_bientirl_model_path)

moe_bientirl_loss_df = pd.DataFrame({
    "epoch": list(range(1, len(moe_bientirl_losses) + 1)),
    "total_loss": moe_bientirl_losses,
    "likelihood_loss": moe_bientirl_likelihood_losses,
    "guidance_loss": moe_bientirl_guidance_losses,
    "inverse_loss": moe_bientirl_inverse_losses
})

moe_bientirl_loss_df.to_csv(moe_bientirl_losses_path, index=False)

print("Saved MoE-BiEntIRL-style outputs:")
print(moe_bientirl_training_with_rewards_path)
print(moe_bientirl_ranking_path)
print(bientirl_feature_importance_path)
print(final_irl_model_comparison_path)
print(moe_bientirl_model_path)
print(moe_bientirl_losses_path)

Saved MoE-BiEntIRL-style outputs:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_bientirl_outputs/moe_bientirl_training_with_rewards.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_bientirl_outputs/moe_bientirl_ranking_results.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_bientirl_outputs/moe_bientirl_feature_importance.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_bientirl_outputs/final_irl_model_comparison.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_bientirl_outputs/moe_bientirl_reward_model.pt
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/moe_bientirl_outputs/moe_bientirl_training_losses.csv


In [243]:



#NEW ATTACKS




In [244]:
from sklearn.metrics import accuracy_score, f1_score
import copy

def edge_index_from_graph(G, node_to_idx):
    edge_pairs = []

    for u, v in G.edges():
        if u in node_to_idx and v in node_to_idx:
            edge_pairs.append([node_to_idx[u], node_to_idx[v]])
            edge_pairs.append([node_to_idx[v], node_to_idx[u]])

    if len(edge_pairs) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    return torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()


def create_data_from_graph_for_generation(base_data, G_state, node_to_idx):
    data_state = base_data.clone()
    data_state.edge_index = edge_index_from_graph(G_state, node_to_idx)
    return data_state


def evaluate_generated_graph(G_state, base_data, node_to_idx, model, device, target_rumor_indices):
    data_state = create_data_from_graph_for_generation(
        base_data=base_data,
        G_state=G_state,
        node_to_idx=node_to_idx
    )

    data_state = data_state.to(device)
    model = model.to(device)
    model.eval()

    with torch.no_grad():
        try:
            out = model(data_state)
        except TypeError:
            out = model(data_state.x, data_state.edge_index)

        probs = F.softmax(out, dim=1)
        preds = out.argmax(dim=1)

    test_mask = data_state.test_mask.bool()

    y_true = data_state.y[test_mask].detach().cpu().numpy()
    y_pred = preds[test_mask].detach().cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    rumor_f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    if isinstance(target_rumor_indices, torch.Tensor):
        target_idx = target_rumor_indices.detach().clone().long().to(device)
    else:
        target_idx = torch.tensor(target_rumor_indices, dtype=torch.long, device=device)

    if target_idx.numel() > 0:
        target_logits = out[target_idx]
        target_labels = data_state.y[target_idx].long()
        target_loss = F.cross_entropy(target_logits, target_labels, reduction="mean").item()
        avg_target_rumor_prob = probs[target_idx, 1].mean().item()
        predicted_rumor = int((preds[target_idx] == 1).sum().item())
    else:
        target_loss = np.nan
        avg_target_rumor_prob = np.nan
        predicted_rumor = 0

    return {
        "accuracy": accuracy,
        "rumor_f1": rumor_f1,
        "target_rumor_loss": target_loss,
        "avg_target_rumor_prob": avg_target_rumor_prob,
        "predicted_rumor": predicted_rumor
    }


def score_candidate_actions_with_reward_model(candidate_feature_df, reward_model, scaler, irl_feature_columns, device):
    candidate_features = candidate_feature_df[irl_feature_columns].copy()

    for col in irl_feature_columns:
        candidate_features[col] = pd.to_numeric(candidate_features[col], errors="coerce")

    candidate_features = candidate_features.fillna(0.0)

    X_candidates = candidate_features.values.astype(np.float32)
    X_candidates_scaled = scaler.transform(X_candidates).astype(np.float32)

    X_candidates_tensor = torch.tensor(X_candidates_scaled, dtype=torch.float32).to(device)

    group_state_array = X_candidates_scaled.mean(axis=0, keepdims=True).astype(np.float32)
    group_state_tensor_single = torch.tensor(group_state_array, dtype=torch.float32).to(device)

    reward_model = reward_model.to(device)
    reward_model.eval()

    with torch.no_grad():
        expert_rewards_tensor, gate_probs_tensor = reward_model(
            X_candidates_tensor,
            group_state_tensor_single
        )

        expert_rewards_np = expert_rewards_tensor.detach().cpu().numpy()
        gate_probs_np = gate_probs_tensor.detach().cpu().numpy()[0]

    mixture_rewards = np.sum(expert_rewards_np * gate_probs_np.reshape(1, -1), axis=1)

    scored_df = candidate_feature_df.copy()
    scored_df["learned_reward"] = mixture_rewards
    scored_df["assigned_expert"] = int(np.argmax(gate_probs_np))

    return scored_df, gate_probs_np


print("Generation helper functions are ready.")

Generation helper functions are ready.


In [245]:
import inspect

def run_gcn_model(model, data_state):
    forward_signature = inspect.signature(model.forward)
    num_forward_args = len(forward_signature.parameters)

    if num_forward_args == 1:
        return model(data_state)

    return model(data_state.x, data_state.edge_index)


def get_model_rumor_probabilities(model, data, node_to_idx, device):
    model.eval()

    data_device = data.to(device)
    model = model.to(device)

    with torch.no_grad():
        out = run_gcn_model(model, data_device)
        probs = F.softmax(out, dim=1)[:, 1].detach().cpu().numpy()

    idx_to_node = {idx: node for node, idx in node_to_idx.items()}

    prob_by_node = {}
    for idx, prob in enumerate(probs):
        node = idx_to_node[idx]
        prob_by_node[node] = float(prob)

    return prob_by_node


def evaluate_generated_graph(G_state, base_data, node_to_idx, model, device, target_rumor_indices):
    data_state = create_data_from_graph_for_generation(
        base_data=base_data,
        G_state=G_state,
        node_to_idx=node_to_idx
    )

    data_state = data_state.to(device)
    model = model.to(device)
    model.eval()

    with torch.no_grad():
        out = run_gcn_model(model, data_state)
        probs = F.softmax(out, dim=1)
        preds = out.argmax(dim=1)

    test_mask = data_state.test_mask.bool()

    y_true = data_state.y[test_mask].detach().cpu().numpy()
    y_pred = preds[test_mask].detach().cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    rumor_f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    if isinstance(target_rumor_indices, torch.Tensor):
        target_idx = target_rumor_indices.detach().clone().long().to(device)
    else:
        target_idx = torch.tensor(target_rumor_indices, dtype=torch.long, device=device)

    if target_idx.numel() > 0:
        target_logits = out[target_idx]
        target_labels = data_state.y[target_idx].long()
        target_loss = F.cross_entropy(target_logits, target_labels, reduction="mean").item()
        avg_target_rumor_prob = probs[target_idx, 1].mean().item()
        predicted_rumor = int((preds[target_idx] == 1).sum().item())
    else:
        target_loss = np.nan
        avg_target_rumor_prob = np.nan
        predicted_rumor = 0

    return {
        "accuracy": accuracy,
        "rumor_f1": rumor_f1,
        "target_rumor_loss": target_loss,
        "avg_target_rumor_prob": avg_target_rumor_prob,
        "predicted_rumor": predicted_rumor
    }


def generate_trajectory_with_learned_reward(
    attack_name,
    T,
    G_base,
    base_data,
    node_to_idx,
    model,
    reward_model,
    scaler,
    irl_feature_columns,
    device,
    target_rumor_indices,
    candidate_sample_size=1000,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)

    G_state = G_base.copy()

    candidate_users, candidate_rumors = get_candidate_users_and_rumors(G_base)

    pagerank_dict = nx.pagerank(G_base.to_undirected())

    clean_prob_by_node = get_model_rumor_probabilities(
        model=model,
        data=base_data,
        node_to_idx=node_to_idx,
        device=device
    )

    user_attack_counter = {}
    rumor_attack_counter = {}

    generated_rows = []
    generated_edges = []

    for step in range(1, T + 1):
        before_metrics = evaluate_generated_graph(
            G_state=G_state,
            base_data=base_data,
            node_to_idx=node_to_idx,
            model=model,
            device=device,
            target_rumor_indices=target_rumor_indices
        )

        source_row = pd.Series({
            "before_avg_target_rumor_prob": before_metrics["avg_target_rumor_prob"],
            "before_target_rumor_loss": before_metrics["target_rumor_loss"],
            "before_accuracy": before_metrics["accuracy"],
            "before_rumor_f1": before_metrics["rumor_f1"]
        })

        candidate_rows = []
        used_pairs = set()
        attempts = 0
        max_attempts = candidate_sample_size * 50

        while len(candidate_rows) < candidate_sample_size and attempts < max_attempts:
            attempts += 1

            user_node = str(rng.choice(candidate_users))
            rumor_node = str(rng.choice(candidate_rumors))

            if (user_node, rumor_node) in used_pairs:
                continue

            if user_node not in G_state or rumor_node not in G_state:
                continue

            if G_state.has_edge(user_node, rumor_node):
                continue

            used_pairs.add((user_node, rumor_node))

            candidate_row = build_action_feature_row(
                attack_name=attack_name,
                T_value=T,
                step=step,
                user_node=user_node,
                rumor_node=rumor_node,
                is_expert=0,
                G_state=G_state,
                pagerank_dict=pagerank_dict,
                clean_prob_by_node=clean_prob_by_node,
                user_attack_counter=user_attack_counter,
                rumor_attack_counter=rumor_attack_counter,
                source_row=source_row
            )

            candidate_rows.append(candidate_row)

        candidate_feature_df = pd.DataFrame(candidate_rows)

        scored_candidate_df, gate_probs_np = score_candidate_actions_with_reward_model(
            candidate_feature_df=candidate_feature_df,
            reward_model=reward_model,
            scaler=scaler,
            irl_feature_columns=irl_feature_columns,
            device=device
        )

        best_action = scored_candidate_df.sort_values(
            "learned_reward",
            ascending=False
        ).iloc[0]

        user_node = best_action["user_node"]
        rumor_node = best_action["rumor_node"]

        G_state.add_edge(user_node, rumor_node)
        generated_edges.append((user_node, rumor_node))

        user_attack_counter[user_node] = user_attack_counter.get(user_node, 0) + 1
        rumor_attack_counter[rumor_node] = rumor_attack_counter.get(rumor_node, 0) + 1

        after_metrics = evaluate_generated_graph(
            G_state=G_state,
            base_data=base_data,
            node_to_idx=node_to_idx,
            model=model,
            device=device,
            target_rumor_indices=target_rumor_indices
        )

        generated_rows.append({
            "attack_name": attack_name,
            "T": T,
            "step": step,
            "user_node": user_node,
            "rumor_node": rumor_node,
            "learned_reward": best_action["learned_reward"],
            "assigned_expert": int(best_action["assigned_expert"]),

            "before_accuracy": before_metrics["accuracy"],
            "after_accuracy": after_metrics["accuracy"],
            "accuracy_change": after_metrics["accuracy"] - before_metrics["accuracy"],

            "before_rumor_f1": before_metrics["rumor_f1"],
            "after_rumor_f1": after_metrics["rumor_f1"],
            "rumor_f1_change": after_metrics["rumor_f1"] - before_metrics["rumor_f1"],

            "before_target_rumor_loss": before_metrics["target_rumor_loss"],
            "after_target_rumor_loss": after_metrics["target_rumor_loss"],
            "target_rumor_loss_change": after_metrics["target_rumor_loss"] - before_metrics["target_rumor_loss"],

            "before_avg_target_rumor_prob": before_metrics["avg_target_rumor_prob"],
            "after_avg_target_rumor_prob": after_metrics["avg_target_rumor_prob"],
            "avg_target_rumor_prob_change": after_metrics["avg_target_rumor_prob"] - before_metrics["avg_target_rumor_prob"],

            "before_predicted_rumor": before_metrics["predicted_rumor"],
            "after_predicted_rumor": after_metrics["predicted_rumor"]
        })

        print(
            f"{attack_name} | step {step}/{T} | "
            f"reward={best_action['learned_reward']:.4f} | "
            f"loss_change={after_metrics['target_rumor_loss'] - before_metrics['target_rumor_loss']:.6f} | "
            f"acc_change={after_metrics['accuracy'] - before_metrics['accuracy']:.6f}"
        )

    generated_df = pd.DataFrame(generated_rows)

    return generated_df, generated_edges, G_state


generated_moe_bientirl_T5_df, generated_moe_bientirl_edges_T5, G_generated_moe_bientirl_T5 = generate_trajectory_with_learned_reward(
    attack_name="Generated_MoE_BiEntIRL_T5",
    T=5,
    G_base=G_processed,
    base_data=data,
    node_to_idx=node_to_idx,
    model=model,
    reward_model=moe_bientirl_model,
    scaler=scaler,
    irl_feature_columns=irl_feature_columns,
    device=device,
    target_rumor_indices=target_rumor_indices,
    candidate_sample_size=1000,
    random_seed=42
)

generated_moe_bientirl_T20_df, generated_moe_bientirl_edges_T20, G_generated_moe_bientirl_T20 = generate_trajectory_with_learned_reward(
    attack_name="Generated_MoE_BiEntIRL_T20",
    T=20,
    G_base=G_processed,
    base_data=data,
    node_to_idx=node_to_idx,
    model=model,
    reward_model=moe_bientirl_model,
    scaler=scaler,
    irl_feature_columns=irl_feature_columns,
    device=device,
    target_rumor_indices=target_rumor_indices,
    candidate_sample_size=1000,
    random_seed=43
)

display(generated_moe_bientirl_T5_df)
display(generated_moe_bientirl_T20_df)

Generated_MoE_BiEntIRL_T5 | step 1/5 | reward=4.8336 | loss_change=0.000374 | acc_change=-0.002907
Generated_MoE_BiEntIRL_T5 | step 2/5 | reward=5.6302 | loss_change=0.000018 | acc_change=0.000000
Generated_MoE_BiEntIRL_T5 | step 3/5 | reward=7.7934 | loss_change=0.000000 | acc_change=0.000000
Generated_MoE_BiEntIRL_T5 | step 4/5 | reward=7.0169 | loss_change=0.000346 | acc_change=0.000000
Generated_MoE_BiEntIRL_T5 | step 5/5 | reward=10.3217 | loss_change=0.000000 | acc_change=0.000000
Generated_MoE_BiEntIRL_T20 | step 1/20 | reward=4.6922 | loss_change=0.000000 | acc_change=0.000000
Generated_MoE_BiEntIRL_T20 | step 2/20 | reward=2.0358 | loss_change=0.000313 | acc_change=-0.002907
Generated_MoE_BiEntIRL_T20 | step 3/20 | reward=5.1481 | loss_change=0.000001 | acc_change=0.000000
Generated_MoE_BiEntIRL_T20 | step 4/20 | reward=8.1656 | loss_change=0.000008 | acc_change=0.000000
Generated_MoE_BiEntIRL_T20 | step 5/20 | reward=9.4964 | loss_change=0.000000 | acc_change=0.000000
Generat

,attack_name,T,step,user_node,rumor_node,learned_reward,assigned_expert,before_accuracy,after_accuracy,accuracy_change,...,after_rumor_f1,rumor_f1_change,before_target_rumor_loss,after_target_rumor_loss,target_rumor_loss_change,before_avg_target_rumor_prob,after_avg_target_rumor_prob,avg_target_rumor_prob_change,before_predicted_rumor,after_predicted_rumor
0,Generated_MoE_BiEntIRL_T5,5,1,user_14849562,message_500359088757555200,4.833586,0,0.741279,0.738372,-0.002907,...,0.618644,-0.002633,0.662331,0.662705,0.000374,0.515824,0.515622,-0.000201,73,73
1,Generated_MoE_BiEntIRL_T5,5,2,user_279390084,message_500314015609147392,5.630210,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662705,0.662723,0.000018,0.515622,0.515613,-0.000010,73,73
2,Generated_MoE_BiEntIRL_T5,5,3,user_279390084,message_500332933098385408,7.793392,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662723,0.662723,0.000000,0.515613,0.515613,0.000000,73,73
3,Generated_MoE_BiEntIRL_T5,5,4,user_14849562,message_500247666924986369,7.016856,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662723,0.663070,0.000346,0.515613,0.515431,-0.000182,73,73
4,Generated_MoE_BiEntIRL_T5,5,5,user_279390084,message_500385868041842688,10.321735,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.663070,0.663070,0.000000,0.515431,0.515431,0.000000,73,73


,attack_name,T,step,user_node,rumor_node,learned_reward,assigned_expert,before_accuracy,after_accuracy,accuracy_change,...,after_rumor_f1,rumor_f1_change,before_target_rumor_loss,after_target_rumor_loss,target_rumor_loss_change,before_avg_target_rumor_prob,after_avg_target_rumor_prob,avg_target_rumor_prob_change,before_predicted_rumor,after_predicted_rumor
0,Generated_MoE_BiEntIRL_T20,20,1,user_279390084,message_500311153583853570,4.692225,0,0.741279,0.741279,0.000000,...,0.621277,0.000000,0.662331,0.662331,0.000000e+00,0.515824,0.515824,0.000000e+00,73,73
1,Generated_MoE_BiEntIRL_T20,20,2,user_7768402,message_500280249629036544,2.035768,0,0.741279,0.738372,-0.002907,...,0.618644,-0.002633,0.662331,0.662644,3.130436e-04,0.515824,0.515661,-1.627803e-04,73,73
2,Generated_MoE_BiEntIRL_T20,20,3,user_279390084,message_500367267901632512,5.148062,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662644,0.662645,1.370907e-06,0.515661,0.515660,-7.152557e-07,73,73
3,Generated_MoE_BiEntIRL_T20,20,4,user_279390084,message_498506343368519680,8.165638,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662645,0.662653,7.808208e-06,0.515660,0.515657,-3.933907e-06,73,73
4,Generated_MoE_BiEntIRL_T20,20,5,user_279390084,message_500325220193153024,9.496408,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662653,0.662653,0.000000e+00,0.515657,0.515657,0.000000e+00,73,73
5,Generated_MoE_BiEntIRL_T20,20,6,user_279390084,message_498254929942028288,10.214056,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662653,0.662653,-5.960464e-08,0.515657,0.515657,0.000000e+00,73,73
6,Generated_MoE_BiEntIRL_T20,20,7,user_7768402,message_500287857727377408,3.693140,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662653,0.662653,-5.960464e-08,0.515657,0.515657,0.000000e+00,73,73
7,Generated_MoE_BiEntIRL_T20,20,8,user_279390084,message_500300944891199488,12.881259,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662653,0.662653,0.000000e+00,0.515657,0.515657,0.000000e+00,73,73
8,Generated_MoE_BiEntIRL_T20,20,9,user_279390084,message_500369391339724802,13.565121,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662653,0.662675,2.193451e-05,0.515657,0.515645,-1.174212e-05,73,73
9,Generated_MoE_BiEntIRL_T20,20,10,user_279390084,message_500280422295937024,13.748791,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.662675,0.662675,0.000000e+00,0.515645,0.515645,0.000000e+00,73,73


In [246]:
generated_policy_summary_rows = []

for generated_df in [
    generated_moe_bientirl_T5_df,
    generated_moe_bientirl_T20_df
]:
    attack_name = generated_df["attack_name"].iloc[0]
    T_value = int(generated_df["T"].iloc[0])

    start_loss = generated_df["before_target_rumor_loss"].iloc[0]
    final_loss = generated_df["after_target_rumor_loss"].iloc[-1]

    start_acc = generated_df["before_accuracy"].iloc[0]
    final_acc = generated_df["after_accuracy"].iloc[-1]

    start_f1 = generated_df["before_rumor_f1"].iloc[0]
    final_f1 = generated_df["after_rumor_f1"].iloc[-1]

    generated_policy_summary_rows.append({
        "attack_name": attack_name,
        "T": T_value,
        "num_steps": len(generated_df),
        "target_loss_increase": final_loss - start_loss,
        "paper_delta_LA_clean_minus_final": start_loss - final_loss,
        "start_target_rumor_loss": start_loss,
        "final_target_rumor_loss": final_loss,
        "accuracy_change": final_acc - start_acc,
        "rumor_f1_change": final_f1 - start_f1,
        "start_accuracy": start_acc,
        "final_accuracy": final_acc,
        "start_rumor_f1": start_f1,
        "final_rumor_f1": final_f1
    })

generated_policy_summary_df = pd.DataFrame(generated_policy_summary_rows)

display(generated_policy_summary_df)

if "expert_trajectory_summary_df" in globals():
    comparison_generated_vs_expert_df = pd.concat(
        [
            expert_trajectory_summary_df.assign(source="expert_attack"),
            generated_policy_summary_df.assign(source="generated_policy")
        ],
        ignore_index=True
    )

    display(
        comparison_generated_vs_expert_df.sort_values(
            ["T", "target_loss_increase"],
            ascending=[True, False]
        )
    )

,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,start_target_rumor_loss,final_target_rumor_loss,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1
0,Generated_MoE_BiEntIRL_T5,5,5,0.000739,-0.000739,0.662331,0.663070,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644
1,Generated_MoE_BiEntIRL_T20,20,20,0.000525,-0.000525,0.662331,0.662856,-0.008721,-0.007831,0.741279,0.732558,0.621277,0.613445


,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,start_target_rumor_loss,final_target_rumor_loss,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1,source
2,Greedy_GC_RWCS_T5,5,5,0.002255,-0.002255,0.662006,0.664261,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644,expert_attack
4,Full_PR_BCD_T5,5,5,0.001883,-0.001883,0.662006,0.663889,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277,expert_attack
6,Generated_MoE_BiEntIRL_T5,5,5,0.000739,-0.000739,0.662331,0.663070,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644,generated_policy
0,PageRank_T5,5,5,-0.000185,0.000185,0.662006,0.661821,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119,expert_attack
5,Full_PR_BCD_T20,20,20,0.006344,-0.006344,0.662006,0.668350,-0.005814,-0.011835,0.741279,0.735465,0.621277,0.609442,expert_attack
3,Greedy_GC_RWCS_T20,20,20,0.005257,-0.005257,0.662006,0.667264,-0.005814,-0.005243,0.741279,0.735465,0.621277,0.616034,expert_attack
7,Generated_MoE_BiEntIRL_T20,20,20,0.000525,-0.000525,0.662331,0.662856,-0.008721,-0.007831,0.741279,0.732558,0.621277,0.613445,generated_policy
1,PageRank_T20,20,20,-0.000306,0.000306,0.662006,0.661701,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119,expert_attack


In [247]:
def generate_trajectory_with_reward_and_loss_check(
    attack_name,
    T,
    G_base,
    base_data,
    node_to_idx,
    model,
    reward_model,
    scaler,
    irl_feature_columns,
    device,
    target_rumor_indices,
    candidate_sample_size=1000,
    evaluation_top_k=50,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)

    G_state = G_base.copy()

    candidate_users, candidate_rumors = get_candidate_users_and_rumors(G_base)

    pagerank_dict = nx.pagerank(G_base.to_undirected())

    clean_prob_by_node = get_model_rumor_probabilities(
        model=model,
        data=base_data,
        node_to_idx=node_to_idx,
        device=device
    )

    user_attack_counter = {}
    rumor_attack_counter = {}

    generated_rows = []
    generated_edges = []

    for step in range(1, T + 1):
        before_metrics = evaluate_generated_graph(
            G_state=G_state,
            base_data=base_data,
            node_to_idx=node_to_idx,
            model=model,
            device=device,
            target_rumor_indices=target_rumor_indices
        )

        source_row = pd.Series({
            "before_avg_target_rumor_prob": before_metrics["avg_target_rumor_prob"],
            "before_target_rumor_loss": before_metrics["target_rumor_loss"],
            "before_accuracy": before_metrics["accuracy"],
            "before_rumor_f1": before_metrics["rumor_f1"]
        })

        candidate_rows = []
        used_pairs = set()
        attempts = 0
        max_attempts = candidate_sample_size * 50

        while len(candidate_rows) < candidate_sample_size and attempts < max_attempts:
            attempts += 1

            user_node = str(rng.choice(candidate_users))
            rumor_node = str(rng.choice(candidate_rumors))

            if (user_node, rumor_node) in used_pairs:
                continue

            if user_node not in G_state or rumor_node not in G_state:
                continue

            if G_state.has_edge(user_node, rumor_node):
                continue

            used_pairs.add((user_node, rumor_node))

            candidate_row = build_action_feature_row(
                attack_name=attack_name,
                T_value=T,
                step=step,
                user_node=user_node,
                rumor_node=rumor_node,
                is_expert=0,
                G_state=G_state,
                pagerank_dict=pagerank_dict,
                clean_prob_by_node=clean_prob_by_node,
                user_attack_counter=user_attack_counter,
                rumor_attack_counter=rumor_attack_counter,
                source_row=source_row
            )

            candidate_rows.append(candidate_row)

        candidate_feature_df = pd.DataFrame(candidate_rows)

        scored_candidate_df, gate_probs_np = score_candidate_actions_with_reward_model(
            candidate_feature_df=candidate_feature_df,
            reward_model=reward_model,
            scaler=scaler,
            irl_feature_columns=irl_feature_columns,
            device=device
        )

        top_candidate_df = scored_candidate_df.sort_values(
            "learned_reward",
            ascending=False
        ).head(evaluation_top_k).copy()

        evaluated_rows = []

        for _, candidate_row in top_candidate_df.iterrows():
            user_node = candidate_row["user_node"]
            rumor_node = candidate_row["rumor_node"]

            G_temp = G_state.copy()
            G_temp.add_edge(user_node, rumor_node)

            after_metrics = evaluate_generated_graph(
                G_state=G_temp,
                base_data=base_data,
                node_to_idx=node_to_idx,
                model=model,
                device=device,
                target_rumor_indices=target_rumor_indices
            )

            evaluated_row = candidate_row.to_dict()
            evaluated_row["candidate_after_accuracy"] = after_metrics["accuracy"]
            evaluated_row["candidate_after_rumor_f1"] = after_metrics["rumor_f1"]
            evaluated_row["candidate_after_target_rumor_loss"] = after_metrics["target_rumor_loss"]
            evaluated_row["candidate_after_avg_target_rumor_prob"] = after_metrics["avg_target_rumor_prob"]
            evaluated_row["candidate_after_predicted_rumor"] = after_metrics["predicted_rumor"]
            evaluated_row["candidate_target_loss_change"] = after_metrics["target_rumor_loss"] - before_metrics["target_rumor_loss"]
            evaluated_row["candidate_accuracy_change"] = after_metrics["accuracy"] - before_metrics["accuracy"]
            evaluated_row["candidate_rumor_f1_change"] = after_metrics["rumor_f1"] - before_metrics["rumor_f1"]

            evaluated_rows.append(evaluated_row)

        evaluated_candidate_df = pd.DataFrame(evaluated_rows)

        best_action = evaluated_candidate_df.sort_values(
            ["candidate_target_loss_change", "learned_reward"],
            ascending=[False, False]
        ).iloc[0]

        user_node = best_action["user_node"]
        rumor_node = best_action["rumor_node"]

        G_state.add_edge(user_node, rumor_node)
        generated_edges.append((user_node, rumor_node))

        user_attack_counter[user_node] = user_attack_counter.get(user_node, 0) + 1
        rumor_attack_counter[rumor_node] = rumor_attack_counter.get(rumor_node, 0) + 1

        after_metrics = evaluate_generated_graph(
            G_state=G_state,
            base_data=base_data,
            node_to_idx=node_to_idx,
            model=model,
            device=device,
            target_rumor_indices=target_rumor_indices
        )

        generated_rows.append({
            "attack_name": attack_name,
            "T": T,
            "step": step,
            "user_node": user_node,
            "rumor_node": rumor_node,
            "learned_reward": best_action["learned_reward"],
            "assigned_expert": int(best_action["assigned_expert"]),

            "before_accuracy": before_metrics["accuracy"],
            "after_accuracy": after_metrics["accuracy"],
            "accuracy_change": after_metrics["accuracy"] - before_metrics["accuracy"],

            "before_rumor_f1": before_metrics["rumor_f1"],
            "after_rumor_f1": after_metrics["rumor_f1"],
            "rumor_f1_change": after_metrics["rumor_f1"] - before_metrics["rumor_f1"],

            "before_target_rumor_loss": before_metrics["target_rumor_loss"],
            "after_target_rumor_loss": after_metrics["target_rumor_loss"],
            "target_rumor_loss_change": after_metrics["target_rumor_loss"] - before_metrics["target_rumor_loss"],

            "before_avg_target_rumor_prob": before_metrics["avg_target_rumor_prob"],
            "after_avg_target_rumor_prob": after_metrics["avg_target_rumor_prob"],
            "avg_target_rumor_prob_change": after_metrics["avg_target_rumor_prob"] - before_metrics["avg_target_rumor_prob"],

            "before_predicted_rumor": before_metrics["predicted_rumor"],
            "after_predicted_rumor": after_metrics["predicted_rumor"]
        })

        print(
            f"{attack_name} | step {step}/{T} | "
            f"reward={best_action['learned_reward']:.4f} | "
            f"loss_change={after_metrics['target_rumor_loss'] - before_metrics['target_rumor_loss']:.6f} | "
            f"acc_change={after_metrics['accuracy'] - before_metrics['accuracy']:.6f}"
        )

    generated_df = pd.DataFrame(generated_rows)

    return generated_df, generated_edges, G_state


generated_lossaware_T5_df, generated_lossaware_edges_T5, G_generated_lossaware_T5 = generate_trajectory_with_reward_and_loss_check(
    attack_name="Generated_LossAware_MoE_BiEntIRL_T5",
    T=5,
    G_base=G_processed,
    base_data=data,
    node_to_idx=node_to_idx,
    model=model,
    reward_model=moe_bientirl_model,
    scaler=scaler,
    irl_feature_columns=irl_feature_columns,
    device=device,
    target_rumor_indices=target_rumor_indices,
    candidate_sample_size=1000,
    evaluation_top_k=50,
    random_seed=52
)

display(generated_lossaware_T5_df)

Generated_LossAware_MoE_BiEntIRL_T5 | step 1/5 | reward=1.8611 | loss_change=0.000459 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T5 | step 2/5 | reward=1.1248 | loss_change=0.000440 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T5 | step 3/5 | reward=1.8417 | loss_change=0.000363 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T5 | step 4/5 | reward=0.3978 | loss_change=0.000277 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T5 | step 5/5 | reward=0.1241 | loss_change=0.000327 | acc_change=0.000000


,attack_name,T,step,user_node,rumor_node,learned_reward,assigned_expert,before_accuracy,after_accuracy,accuracy_change,...,after_rumor_f1,rumor_f1_change,before_target_rumor_loss,after_target_rumor_loss,target_rumor_loss_change,before_avg_target_rumor_prob,after_avg_target_rumor_prob,avg_target_rumor_prob_change,before_predicted_rumor,after_predicted_rumor
0,Generated_LossAware_MoE_BiEntIRL_T5,5,1,user_22416658,message_500359377585704961,1.861147,0,0.741279,0.741279,0.0,...,0.621277,0.0,0.662331,0.662790,0.000459,0.515824,0.515579,-0.000245,73,73
1,Generated_LossAware_MoE_BiEntIRL_T5,5,2,user_44361097,message_500278858156085248,1.124784,0,0.741279,0.741279,0.0,...,0.621277,0.0,0.662790,0.663230,0.000440,0.515579,0.515349,-0.000230,73,73
2,Generated_LossAware_MoE_BiEntIRL_T5,5,3,user_27966935,message_500359088757555200,1.841701,0,0.741279,0.741279,0.0,...,0.621277,0.0,0.663230,0.663594,0.000363,0.515349,0.515153,-0.000196,73,73
3,Generated_LossAware_MoE_BiEntIRL_T5,5,4,user_24500377,message_500301519854379008,0.397824,0,0.741279,0.741279,0.0,...,0.621277,0.0,0.663594,0.663871,0.000277,0.515153,0.515006,-0.000147,73,73
4,Generated_LossAware_MoE_BiEntIRL_T5,5,5,user_14715424,message_500309381930844160,0.124060,0,0.741279,0.741279,0.0,...,0.621277,0.0,0.663871,0.664198,0.000327,0.515006,0.514836,-0.000170,73,73


In [248]:
generated_lossaware_T20_df, generated_lossaware_edges_T20, G_generated_lossaware_T20 = generate_trajectory_with_reward_and_loss_check(
    attack_name="Generated_LossAware_MoE_BiEntIRL_T20",
    T=20,
    G_base=G_processed,
    base_data=data,
    node_to_idx=node_to_idx,
    model=model,
    reward_model=moe_bientirl_model,
    scaler=scaler,
    irl_feature_columns=irl_feature_columns,
    device=device,
    target_rumor_indices=target_rumor_indices,
    candidate_sample_size=1000,
    evaluation_top_k=50,
    random_seed=53
)

display(generated_lossaware_T20_df)

Generated_LossAware_MoE_BiEntIRL_T20 | step 1/20 | reward=0.6883 | loss_change=0.000459 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 2/20 | reward=1.2550 | loss_change=0.000389 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 3/20 | reward=2.8605 | loss_change=0.000287 | acc_change=-0.002907
Generated_LossAware_MoE_BiEntIRL_T20 | step 4/20 | reward=2.5422 | loss_change=0.000389 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 5/20 | reward=2.1212 | loss_change=0.000345 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 6/20 | reward=1.5495 | loss_change=0.000350 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 7/20 | reward=1.6477 | loss_change=0.000510 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 8/20 | reward=2.0081 | loss_change=0.000267 | acc_change=0.000000
Generated_LossAware_MoE_BiEntIRL_T20 | step 9/20 | reward=2.5167 | loss_change=0.000211 | acc_change=0.000000
Generated

,attack_name,T,step,user_node,rumor_node,learned_reward,assigned_expert,before_accuracy,after_accuracy,accuracy_change,...,after_rumor_f1,rumor_f1_change,before_target_rumor_loss,after_target_rumor_loss,target_rumor_loss_change,before_avg_target_rumor_prob,after_avg_target_rumor_prob,avg_target_rumor_prob_change,before_predicted_rumor,after_predicted_rumor
0,Generated_LossAware_MoE_BiEntIRL_T20,20,1,user_8191652,message_500359377585704961,0.688317,0,0.741279,0.741279,0.000000,...,0.621277,0.000000,0.662331,0.662790,0.000459,0.515824,0.515578,-0.000246,73,73
1,Generated_LossAware_MoE_BiEntIRL_T20,20,2,user_16560657,message_500359088757555200,1.255050,0,0.741279,0.741279,0.000000,...,0.621277,0.000000,0.662790,0.663179,0.000389,0.515578,0.515369,-0.000209,73,73
2,Generated_LossAware_MoE_BiEntIRL_T20,20,3,user_14849562,message_500335355904540674,2.860486,0,0.741279,0.738372,-0.002907,...,0.618644,-0.002633,0.663179,0.663467,0.000287,0.515369,0.515220,-0.000149,73,73
3,Generated_LossAware_MoE_BiEntIRL_T20,20,4,user_13393052,message_500275363558064128,2.542199,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.663467,0.663855,0.000389,0.515220,0.515020,-0.000200,73,73
4,Generated_LossAware_MoE_BiEntIRL_T20,20,5,user_534675241,message_500301519854379008,2.121227,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.663855,0.664200,0.000345,0.515020,0.514838,-0.000183,73,73
5,Generated_LossAware_MoE_BiEntIRL_T20,20,6,user_14267557,message_500326267640905728,1.549461,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.664200,0.664550,0.000350,0.514838,0.514653,-0.000184,73,73
6,Generated_LossAware_MoE_BiEntIRL_T20,20,7,user_15907183,message_500278858156085248,1.647683,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.664550,0.665060,0.000510,0.514653,0.514388,-0.000265,73,73
7,Generated_LossAware_MoE_BiEntIRL_T20,20,8,user_1426406143,message_500360095738626049,2.008066,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.665060,0.665327,0.000267,0.514388,0.514244,-0.000144,73,73
8,Generated_LossAware_MoE_BiEntIRL_T20,20,9,user_110363115,message_500359377585704961,2.516700,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.665327,0.665538,0.000211,0.514244,0.514135,-0.000110,73,73
9,Generated_LossAware_MoE_BiEntIRL_T20,20,10,user_14849562,message_500280249629036544,5.334073,0,0.738372,0.738372,0.000000,...,0.618644,0.000000,0.665538,0.665851,0.000313,0.514135,0.513972,-0.000163,73,73


In [249]:
lossaware_summary_rows = []

for generated_df in [
    generated_lossaware_T5_df,
    generated_lossaware_T20_df
]:
    attack_name = generated_df["attack_name"].iloc[0]
    T_value = int(generated_df["T"].iloc[0])

    start_loss = generated_df["before_target_rumor_loss"].iloc[0]
    final_loss = generated_df["after_target_rumor_loss"].iloc[-1]

    start_acc = generated_df["before_accuracy"].iloc[0]
    final_acc = generated_df["after_accuracy"].iloc[-1]

    start_f1 = generated_df["before_rumor_f1"].iloc[0]
    final_f1 = generated_df["after_rumor_f1"].iloc[-1]

    lossaware_summary_rows.append({
        "attack_name": attack_name,
        "T": T_value,
        "num_steps": len(generated_df),
        "target_loss_increase": final_loss - start_loss,
        "paper_delta_LA_clean_minus_final": start_loss - final_loss,
        "start_target_rumor_loss": start_loss,
        "final_target_rumor_loss": final_loss,
        "accuracy_change": final_acc - start_acc,
        "rumor_f1_change": final_f1 - start_f1,
        "start_accuracy": start_acc,
        "final_accuracy": final_acc,
        "start_rumor_f1": start_f1,
        "final_rumor_f1": final_f1
    })

lossaware_summary_df = pd.DataFrame(lossaware_summary_rows)

display(lossaware_summary_df)

comparison_with_lossaware_df = pd.concat(
    [
        expert_trajectory_summary_df.assign(source="expert_attack"),
        generated_policy_summary_df.assign(source="generated_reward_only"),
        lossaware_summary_df.assign(source="generated_reward_lossaware")
    ],
    ignore_index=True
)

display(
    comparison_with_lossaware_df.sort_values(
        ["T", "target_loss_increase"],
        ascending=[True, False]
    )
)

,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,start_target_rumor_loss,final_target_rumor_loss,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1
0,Generated_LossAware_MoE_BiEntIRL_T5,5,5,0.001867,-0.001867,0.662331,0.664198,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277
1,Generated_LossAware_MoE_BiEntIRL_T20,20,20,0.006008,-0.006008,0.662331,0.668339,-0.008721,-0.011107,0.741279,0.732558,0.621277,0.610169


,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,start_target_rumor_loss,final_target_rumor_loss,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1,source
2,Greedy_GC_RWCS_T5,5,5,0.002255,-0.002255,0.662006,0.664261,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644,expert_attack
4,Full_PR_BCD_T5,5,5,0.001883,-0.001883,0.662006,0.663889,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277,expert_attack
8,Generated_LossAware_MoE_BiEntIRL_T5,5,5,0.001867,-0.001867,0.662331,0.664198,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277,generated_reward_lossaware
6,Generated_MoE_BiEntIRL_T5,5,5,0.000739,-0.000739,0.662331,0.663070,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644,generated_reward_only
0,PageRank_T5,5,5,-0.000185,0.000185,0.662006,0.661821,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119,expert_attack
5,Full_PR_BCD_T20,20,20,0.006344,-0.006344,0.662006,0.668350,-0.005814,-0.011835,0.741279,0.735465,0.621277,0.609442,expert_attack
9,Generated_LossAware_MoE_BiEntIRL_T20,20,20,0.006008,-0.006008,0.662331,0.668339,-0.008721,-0.011107,0.741279,0.732558,0.621277,0.610169,generated_reward_lossaware
3,Greedy_GC_RWCS_T20,20,20,0.005257,-0.005257,0.662006,0.667264,-0.005814,-0.005243,0.741279,0.735465,0.621277,0.616034,expert_attack
7,Generated_MoE_BiEntIRL_T20,20,20,0.000525,-0.000525,0.662331,0.662856,-0.008721,-0.007831,0.741279,0.732558,0.621277,0.613445,generated_reward_only
1,PageRank_T20,20,20,-0.000306,0.000306,0.662006,0.661701,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119,expert_attack


In [250]:
lossaware_generated_samples_dir = os.path.join(
    results_dir,
    "lossaware_generated_policy_samples"
)

os.makedirs(lossaware_generated_samples_dir, exist_ok=True)

generated_lossaware_T5_path = os.path.join(
    lossaware_generated_samples_dir,
    "generated_lossaware_moe_bientirl_T5.csv"
)

generated_lossaware_T20_path = os.path.join(
    lossaware_generated_samples_dir,
    "generated_lossaware_moe_bientirl_T20.csv"
)

lossaware_summary_path = os.path.join(
    lossaware_generated_samples_dir,
    "lossaware_generated_policy_summary.csv"
)

comparison_with_lossaware_path = os.path.join(
    lossaware_generated_samples_dir,
    "comparison_with_lossaware_generated_policy.csv"
)

lossaware_edges_T5_path = os.path.join(
    lossaware_generated_samples_dir,
    "generated_lossaware_edges_T5.pkl"
)

lossaware_edges_T20_path = os.path.join(
    lossaware_generated_samples_dir,
    "generated_lossaware_edges_T20.pkl"
)

generated_lossaware_T5_df.to_csv(generated_lossaware_T5_path, index=False)
generated_lossaware_T20_df.to_csv(generated_lossaware_T20_path, index=False)
lossaware_summary_df.to_csv(lossaware_summary_path, index=False)
comparison_with_lossaware_df.to_csv(comparison_with_lossaware_path, index=False)

with open(lossaware_edges_T5_path, "wb") as f:
    pickle.dump(generated_lossaware_edges_T5, f)

with open(lossaware_edges_T20_path, "wb") as f:
    pickle.dump(generated_lossaware_edges_T20, f)

print("Saved loss-aware generated samples:")
print(generated_lossaware_T5_path)
print(generated_lossaware_T20_path)
print(lossaware_summary_path)
print(comparison_with_lossaware_path)
print(lossaware_edges_T5_path)
print(lossaware_edges_T20_path)

Saved loss-aware generated samples:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/lossaware_generated_policy_samples/generated_lossaware_moe_bientirl_T5.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/lossaware_generated_policy_samples/generated_lossaware_moe_bientirl_T20.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/lossaware_generated_policy_samples/lossaware_generated_policy_summary.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/lossaware_generated_policy_samples/comparison_with_lossaware_generated_policy.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/lossaware_generated_policy_samples/generated_lossaware_edges_T5.pkl
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/lossaware_generated_policy_samples/generated_lossaware_edges_T20.pkl


In [251]:


#DEFENCE



In [252]:
import copy
from sklearn.metrics import accuracy_score, f1_score

def apply_edge_sequence_to_graph(G_base, edge_sequence):
    G_new = G_base.copy()

    for edge in edge_sequence:
        user_node, rumor_node = edge

        if user_node in G_new and rumor_node in G_new:
            G_new.add_edge(user_node, rumor_node)

    return G_new


def compute_training_class_weights(data, device):
    train_labels = data.y[data.train_mask.bool()].detach().cpu().numpy()

    count_0 = np.sum(train_labels == 0)
    count_1 = np.sum(train_labels == 1)

    total = count_0 + count_1

    weight_0 = total / max(2 * count_0, 1)
    weight_1 = total / max(2 * count_1, 1)

    return torch.tensor([weight_0, weight_1], dtype=torch.float32, device=device)


def evaluate_defense_model_on_graph(defense_model, G_eval, base_data, node_to_idx, device, target_rumor_indices):
    data_eval = create_data_from_graph_for_generation(
        base_data=base_data,
        G_state=G_eval,
        node_to_idx=node_to_idx
    )

    data_eval = data_eval.to(device)
    defense_model = defense_model.to(device)
    defense_model.eval()

    with torch.no_grad():
        out = run_gcn_model(defense_model, data_eval)
        probs = F.softmax(out, dim=1)
        preds = out.argmax(dim=1)

    test_mask = data_eval.test_mask.bool()

    y_true = data_eval.y[test_mask].detach().cpu().numpy()
    y_pred = preds[test_mask].detach().cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    rumor_f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    if isinstance(target_rumor_indices, torch.Tensor):
        target_idx = target_rumor_indices.detach().clone().long().to(device)
    else:
        target_idx = torch.tensor(target_rumor_indices, dtype=torch.long, device=device)

    if target_idx.numel() > 0:
        target_logits = out[target_idx]
        target_labels = data_eval.y[target_idx].long()
        target_loss = F.cross_entropy(target_logits, target_labels, reduction="mean").item()
        avg_target_rumor_prob = probs[target_idx, 1].mean().item()
        predicted_rumor = int((preds[target_idx] == 1).sum().item())
    else:
        target_loss = np.nan
        avg_target_rumor_prob = np.nan
        predicted_rumor = 0

    return {
        "accuracy": accuracy,
        "rumor_f1": rumor_f1,
        "target_rumor_loss": target_loss,
        "avg_target_rumor_prob": avg_target_rumor_prob,
        "predicted_rumor": predicted_rumor
    }


def train_defense_model_with_adversarial_graphs(
    base_model,
    base_data,
    adversarial_graphs,
    node_to_idx,
    device,
    beta=PAPER_DEFENSE_BETA,
    epochs=PAPER_PHEME_EPOCHS,
    lr=PAPER_TARGET_MODEL_LR,
    weight_decay=5e-4,
    print_every=10
):
    defense_model = copy.deepcopy(base_model).to(device)

    for param in defense_model.parameters():
        param.requires_grad = True

    clean_data_device = base_data.clone().to(device)

    adversarial_data_list = []

    for G_adv in adversarial_graphs:
        data_adv = create_data_from_graph_for_generation(
            base_data=base_data,
            G_state=G_adv,
            node_to_idx=node_to_idx
        )
        adversarial_data_list.append(data_adv.to(device))

    class_weights = compute_training_class_weights(clean_data_device, device)

    optimizer = torch.optim.Adam(
        [param for param in defense_model.parameters() if param.requires_grad],
        lr=lr,
        weight_decay=weight_decay
    )

    train_mask = clean_data_device.train_mask.bool()

    training_rows = []

    for epoch in range(1, epochs + 1):
        defense_model.train()
        optimizer.zero_grad()

        clean_out = run_gcn_model(defense_model, clean_data_device)
        clean_loss = F.cross_entropy(
            clean_out[train_mask],
            clean_data_device.y[train_mask].long(),
            weight=class_weights
        )

        adversarial_losses = []

        for data_adv in adversarial_data_list:
            adv_out = run_gcn_model(defense_model, data_adv)
            adv_loss = F.cross_entropy(
                adv_out[train_mask],
                data_adv.y[train_mask].long(),
                weight=class_weights
            )
            adversarial_losses.append(adv_loss)

        if len(adversarial_losses) > 0:
            adversarial_loss = torch.stack(adversarial_losses).mean()
        else:
            adversarial_loss = torch.tensor(0.0, device=device)

        total_loss = clean_loss + beta * adversarial_loss

        total_loss.backward()
        optimizer.step()

        training_rows.append({
            "epoch": epoch,
            "total_loss": float(total_loss.detach().cpu()),
            "clean_loss": float(clean_loss.detach().cpu()),
            "adversarial_loss": float(adversarial_loss.detach().cpu())
        })

        if epoch % print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | "
                f"total={training_rows[-1]['total_loss']:.6f} | "
                f"clean={training_rows[-1]['clean_loss']:.6f} | "
                f"adv={training_rows[-1]['adversarial_loss']:.6f}"
            )

    training_df = pd.DataFrame(training_rows)

    return defense_model, training_df


print("Defense helper functions are ready.")

Defense helper functions are ready.


In [253]:
expert_T5_edges = []

for edge_list_name in [
    "added_edges",
    "greedy_added_edges",
    "full_prbcd_edges_T5"
]:
    if edge_list_name in globals():
        expert_T5_edges.extend(globals()[edge_list_name])

expert_T5_edges = list(dict.fromkeys(expert_T5_edges))

G_expert_T5 = apply_edge_sequence_to_graph(
    G_base=G_processed,
    edge_sequence=expert_T5_edges
)

G_generated_lossaware_T5_for_defense = apply_edge_sequence_to_graph(
    G_base=G_processed,
    edge_sequence=generated_lossaware_edges_T5
)

G_generated_lossaware_T20_for_defense = apply_edge_sequence_to_graph(
    G_base=G_processed,
    edge_sequence=generated_lossaware_edges_T20
)

defense_models = {}
defense_training_logs = {}

defense_models["w/o Def."] = copy.deepcopy(model)

defense_models["EDA_expert_T5"], defense_training_logs["EDA_expert_T5"] = train_defense_model_with_adversarial_graphs(
    base_model=model,
    base_data=data,
    adversarial_graphs=[G_expert_T5],
    node_to_idx=node_to_idx,
    device=device,
    beta=PAPER_DEFENSE_BETA,
    epochs=PAPER_PHEME_EPOCHS,
    lr=PAPER_TARGET_MODEL_LR,
    weight_decay=5e-4,
    print_every=10
)

defense_models["DA_lossaware_T5"], defense_training_logs["DA_lossaware_T5"] = train_defense_model_with_adversarial_graphs(
    base_model=model,
    base_data=data,
    adversarial_graphs=[G_generated_lossaware_T5_for_defense],
    node_to_idx=node_to_idx,
    device=device,
    beta=PAPER_DEFENSE_BETA,
    epochs=PAPER_PHEME_EPOCHS,
    lr=PAPER_TARGET_MODEL_LR,
    weight_decay=5e-4,
    print_every=10
)

defense_models["AT_lossaware_T5_T20"], defense_training_logs["AT_lossaware_T5_T20"] = train_defense_model_with_adversarial_graphs(
    base_model=model,
    base_data=data,
    adversarial_graphs=[
        G_generated_lossaware_T5_for_defense,
        G_generated_lossaware_T20_for_defense
    ],
    node_to_idx=node_to_idx,
    device=device,
    beta=PAPER_DEFENSE_BETA,
    epochs=PAPER_PHEME_EPOCHS,
    lr=PAPER_TARGET_MODEL_LR,
    weight_decay=5e-4,
    print_every=10
)

print("Finished training defense models.")
print(list(defense_models.keys()))

Epoch 001 | total=6.034264 | clean=0.672970 | adv=0.670162
Epoch 010 | total=6.008535 | clean=0.665199 | adv=0.667917
Epoch 020 | total=5.970434 | clean=0.665383 | adv=0.663131
Epoch 030 | total=5.932499 | clean=0.657700 | adv=0.659350
Epoch 040 | total=5.909179 | clean=0.656411 | adv=0.656596
Epoch 050 | total=5.869178 | clean=0.654586 | adv=0.651824
Epoch 060 | total=5.857135 | clean=0.652249 | adv=0.650611
Epoch 070 | total=5.815259 | clean=0.644122 | adv=0.646392
Epoch 080 | total=5.811105 | clean=0.645890 | adv=0.645652
Epoch 090 | total=5.786185 | clean=0.641740 | adv=0.643056
Epoch 100 | total=5.749377 | clean=0.638754 | adv=0.638828
Epoch 110 | total=5.692221 | clean=0.633179 | adv=0.632380
Epoch 120 | total=5.675461 | clean=0.630639 | adv=0.630603
Epoch 001 | total=6.016589 | clean=0.669266 | adv=0.668415
Epoch 010 | total=5.998684 | clean=0.661617 | adv=0.667133
Epoch 020 | total=5.976809 | clean=0.660673 | adv=0.664517
Epoch 030 | total=5.976328 | clean=0.661410 | adv=0.6643

In [254]:
attack_eval_graphs = {}

attack_eval_graphs["Clean"] = G_processed

if "added_edges" in globals():
    attack_eval_graphs["PageRank_T5"] = apply_edge_sequence_to_graph(
        G_processed,
        added_edges
    )

if "greedy_added_edges" in globals():
    attack_eval_graphs["Greedy_GC_RWCS_T5"] = apply_edge_sequence_to_graph(
        G_processed,
        greedy_added_edges
    )

if "full_prbcd_edges_T5" in globals():
    attack_eval_graphs["Full_PR_BCD_T5"] = apply_edge_sequence_to_graph(
        G_processed,
        full_prbcd_edges_T5
    )

attack_eval_graphs["Generated_LossAware_T5"] = apply_edge_sequence_to_graph(
    G_processed,
    generated_lossaware_edges_T5
)

defense_eval_rows = []

for defense_name, defense_model in defense_models.items():
    clean_metrics = evaluate_defense_model_on_graph(
        defense_model=defense_model,
        G_eval=G_processed,
        base_data=data,
        node_to_idx=node_to_idx,
        device=device,
        target_rumor_indices=target_rumor_indices
    )

    for attack_name, G_eval in attack_eval_graphs.items():
        metrics = evaluate_defense_model_on_graph(
            defense_model=defense_model,
            G_eval=G_eval,
            base_data=data,
            node_to_idx=node_to_idx,
            device=device,
            target_rumor_indices=target_rumor_indices
        )

        defense_eval_rows.append({
            "defense_name": defense_name,
            "attack_name": attack_name,

            "clean_accuracy": clean_metrics["accuracy"],
            "attacked_accuracy": metrics["accuracy"],
            "accuracy_change_from_clean": metrics["accuracy"] - clean_metrics["accuracy"],

            "clean_rumor_f1": clean_metrics["rumor_f1"],
            "attacked_rumor_f1": metrics["rumor_f1"],
            "rumor_f1_change_from_clean": metrics["rumor_f1"] - clean_metrics["rumor_f1"],

            "clean_target_rumor_loss": clean_metrics["target_rumor_loss"],
            "attacked_target_rumor_loss": metrics["target_rumor_loss"],
            "target_loss_increase_from_clean": metrics["target_rumor_loss"] - clean_metrics["target_rumor_loss"],

            "clean_avg_target_rumor_prob": clean_metrics["avg_target_rumor_prob"],
            "attacked_avg_target_rumor_prob": metrics["avg_target_rumor_prob"],
            "avg_target_rumor_prob_change_from_clean": metrics["avg_target_rumor_prob"] - clean_metrics["avg_target_rumor_prob"]
        })

defense_eval_df = pd.DataFrame(defense_eval_rows)

display(defense_eval_df)

accuracy_table = defense_eval_df.pivot(
    index="defense_name",
    columns="attack_name",
    values="accuracy_change_from_clean"
).reset_index()

loss_table = defense_eval_df.pivot(
    index="defense_name",
    columns="attack_name",
    values="target_loss_increase_from_clean"
).reset_index()

print("Accuracy change from clean graph:")
display(accuracy_table)

print("Target rumor loss increase from clean graph:")
display(loss_table)

,defense_name,attack_name,clean_accuracy,attacked_accuracy,accuracy_change_from_clean,clean_rumor_f1,attacked_rumor_f1,rumor_f1_change_from_clean,clean_target_rumor_loss,attacked_target_rumor_loss,target_loss_increase_from_clean,clean_avg_target_rumor_prob,attacked_avg_target_rumor_prob,avg_target_rumor_prob_change_from_clean
0,w/o Def.,Clean,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662331,0.662331,0.000000,0.515824,0.515824,0.000000
1,w/o Def.,PageRank_T5,0.741279,0.744186,0.002907,0.621277,0.627119,0.005842,0.662331,0.662153,-0.000178,0.515824,0.515913,0.000089
2,w/o Def.,Greedy_GC_RWCS_T5,0.741279,0.738372,-0.002907,0.621277,0.618644,-0.002633,0.662331,0.664588,0.002257,0.515824,0.514631,-0.001193
3,w/o Def.,Full_PR_BCD_T5,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662331,0.664213,0.001882,0.515824,0.514832,-0.000992
4,w/o Def.,Generated_LossAware_T5,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662331,0.664198,0.001867,0.515824,0.514836,-0.000988
5,EDA_expert_T5,Clean,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.634742,0.000000,0.530975,0.530975,0.000000
6,EDA_expert_T5,PageRank_T5,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.634240,-0.000503,0.530975,0.531210,0.000235
7,EDA_expert_T5,Greedy_GC_RWCS_T5,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.639029,0.004286,0.530975,0.528581,-0.002394
8,EDA_expert_T5,Full_PR_BCD_T5,0.863372,0.860465,-0.002907,0.763819,0.760000,-0.003819,0.634742,0.637729,0.002986,0.530975,0.529303,-0.001672
9,EDA_expert_T5,Generated_LossAware_T5,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.638202,0.003459,0.530975,0.529038,-0.001936


Accuracy change from clean graph:


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,-0.002907,0.0,0.000000,0.000000
1,DA_lossaware_T5,0.0,-0.002907,0.0,0.000000,0.000000
2,EDA_expert_T5,0.0,-0.002907,0.0,0.000000,0.000000
3,w/o Def.,0.0,0.000000,0.0,-0.002907,0.002907


Target rumor loss increase from clean graph:


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,0.003012,0.003487,0.004353,-0.000517
1,DA_lossaware_T5,0.0,0.002956,0.003432,0.004271,-0.000509
2,EDA_expert_T5,0.0,0.002986,0.003459,0.004286,-0.000503
3,w/o Def.,0.0,0.001882,0.001867,0.002257,-0.000178


In [255]:
defense_output_dir = os.path.join(results_dir, "defense_outputs")
os.makedirs(defense_output_dir, exist_ok=True)

defense_eval_path = os.path.join(
    defense_output_dir,
    "defense_evaluation_results.csv"
)

accuracy_table_path = os.path.join(
    defense_output_dir,
    "defense_accuracy_change_table.csv"
)

loss_table_path = os.path.join(
    defense_output_dir,
    "defense_target_loss_increase_table.csv"
)

defense_eval_df.to_csv(defense_eval_path, index=False)
accuracy_table.to_csv(accuracy_table_path, index=False)
loss_table.to_csv(loss_table_path, index=False)

for defense_name, defense_model in defense_models.items():
    safe_name = defense_name.replace("/", "_").replace(" ", "_")
    model_path = os.path.join(
        defense_output_dir,
        f"{safe_name}_model.pt"
    )
    torch.save(defense_model.state_dict(), model_path)

for log_name, log_df in defense_training_logs.items():
    log_path = os.path.join(
        defense_output_dir,
        f"{log_name}_training_log.csv"
    )
    log_df.to_csv(log_path, index=False)

print("Saved defense outputs:")
print(defense_eval_path)
print(accuracy_table_path)
print(loss_table_path)
print(defense_output_dir)

Saved defense outputs:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/defense_evaluation_results.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/defense_accuracy_change_table.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/defense_target_loss_increase_table.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs


In [256]:
defense_report_df = defense_eval_df.copy()

defense_report_df["clean_accuracy_percent"] = defense_report_df["clean_accuracy"] * 100
defense_report_df["attacked_accuracy_percent"] = defense_report_df["attacked_accuracy"] * 100
defense_report_df["accuracy_change_percent_points"] = defense_report_df["accuracy_change_from_clean"] * 100
defense_report_df["rumor_f1_change_percent_points"] = defense_report_df["rumor_f1_change_from_clean"] * 100

paper_style_accuracy_table = defense_report_df.pivot(
    index="defense_name",
    columns="attack_name",
    values="accuracy_change_percent_points"
).reset_index()

paper_style_clean_accuracy = (
    defense_report_df[defense_report_df["attack_name"] == "Clean"]
    [["defense_name", "clean_accuracy_percent", "clean_rumor_f1"]]
    .copy()
)

paper_style_loss_table = defense_report_df.pivot(
    index="defense_name",
    columns="attack_name",
    values="target_loss_increase_from_clean"
).reset_index()

print("Clean performance:")
display(paper_style_clean_accuracy)

print("Paper-style accuracy change table, percentage points:")
display(paper_style_accuracy_table)

print("Target rumor loss increase table:")
display(paper_style_loss_table)

Clean performance:


,defense_name,clean_accuracy_percent,clean_rumor_f1
0,w/o Def.,74.127907,0.621277
5,EDA_expert_T5,86.337209,0.763819
10,DA_lossaware_T5,86.046512,0.760000
15,AT_lossaware_T5_T20,86.046512,0.760000


Paper-style accuracy change table, percentage points:


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,-0.290698,0.0,0.000000,0.000000
1,DA_lossaware_T5,0.0,-0.290698,0.0,0.000000,0.000000
2,EDA_expert_T5,0.0,-0.290698,0.0,0.000000,0.000000
3,w/o Def.,0.0,0.000000,0.0,-0.290698,0.290698


Target rumor loss increase table:


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,0.003012,0.003487,0.004353,-0.000517
1,DA_lossaware_T5,0.0,0.002956,0.003432,0.004271,-0.000509
2,EDA_expert_T5,0.0,0.002986,0.003459,0.004286,-0.000503
3,w/o Def.,0.0,0.001882,0.001867,0.002257,-0.000178


In [257]:
defense_models["Clean_finetune"], defense_training_logs["Clean_finetune"] = train_defense_model_with_adversarial_graphs(
    base_model=model,
    base_data=data,
    adversarial_graphs=[],
    node_to_idx=node_to_idx,
    device=device,
    beta=PAPER_DEFENSE_BETA,
    epochs=PAPER_PHEME_EPOCHS,
    lr=PAPER_TARGET_MODEL_LR,
    weight_decay=5e-4,
    print_every=10
)

print("Finished clean fine-tuning control.")
print(list(defense_models.keys()))

Epoch 001 | total=0.669617 | clean=0.669617 | adv=0.000000
Epoch 010 | total=0.668381 | clean=0.668381 | adv=0.000000
Epoch 020 | total=0.664148 | clean=0.664148 | adv=0.000000
Epoch 030 | total=0.661857 | clean=0.661857 | adv=0.000000
Epoch 040 | total=0.657486 | clean=0.657486 | adv=0.000000
Epoch 050 | total=0.654258 | clean=0.654258 | adv=0.000000
Epoch 060 | total=0.651753 | clean=0.651753 | adv=0.000000
Epoch 070 | total=0.647967 | clean=0.647967 | adv=0.000000
Epoch 080 | total=0.645879 | clean=0.645879 | adv=0.000000
Epoch 090 | total=0.642985 | clean=0.642985 | adv=0.000000
Epoch 100 | total=0.642393 | clean=0.642393 | adv=0.000000
Epoch 110 | total=0.633255 | clean=0.633255 | adv=0.000000
Epoch 120 | total=0.632739 | clean=0.632739 | adv=0.000000
Finished clean fine-tuning control.
['w/o Def.', 'EDA_expert_T5', 'DA_lossaware_T5', 'AT_lossaware_T5_T20', 'Clean_finetune']


In [258]:
control_eval_rows = []

defense_name = "Clean_finetune"
defense_model = defense_models[defense_name]

clean_metrics = evaluate_defense_model_on_graph(
    defense_model=defense_model,
    G_eval=G_processed,
    base_data=data,
    node_to_idx=node_to_idx,
    device=device,
    target_rumor_indices=target_rumor_indices
)

for attack_name, G_eval in attack_eval_graphs.items():
    metrics = evaluate_defense_model_on_graph(
        defense_model=defense_model,
        G_eval=G_eval,
        base_data=data,
        node_to_idx=node_to_idx,
        device=device,
        target_rumor_indices=target_rumor_indices
    )

    control_eval_rows.append({
        "defense_name": defense_name,
        "attack_name": attack_name,

        "clean_accuracy": clean_metrics["accuracy"],
        "attacked_accuracy": metrics["accuracy"],
        "accuracy_change_from_clean": metrics["accuracy"] - clean_metrics["accuracy"],

        "clean_rumor_f1": clean_metrics["rumor_f1"],
        "attacked_rumor_f1": metrics["rumor_f1"],
        "rumor_f1_change_from_clean": metrics["rumor_f1"] - clean_metrics["rumor_f1"],

        "clean_target_rumor_loss": clean_metrics["target_rumor_loss"],
        "attacked_target_rumor_loss": metrics["target_rumor_loss"],
        "target_loss_increase_from_clean": metrics["target_rumor_loss"] - clean_metrics["target_rumor_loss"],

        "clean_avg_target_rumor_prob": clean_metrics["avg_target_rumor_prob"],
        "attacked_avg_target_rumor_prob": metrics["avg_target_rumor_prob"],
        "avg_target_rumor_prob_change_from_clean": metrics["avg_target_rumor_prob"] - clean_metrics["avg_target_rumor_prob"]
    })

control_eval_df = pd.DataFrame(control_eval_rows)

defense_eval_with_control_df = pd.concat(
    [defense_eval_df, control_eval_df],
    ignore_index=True
)

display(defense_eval_with_control_df)

,defense_name,attack_name,clean_accuracy,attacked_accuracy,accuracy_change_from_clean,clean_rumor_f1,attacked_rumor_f1,rumor_f1_change_from_clean,clean_target_rumor_loss,attacked_target_rumor_loss,target_loss_increase_from_clean,clean_avg_target_rumor_prob,attacked_avg_target_rumor_prob,avg_target_rumor_prob_change_from_clean
0,w/o Def.,Clean,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662331,0.662331,0.000000,0.515824,0.515824,0.000000
1,w/o Def.,PageRank_T5,0.741279,0.744186,0.002907,0.621277,0.627119,0.005842,0.662331,0.662153,-0.000178,0.515824,0.515913,0.000089
2,w/o Def.,Greedy_GC_RWCS_T5,0.741279,0.738372,-0.002907,0.621277,0.618644,-0.002633,0.662331,0.664588,0.002257,0.515824,0.514631,-0.001193
3,w/o Def.,Full_PR_BCD_T5,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662331,0.664213,0.001882,0.515824,0.514832,-0.000992
4,w/o Def.,Generated_LossAware_T5,0.741279,0.741279,0.000000,0.621277,0.621277,0.000000,0.662331,0.664198,0.001867,0.515824,0.514836,-0.000988
5,EDA_expert_T5,Clean,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.634742,0.000000,0.530975,0.530975,0.000000
6,EDA_expert_T5,PageRank_T5,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.634240,-0.000503,0.530975,0.531210,0.000235
7,EDA_expert_T5,Greedy_GC_RWCS_T5,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.639029,0.004286,0.530975,0.528581,-0.002394
8,EDA_expert_T5,Full_PR_BCD_T5,0.863372,0.860465,-0.002907,0.763819,0.760000,-0.003819,0.634742,0.637729,0.002986,0.530975,0.529303,-0.001672
9,EDA_expert_T5,Generated_LossAware_T5,0.863372,0.863372,0.000000,0.763819,0.763819,0.000000,0.634742,0.638202,0.003459,0.530975,0.529038,-0.001936


In [259]:
defense_report_with_control_df = defense_eval_with_control_df.copy()

defense_report_with_control_df["clean_accuracy_percent"] = defense_report_with_control_df["clean_accuracy"] * 100
defense_report_with_control_df["attacked_accuracy_percent"] = defense_report_with_control_df["attacked_accuracy"] * 100
defense_report_with_control_df["accuracy_change_percent_points"] = defense_report_with_control_df["accuracy_change_from_clean"] * 100
defense_report_with_control_df["rumor_f1_change_percent_points"] = defense_report_with_control_df["rumor_f1_change_from_clean"] * 100

paper_style_clean_accuracy_with_control = (
    defense_report_with_control_df[defense_report_with_control_df["attack_name"] == "Clean"]
    [["defense_name", "clean_accuracy_percent", "clean_rumor_f1"]]
    .copy()
)

paper_style_accuracy_table_with_control = defense_report_with_control_df.pivot(
    index="defense_name",
    columns="attack_name",
    values="accuracy_change_percent_points"
).reset_index()

paper_style_loss_table_with_control = defense_report_with_control_df.pivot(
    index="defense_name",
    columns="attack_name",
    values="target_loss_increase_from_clean"
).reset_index()

print("Clean performance with control:")
display(paper_style_clean_accuracy_with_control)

print("Paper-style accuracy change table with control, percentage points:")
display(paper_style_accuracy_table_with_control)

print("Target rumor loss increase table with control:")
display(paper_style_loss_table_with_control)

Clean performance with control:


,defense_name,clean_accuracy_percent,clean_rumor_f1
0,w/o Def.,74.127907,0.621277
5,EDA_expert_T5,86.337209,0.763819
10,DA_lossaware_T5,86.046512,0.760000
15,AT_lossaware_T5_T20,86.046512,0.760000
20,Clean_finetune,86.046512,0.760000


Paper-style accuracy change table with control, percentage points:


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,-0.290698,0.0,0.000000,0.000000
1,Clean_finetune,0.0,-0.290698,0.0,0.000000,0.000000
2,DA_lossaware_T5,0.0,-0.290698,0.0,0.000000,0.000000
3,EDA_expert_T5,0.0,-0.290698,0.0,0.000000,0.000000
4,w/o Def.,0.0,0.000000,0.0,-0.290698,0.290698


Target rumor loss increase table with control:


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,0.003012,0.003487,0.004353,-0.000517
1,Clean_finetune,0.0,0.002831,0.003384,0.004177,-0.000481
2,DA_lossaware_T5,0.0,0.002956,0.003432,0.004271,-0.000509
3,EDA_expert_T5,0.0,0.002986,0.003459,0.004286,-0.000503
4,w/o Def.,0.0,0.001882,0.001867,0.002257,-0.000178


In [260]:
defense_output_dir = os.path.join(results_dir, "defense_outputs")
os.makedirs(defense_output_dir, exist_ok=True)

defense_eval_with_control_path = os.path.join(
    defense_output_dir,
    "defense_evaluation_results_with_control.csv"
)

paper_style_clean_accuracy_with_control_path = os.path.join(
    defense_output_dir,
    "paper_style_clean_accuracy_with_control.csv"
)

paper_style_accuracy_table_with_control_path = os.path.join(
    defense_output_dir,
    "paper_style_accuracy_change_with_control.csv"
)

paper_style_loss_table_with_control_path = os.path.join(
    defense_output_dir,
    "paper_style_target_loss_increase_with_control.csv"
)

defense_eval_with_control_df.to_csv(defense_eval_with_control_path, index=False)
paper_style_clean_accuracy_with_control.to_csv(paper_style_clean_accuracy_with_control_path, index=False)
paper_style_accuracy_table_with_control.to_csv(paper_style_accuracy_table_with_control_path, index=False)
paper_style_loss_table_with_control.to_csv(paper_style_loss_table_with_control_path, index=False)

if "Clean_finetune" in defense_models:
    clean_finetune_model_path = os.path.join(
        defense_output_dir,
        "Clean_finetune_model.pt"
    )
    torch.save(defense_models["Clean_finetune"].state_dict(), clean_finetune_model_path)

if "Clean_finetune" in defense_training_logs:
    clean_finetune_log_path = os.path.join(
        defense_output_dir,
        "Clean_finetune_training_log.csv"
    )
    defense_training_logs["Clean_finetune"].to_csv(clean_finetune_log_path, index=False)

print("Saved defense control outputs:")
print(defense_eval_with_control_path)
print(paper_style_clean_accuracy_with_control_path)
print(paper_style_accuracy_table_with_control_path)
print(paper_style_loss_table_with_control_path)

Saved defense control outputs:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/defense_evaluation_results_with_control.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/paper_style_clean_accuracy_with_control.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/paper_style_accuracy_change_with_control.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/defense_outputs/paper_style_target_loss_increase_with_control.csv


In [261]:



#FINAL RESULTS




In [262]:
final_irl_summary_rows = []

if "top1_accuracy" in globals():
    final_irl_summary_rows.append({
        "model": "EntIRL",
        "top1_expert_accuracy": top1_accuracy,
        "mean_expert_rank": mean_expert_rank,
        "mean_reciprocal_rank": mean_reciprocal_rank,
        "mean_expert_probability": mean_expert_probability
    })

if "moe_top1_accuracy" in globals():
    final_irl_summary_rows.append({
        "model": "MoE-EntIRL",
        "top1_expert_accuracy": moe_top1_accuracy,
        "mean_expert_rank": moe_mean_expert_rank,
        "mean_reciprocal_rank": moe_mean_reciprocal_rank,
        "mean_expert_probability": moe_mean_expert_probability
    })

if "bientirl_top1_accuracy" in globals():
    final_irl_summary_rows.append({
        "model": "MoE-BiEntIRL-style",
        "top1_expert_accuracy": bientirl_top1_accuracy,
        "mean_expert_rank": bientirl_mean_expert_rank,
        "mean_reciprocal_rank": bientirl_mean_reciprocal_rank,
        "mean_expert_probability": bientirl_mean_expert_probability
    })

final_irl_summary_df = pd.DataFrame(final_irl_summary_rows)

display(final_irl_summary_df)

,model,top1_expert_accuracy,mean_expert_rank,mean_reciprocal_rank,mean_expert_probability
0,EntIRL,0.546667,15.693333,0.596295,0.264288
1,MoE-EntIRL,0.560000,15.160000,0.609265,0.269701
2,MoE-BiEntIRL-style,0.586667,15.360000,0.622469,0.256678


In [263]:
generated_attack_tables = []

if "expert_trajectory_summary_df" in globals():
    generated_attack_tables.append(
        expert_trajectory_summary_df.assign(source="expert_attack")
    )

if "generated_policy_summary_df" in globals():
    generated_attack_tables.append(
        generated_policy_summary_df.assign(source="generated_reward_only")
    )

if "lossaware_summary_df" in globals():
    generated_attack_tables.append(
        lossaware_summary_df.assign(source="generated_reward_lossaware")
    )

final_attack_generation_summary_df = pd.concat(
    generated_attack_tables,
    ignore_index=True
)

final_attack_generation_summary_df = final_attack_generation_summary_df[
    [
        "source",
        "attack_name",
        "T",
        "num_steps",
        "target_loss_increase",
        "paper_delta_LA_clean_minus_final",
        "accuracy_change",
        "rumor_f1_change",
        "start_accuracy",
        "final_accuracy",
        "start_rumor_f1",
        "final_rumor_f1"
    ]
].sort_values(
    ["T", "target_loss_increase"],
    ascending=[True, False]
).reset_index(drop=True)

display(final_attack_generation_summary_df)

,source,attack_name,T,num_steps,target_loss_increase,paper_delta_LA_clean_minus_final,accuracy_change,rumor_f1_change,start_accuracy,final_accuracy,start_rumor_f1,final_rumor_f1
0,expert_attack,Greedy_GC_RWCS_T5,5,5,0.002255,-0.002255,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644
1,expert_attack,Full_PR_BCD_T5,5,5,0.001883,-0.001883,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277
2,generated_reward_lossaware,Generated_LossAware_MoE_BiEntIRL_T5,5,5,0.001867,-0.001867,0.000000,0.000000,0.741279,0.741279,0.621277,0.621277
3,generated_reward_only,Generated_MoE_BiEntIRL_T5,5,5,0.000739,-0.000739,-0.002907,-0.002633,0.741279,0.738372,0.621277,0.618644
4,expert_attack,PageRank_T5,5,5,-0.000185,0.000185,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119
5,expert_attack,Full_PR_BCD_T20,20,20,0.006344,-0.006344,-0.005814,-0.011835,0.741279,0.735465,0.621277,0.609442
6,generated_reward_lossaware,Generated_LossAware_MoE_BiEntIRL_T20,20,20,0.006008,-0.006008,-0.008721,-0.011107,0.741279,0.732558,0.621277,0.610169
7,expert_attack,Greedy_GC_RWCS_T20,20,20,0.005257,-0.005257,-0.005814,-0.005243,0.741279,0.735465,0.621277,0.616034
8,generated_reward_only,Generated_MoE_BiEntIRL_T20,20,20,0.000525,-0.000525,-0.008721,-0.007831,0.741279,0.732558,0.621277,0.613445
9,expert_attack,PageRank_T20,20,20,-0.000306,0.000306,0.002907,0.005842,0.741279,0.744186,0.621277,0.627119


In [264]:
final_defense_clean_summary_df = paper_style_clean_accuracy_with_control.copy()

final_defense_clean_summary_df = final_defense_clean_summary_df.sort_values(
    "clean_accuracy_percent",
    ascending=False
).reset_index(drop=True)

final_defense_accuracy_change_df = paper_style_accuracy_table_with_control.copy()
final_defense_target_loss_df = paper_style_loss_table_with_control.copy()

display(final_defense_clean_summary_df)

display(final_defense_accuracy_change_df)

display(final_defense_target_loss_df)

,defense_name,clean_accuracy_percent,clean_rumor_f1
0,EDA_expert_T5,86.337209,0.763819
1,AT_lossaware_T5_T20,86.046512,0.760000
2,DA_lossaware_T5,86.046512,0.760000
3,Clean_finetune,86.046512,0.760000
4,w/o Def.,74.127907,0.621277


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,-0.290698,0.0,0.000000,0.000000
1,Clean_finetune,0.0,-0.290698,0.0,0.000000,0.000000
2,DA_lossaware_T5,0.0,-0.290698,0.0,0.000000,0.000000
3,EDA_expert_T5,0.0,-0.290698,0.0,0.000000,0.000000
4,w/o Def.,0.0,0.000000,0.0,-0.290698,0.290698


attack_name,defense_name,Clean,Full_PR_BCD_T5,Generated_LossAware_T5,Greedy_GC_RWCS_T5,PageRank_T5
0,AT_lossaware_T5_T20,0.0,0.003012,0.003487,0.004353,-0.000517
1,Clean_finetune,0.0,0.002831,0.003384,0.004177,-0.000481
2,DA_lossaware_T5,0.0,0.002956,0.003432,0.004271,-0.000509
3,EDA_expert_T5,0.0,0.002986,0.003459,0.004286,-0.000503
4,w/o Def.,0.0,0.001882,0.001867,0.002257,-0.000178


In [265]:
best_irl_model_row = final_irl_summary_df.sort_values(
    ["top1_expert_accuracy", "mean_reciprocal_rank"],
    ascending=[False, False]
).iloc[0]

best_generated_T5_row = final_attack_generation_summary_df[
    final_attack_generation_summary_df["T"] == 5
].sort_values(
    "target_loss_increase",
    ascending=False
).iloc[0]

best_generated_T20_row = final_attack_generation_summary_df[
    final_attack_generation_summary_df["T"] == 20
].sort_values(
    "target_loss_increase",
    ascending=False
).iloc[0]

best_defense_clean_row = final_defense_clean_summary_df.sort_values(
    "clean_accuracy_percent",
    ascending=False
).iloc[0]

final_key_results_df = pd.DataFrame([
    {
        "result_type": "Best IRL reconstruction model",
        "name": best_irl_model_row["model"],
        "main_metric": "top1_expert_accuracy",
        "value": best_irl_model_row["top1_expert_accuracy"],
        "extra_info": f"mean_rank={best_irl_model_row['mean_expert_rank']:.4f}"
    },
    {
        "result_type": "Best T=5 attack/generation result",
        "name": best_generated_T5_row["attack_name"],
        "main_metric": "target_loss_increase",
        "value": best_generated_T5_row["target_loss_increase"],
        "extra_info": f"source={best_generated_T5_row['source']}"
    },
    {
        "result_type": "Best T=20 attack/generation result",
        "name": best_generated_T20_row["attack_name"],
        "main_metric": "target_loss_increase",
        "value": best_generated_T20_row["target_loss_increase"],
        "extra_info": f"source={best_generated_T20_row['source']}"
    },
    {
        "result_type": "Best defense clean performance",
        "name": best_defense_clean_row["defense_name"],
        "main_metric": "clean_accuracy_percent",
        "value": best_defense_clean_row["clean_accuracy_percent"],
        "extra_info": f"clean_rumor_f1={best_defense_clean_row['clean_rumor_f1']:.4f}"
    }
])

display(final_key_results_df)

,result_type,name,main_metric,value,extra_info
0,Best IRL reconstruction model,MoE-BiEntIRL-style,top1_expert_accuracy,0.586667,mean_rank=15.3600
1,Best T=5 attack/generation result,Greedy_GC_RWCS_T5,target_loss_increase,0.002255,source=expert_attack
2,Best T=20 attack/generation result,Full_PR_BCD_T20,target_loss_increase,0.006344,source=expert_attack
3,Best defense clean performance,EDA_expert_T5,clean_accuracy_percent,86.337209,clean_rumor_f1=0.7638


In [266]:
final_results_output_dir = os.path.join(
    results_dir,
    "final_report_tables"
)

os.makedirs(final_results_output_dir, exist_ok=True)

final_irl_summary_path = os.path.join(
    final_results_output_dir,
    "final_irl_reconstruction_summary.csv"
)

final_attack_generation_summary_path = os.path.join(
    final_results_output_dir,
    "final_attack_generation_summary.csv"
)

final_defense_clean_summary_path = os.path.join(
    final_results_output_dir,
    "final_defense_clean_summary.csv"
)

final_defense_accuracy_change_path = os.path.join(
    final_results_output_dir,
    "final_defense_accuracy_change_table.csv"
)

final_defense_target_loss_path = os.path.join(
    final_results_output_dir,
    "final_defense_target_loss_table.csv"
)

final_key_results_path = os.path.join(
    final_results_output_dir,
    "final_key_results.csv"
)

final_irl_summary_df.to_csv(final_irl_summary_path, index=False)
final_attack_generation_summary_df.to_csv(final_attack_generation_summary_path, index=False)
final_defense_clean_summary_df.to_csv(final_defense_clean_summary_path, index=False)
final_defense_accuracy_change_df.to_csv(final_defense_accuracy_change_path, index=False)
final_defense_target_loss_df.to_csv(final_defense_target_loss_path, index=False)
final_key_results_df.to_csv(final_key_results_path, index=False)

print("Saved final report tables:")
print(final_irl_summary_path)
print(final_attack_generation_summary_path)
print(final_defense_clean_summary_path)
print(final_defense_accuracy_change_path)
print(final_defense_target_loss_path)
print(final_key_results_path)

Saved final report tables:
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/final_report_tables/final_irl_reconstruction_summary.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/final_report_tables/final_attack_generation_summary.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/final_report_tables/final_defense_clean_summary.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/final_report_tables/final_defense_accuracy_change_table.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/final_report_tables/final_defense_target_loss_table.csv
/content/drive/MyDrive/7880TermProjectDatasets/saved_graphs/final_report_tables/final_key_results.csv
